In [ ]:
import requests

# Get series information for KXHIGHNY
url = "https://external-api.kalshi.com/trade-api/v2/series/KXHIGHNY"
response = requests.get(url)
series_data = response.json()

print(type(series_data))

print(series_data)
print(f"Series Title: {series_data['series']['title']}")
print(f"Frequency: {series_data['series']['frequency']}")
print(f"Category: {series_data['series']['category']}")

<class 'dict'>
{'series': {'additional_prohibitions': ['Persons who are employed by any of the Source Agencies are not permitted to trade on the Contract.', 'Persons who hold any material, non-public information on the Underlying are not permitted to trade on the Contract.'], 'category': 'Climate and Weather', 'contract_terms_url': 'https://kalshi-public-docs.s3.amazonaws.com/contract_terms/NHIGH.pdf', 'contract_url': 'https://kalshi-public-docs.s3.us-east-1.amazonaws.com/regulatory/product-certifications/NHIGH.pdf', 'fee_multiplier': 1, 'fee_type': 'quadratic', 'frequency': 'daily', 'last_updated_ts': '2026-03-16T15:04:55.113254Z', 'product_metadata': {'important_info': {'id': 'WEATHER-2025-3-3', 'markdown': '**Important information:** Not all weather data is the same. See the rules for more details.', 'message': 'Not all weather data is the same. While checking a source like AccuWeather or Google Weather may help guide your decision, the official and final value used to determine thi

In [ ]:
# Get all open markets for the KXHIGHNY series
markets_url = f"https://external-api.kalshi.com/trade-api/v2/markets?series_ticker=KXHIGHNY&"
markets_response = requests.get(markets_url)
markets_data = markets_response.json()
print(markets_data)


print(f"\nActive markets in KXHIGHNY series:")
for market in markets_data['markets']:
    print(f"- {market['ticker']}: {market['title']}")
    print(f"  Event: {market['event_ticker']}")
    print(f"  Yes Price: ${market['yes_bid_dollars']} | Volume: {market['volume_fp']}")
    print()

# Get details for a specific event if you have its ticker
if markets_data['markets']:
    # Let's get details for the first market's event
    event_ticker = markets_data['markets'][0]['event_ticker']
    event_url = f"https://external-api.kalshi.com/trade-api/v2/events/{event_ticker}"
    event_response = requests.get(event_url)
    event_data = event_response.json()

    print(f"Event Details:")
    print(f"Title: {event_data['event']['title']}")
    print(f"Category: {event_data['event']['category']}")


{'cursor': 'CgwIwtDV0AYQgNub0wMSFktYSElHSE5ZLTI2TUFZMjctQjgxLjU', 'markets': [{'can_close_early': True, 'close_time': '2026-06-13T04:59:00Z', 'created_time': '2026-06-11T09:30:48.09639Z', 'early_close_condition': 'The Last Trading Time will be 11:59 PM ET on June 12, 2026 regardless of any data releases or events occurring. Expiration will occur on the sooner of the first 7:00 or 8:00\nAM ET following the release of the data for June 12, 2026, or one week after June 12, 2026.', 'event_ticker': 'KXHIGHNY-26JUN12', 'expected_expiration_time': '2026-06-13T14:00:00Z', 'expiration_time': '2026-06-19T14:00:00Z', 'expiration_value': '', 'floor_strike': 97, 'fractional_trading_enabled': True, 'last_price_dollars': '0.0200', 'latest_expiration_time': '2026-06-19T14:00:00Z', 'liquidity_dollars': '0.0000', 'market_type': 'binary', 'no_ask_dollars': '0.9800', 'no_bid_dollars': '0.9700', 'no_sub_title': '98° or above', 'notional_value_dollars': '1.0000', 'occurrence_datetime': '2026-06-12T14:00:00Z

In [ ]:
# Get orderbook for a specific market
# Replace with an actual market ticker from the markets list
if not markets_data['markets']:
    raise ValueError("No open markets found. Try removing status=open or choose another series.")

market_ticker = markets_data['markets'][0]['ticker']
orderbook_url = f"https://external-api.kalshi.com/trade-api/v2/markets/{market_ticker}/orderbook"

orderbook_response = requests.get(orderbook_url)
orderbook_data = orderbook_response.json()

print(f"\nOrderbook for {market_ticker}:")
print("YES BIDS:")
for price_dollars, count_fp in orderbook_data['orderbook_fp']['yes_dollars'][:5]:  # Show top 5
    print(f"  Price: ${price_dollars}, Quantity: {count_fp}")

print("\nNO BIDS:")
for price_dollars, count_fp in orderbook_data['orderbook_fp']['no_dollars'][:5]:  # Show top 5
    print(f"  Price: ${price_dollars}, Quantity: {count_fp}")

In [ ]:
#Getting Economics category tickers for all events
import requests
import pandas as pd
import time
from typing import Optional, Dict, Any, List

BASE_URL = "https://external-api.kalshi.com/trade-api/v2"
CATEGORY = "Economics"


def kalshi_get(path: str, params: Optional[Dict[str, Any]] = None, max_retries: int = 3) -> Dict[str, Any]:
    """
    Simple GET helper with basic retry handling.
    Public market-data endpoints do not require authentication.
    """
    url = f"{BASE_URL}{path}"
    params = params or {}

    for attempt in range(max_retries):
        try:
            response = requests.get(url, params=params, timeout=30)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            if attempt == max_retries - 1:
                raise
            time.sleep(1.5 * (attempt + 1))


def get_series_by_category(category: str = "Economics") -> List[Dict[str, Any]]:
    """
    Get all Kalshi series in a category.
    """
    data = kalshi_get(
        "/series",
        params={
            "category": category,
            "include_product_metadata": "true",
            "include_volume": "true",
        },
    )
    return data.get("series", [])


def get_events_for_series(
    series_ticker: str,
    status: Optional[str] = None,
    with_nested_markets: bool = True,
) -> List[Dict[str, Any]]:
    """
    Get all events for one series ticker, handling cursor pagination.

    status options:
      None      -> all statuses
      "unopened"
      "open"
      "closed"
      "settled"
    """
    all_events = []
    cursor = None

    while True:
        params = {
            "series_ticker": series_ticker,
            "limit": 200,
            "with_nested_markets": str(with_nested_markets).lower(),
        }

        if status is not None:
            params["status"] = status

        if cursor:
            params["cursor"] = cursor

        data = kalshi_get("/events", params=params)

        events = data.get("events", [])
        all_events.extend(events)

        cursor = data.get("cursor")
        if not cursor:
            break

    return all_events


def economics_events_markets_df(
    category: str = "Economics",
    status: Optional[str] = None,
) -> pd.DataFrame:
    """
    Returns one row per market.

    If an event has no nested markets returned, it still creates one row
    with market fields set to None so you do not lose the event.
    """
    series_list = get_series_by_category(category)
    rows = []

    print(f"Found {len(series_list)} series in category={category!r}")

    for i, series in enumerate(series_list, start=1):
        series_ticker = series.get("ticker")
        series_title = series.get("title")

        print(f"[{i}/{len(series_list)}] Fetching events for {series_ticker}: {series_title}")

        events = get_events_for_series(
            series_ticker=series_ticker,
            status=status,
            with_nested_markets=True,
        )

        for event in events:
            markets = event.get("markets") or []

            event_base = {
                "category": event.get("category"),
                "series_ticker": event.get("series_ticker"),
                "series_title": series.get("title"),
                "series_frequency": series.get("frequency"),
                "series_tags": series.get("tags"),
                "event_ticker": event.get("event_ticker"),
                "event_title": event.get("title"),
                "event_sub_title": event.get("sub_title"),
                "event_strike_date": event.get("strike_date"),
                "event_strike_period": event.get("strike_period"),
                "event_last_updated_ts": event.get("last_updated_ts"),
                "event_product_metadata": event.get("product_metadata"),
            }

            if not markets:
                rows.append({
                    **event_base,
                    "market_ticker": None,
                    "market_question": None,
                    "market_title": None,
                    "market_subtitle": None,
                    "yes_sub_title": None,
                    "no_sub_title": None,
                    "market_status_open_time": None,
                    "market_close_time": None,
                    "market_expiration_time": None,
                    "yes_bid_dollars": None,
                    "yes_ask_dollars": None,
                    "no_bid_dollars": None,
                    "no_ask_dollars": None,
                    "last_price_dollars": None,
                    "volume_fp": None,
                    "volume_24h_fp": None,
                    "open_interest_fp": None,
                    "liquidity_dollars": None,
                    "rules_primary": None,
                    "rules_secondary": None,
                })
            else:
                for market in markets:
                    rows.append({
                        **event_base,
                        "market_ticker": market.get("ticker"),
                        # Kalshi's user-facing market question is usually the market title.
                        "market_question": market.get("title"),
                        "market_title": market.get("title"),
                        "market_subtitle": market.get("subtitle"),
                        "yes_sub_title": market.get("yes_sub_title"),
                        "no_sub_title": market.get("no_sub_title"),
                        "market_status_open_time": market.get("open_time"),
                        "market_close_time": market.get("close_time"),
                        "market_expiration_time": market.get("expiration_time"),
                        "yes_bid_dollars": market.get("yes_bid_dollars"),
                        "yes_ask_dollars": market.get("yes_ask_dollars"),
                        "no_bid_dollars": market.get("no_bid_dollars"),
                        "no_ask_dollars": market.get("no_ask_dollars"),
                        "last_price_dollars": market.get("last_price_dollars"),
                        "volume_fp": market.get("volume_fp"),
                        "volume_24h_fp": market.get("volume_24h_fp"),
                        "open_interest_fp": market.get("open_interest_fp"),
                        "liquidity_dollars": market.get("liquidity_dollars"),
                        "rules_primary": market.get("rules_primary"),
                        "rules_secondary": market.get("rules_secondary"),
                    })

    df = pd.DataFrame(rows)

    # Optional cleanup: remove duplicate market rows if pagination/API overlap ever occurs.
    if not df.empty:
        df = df.drop_duplicates(
            subset=["series_ticker", "event_ticker", "market_ticker"],
            keep="first",
        ).reset_index(drop=True)

    return df


# Use status=None for all events regardless of status.
# Use status="open" if you only want currently open economics markets.
df = economics_events_markets_df(
    category="Economics",
    status=None,
)

print("Rows:", len(df))
print("Unique series:", df["series_ticker"].nunique() if not df.empty else 0)
print("Unique events:", df["event_ticker"].nunique() if not df.empty else 0)
print("Unique markets:", df["market_ticker"].nunique() if not df.empty else 0)



Found 578 series in category='Economics'
[1/578] Fetching events for REALWAGES: Real wage growth
[2/578] Fetching events for KXCBDECISIONKOREA: Bank Of KOREA policy interest rate decision
[3/578] Fetching events for PCECORE: US Core PCE inflation
[4/578] Fetching events for KXUTILITYSOCAL: SoCal electricity prices
[5/578] Fetching events for KXCPICORE220: Will the BLS Core CPI YoY print below 2.20% 
[6/578] Fetching events for KXBRAZILJOBS: Brazil New Caged net formal job creation 
[7/578] Fetching events for KXECONSTATCORECPIYOY: year over year core inflation
[8/578] Fetching events for OILW: Price of oil weekly
[9/578] Fetching events for KXHOUSESTART: US housing starts rise
[10/578] Fetching events for KXEZCPIYOYF: Euro Area Inflation Rate YoY Flash
[11/578] Fetching events for NYCRENTSM: NYC rent monthly increase
[12/578] Fetching events for TBILL: Treasury bill rate
[13/578] Fetching events for CPIAPPAREL: CPI on apparel
[14/578] Fetching events for OIL: Price of oil monthly
[15/5

,category,series_ticker,series_title,series_frequency,series_tags,event_ticker,event_title,event_sub_title,event_strike_date,event_strike_period,...,yes_ask_dollars,no_bid_dollars,no_ask_dollars,last_price_dollars,volume_fp,volume_24h_fp,open_interest_fp,liquidity_dollars,rules_primary,rules_secondary
0,Economics,KXCBDECISIONKOREA,Bank Of KOREA policy interest rate decision,custom,[Global Central Banks],KXCBDECISIONKOREA-26JUL15,Bank of Korea rate decision in July,"Jul 15, 2026 meeting",None,,...,0.0100,0.9900,1.0000,0.0100,2245.04,0.00,2044.99,0.0000,If the Bank of Korea takes the action of Cut m...,The market resolves based on the official poli...
1,Economics,KXCBDECISIONKOREA,Bank Of KOREA policy interest rate decision,custom,[Global Central Banks],KXCBDECISIONKOREA-26JUL15,Bank of Korea rate decision in July,"Jul 15, 2026 meeting",None,,...,0.0500,0.9500,0.9800,0.0400,2393.00,0.00,1522.00,0.0000,If the Bank of Korea takes the action of Cut 1...,The market resolves based on the official poli...
2,Economics,KXCBDECISIONKOREA,Bank Of KOREA policy interest rate decision,custom,[Global Central Banks],KXCBDECISIONKOREA-26JUL15,Bank of Korea rate decision in July,"Jul 15, 2026 meeting",None,,...,0.3100,0.6900,0.7600,0.2800,2914.00,138.00,975.00,0.0000,If the Bank of Korea takes the action of Maint...,The market resolves based on the official poli...
3,Economics,KXCBDECISIONKOREA,Bank Of KOREA policy interest rate decision,custom,[Global Central Banks],KXCBDECISIONKOREA-26JUL15,Bank of Korea rate decision in July,"Jul 15, 2026 meeting",None,,...,0.7100,0.2900,0.3500,0.6500,4023.14,254.00,2175.57,0.0000,If the Bank of Korea takes the action of Hike ...,The market resolves based on the official poli...
4,Economics,KXCBDECISIONKOREA,Bank Of KOREA policy interest rate decision,custom,[Global Central Banks],KXCBDECISIONKOREA-26JUL15,Bank of Korea rate decision in July,"Jul 15, 2026 meeting",None,,...,0.0500,0.9500,0.9900,0.0100,914.05,0.00,862.05,0.0000,If the Bank of Korea takes the action of Hike ...,The market resolves based on the official poli...


In [ ]:
df.head(90)

,category,series_ticker,series_title,series_frequency,series_tags,event_ticker,event_title,event_sub_title,event_strike_date,event_strike_period,...,yes_ask_dollars,no_bid_dollars,no_ask_dollars,last_price_dollars,volume_fp,volume_24h_fp,open_interest_fp,liquidity_dollars,rules_primary,rules_secondary
0,Economics,KXCBDECISIONKOREA,Bank Of KOREA policy interest rate decision,custom,[Global Central Banks],KXCBDECISIONKOREA-26JUL15,Bank of Korea rate decision in July,"Jul 15, 2026 meeting",None,,...,0.0100,0.9900,1.0000,0.0100,2245.04,0.00,2044.99,0.0000,If the Bank of Korea takes the action of Cut m...,The market resolves based on the official poli...
1,Economics,KXCBDECISIONKOREA,Bank Of KOREA policy interest rate decision,custom,[Global Central Banks],KXCBDECISIONKOREA-26JUL15,Bank of Korea rate decision in July,"Jul 15, 2026 meeting",None,,...,0.0500,0.9500,0.9800,0.0400,2393.00,0.00,1522.00,0.0000,If the Bank of Korea takes the action of Cut 1...,The market resolves based on the official poli...
2,Economics,KXCBDECISIONKOREA,Bank Of KOREA policy interest rate decision,custom,[Global Central Banks],KXCBDECISIONKOREA-26JUL15,Bank of Korea rate decision in July,"Jul 15, 2026 meeting",None,,...,0.3100,0.6900,0.7600,0.2800,2914.00,138.00,975.00,0.0000,If the Bank of Korea takes the action of Maint...,The market resolves based on the official poli...
3,Economics,KXCBDECISIONKOREA,Bank Of KOREA policy interest rate decision,custom,[Global Central Banks],KXCBDECISIONKOREA-26JUL15,Bank of Korea rate decision in July,"Jul 15, 2026 meeting",None,,...,0.7100,0.2900,0.3500,0.6500,4023.14,254.00,2175.57,0.0000,If the Bank of Korea takes the action of Hike ...,The market resolves based on the official poli...
4,Economics,KXCBDECISIONKOREA,Bank Of KOREA policy interest rate decision,custom,[Global Central Banks],KXCBDECISIONKOREA-26JUL15,Bank of Korea rate decision in July,"Jul 15, 2026 meeting",None,,...,0.0500,0.9500,0.9900,0.0100,914.05,0.00,862.05,0.0000,If the Bank of Korea takes the action of Hike ...,The market resolves based on the official poli...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,Economics,KXECONSTATCORECPIYOY,year over year core inflation,custom,[Inflation],KXECONSTATCORECPIYOY-26JUN,CPI core year-over-year in Jun 2026?,In Jun 2026,None,,...,0.0100,0.9900,1.0000,0.0100,1510.00,0.00,657.00,0.0000,If the CPI core year-over-year is exactly 2.5%...,
86,Economics,KXECONSTATCORECPIYOY,year over year core inflation,custom,[Inflation],KXECONSTATCORECPIYOY-26JUN,CPI core year-over-year in Jun 2026?,In Jun 2026,None,,...,0.0400,0.9600,1.0000,0.0400,1499.00,20.00,944.00,0.0000,If the CPI core year-over-year is exactly 2.6%...,
87,Economics,KXECONSTATCORECPIYOY,year over year core inflation,custom,[Inflation],KXECONSTATCORECPIYOY-26JUN,CPI core year-over-year in Jun 2026?,In Jun 2026,None,,...,0.1100,0.8900,0.9000,0.1700,4279.85,0.00,1709.80,0.0000,If the CPI core year-over-year is exactly 2.7%...,
88,Economics,KXECONSTATCORECPIYOY,year over year core inflation,custom,[Inflation],KXECONSTATCORECPIYOY-26JUN,CPI core year-over-year in Jun 2026?,In Jun 2026,None,,...,0.2500,0.7500,0.7900,0.2100,4813.82,0.00,3292.37,0.0000,If the CPI core year-over-year is exactly 2.8%...,


In [ ]:
df['category']

,category
0,Economics
1,Economics
2,Economics
3,Economics
4,Economics
...,...
14141,Economics
14142,Economics
14143,Economics
14144,World


In [ ]:
filtered = df[df["series_ticker"].str.contains("KXNJDRONES", na=False)]
filtered

,category,series_ticker,series_title,series_frequency,series_tags,event_ticker,event_title,event_sub_title,event_strike_date,event_strike_period,...,yes_ask_dollars,no_bid_dollars,no_ask_dollars,last_price_dollars,volume_fp,volume_24h_fp,open_interest_fp,liquidity_dollars,rules_primary,rules_secondary
1714,Financials,KXNJDRONES,NJ DRONES,custom,None,KXNJDRONES-25,Will the New Jersey Drone Sightings reoccur in...,In 2025,None,,...,None,None,None,None,None,None,None,None,None,None


In [ ]:
filtered_df = df[df["category"].str.contains("Financials", na=False)]
filtered_df

,category,series_ticker,series_title,series_frequency,series_tags,event_ticker,event_title,event_sub_title,event_strike_date,event_strike_period,...,yes_ask_dollars,no_bid_dollars,no_ask_dollars,last_price_dollars,volume_fp,volume_24h_fp,open_interest_fp,liquidity_dollars,rules_primary,rules_secondary
1714,Financials,KXNJDRONES,NJ DRONES,custom,None,KXNJDRONES-25,Will the New Jersey Drone Sightings reoccur in...,In 2025,None,,...,None,None,None,None,None,None,None,None,None,None
4606,Financials,KXXINDIA,INDIAN ACCOUNTS X,custom,None,KXXINDIA-25,Will a story be published that at least 70% of...,In 2025,None,,...,None,None,None,None,None,None,None,None,None,None
4719,Financials,KXMUSKX,Musk Manipulating X,one_off,None,KXMUSKX-25,Will a report about Musk manipulating X to ben...,In 2025,None,,...,None,None,None,None,None,None,None,None,None,None
5606,Financials,KXCREDITC,SOFR spike: Credit crunch,annual,[Fed],KXCREDITC-25DEC31,Will SOFR spike in 2025?,"By Dec 31, 2025",None,,...,None,None,None,None,None,None,None,None,None,None
5607,Financials,KXCREDITC,SOFR spike: Credit crunch,annual,[Fed],CREDITC-24DEC31,"SOFR spike: Credit crunch by Dec 31, 2024?","By Dec 31, 2024",None,,...,None,None,None,None,None,None,None,None,None,None
5608,Financials,KXCREDITC,SOFR spike: Credit crunch,annual,[Fed],CREDITC-23DEC31,"SOFR spike: Credit crunch, 2023?",2023,None,,...,None,None,None,None,None,None,None,None,None,None
12792,Financials,KXNYSEOPEN,Will NYSE be open?,one_off,None,KXNYSEOPEN-26,Will the New York Stock Exchange be Open for t...,In 2026,None,,...,None,None,None,None,None,None,None,None,None,None
12860,Financials,KXFOMCDISSENTCOUNT,FOMC dissent count,custom,[Fed],KXFOMCDISSENTCOUNT-25DEC,How many dissenting votes at the next Fed meet...,December FOMC meeting,None,,...,None,None,None,None,None,None,None,None,None,None
12861,Financials,KXFOMCDISSENTCOUNT,FOMC dissent count,custom,[Fed],KXFOMCDISSENTCOUNT-25JUL,How many dissenting votes at the next Fed meet...,Before 2025,None,,...,None,None,None,None,None,None,None,None,None,None


In [ ]:
df['category'].value_counts()

,count
category,
Economics,13780
Transportation,197
Science and Technology,76
World,54
Politics,29
Financials,9
Health,1


In [ ]:
df.to_csv("kalshi_economics_events_markets.csv", index=False)

In [ ]:
import requests
import pandas as pd
import time
from typing import Dict, Any, Optional, List

BASE_URL = "https://external-api.kalshi.com/trade-api/v2"

CATEGORY = "Economics"
START_DATE = "2024-01-01"
END_DATE   = "2026-04-14"

# For "closed markets in the time window", close_time is usually the right filter.
# Change to "settlement_ts" if you want settlement date instead.
DATE_FIELD = "close_time"


def kalshi_get(
    path: str,
    params: Optional[Dict[str, Any]] = None,
    max_retries: int = 4,
    sleep_base: float = 1.5,
) -> Dict[str, Any]:
    url = f"{BASE_URL}{path}"
    params = params or {}

    for attempt in range(max_retries):
        try:
            r = requests.get(url, params=params, timeout=30)

            if not r.ok:
                print("Failed URL:", url)
                print("Params:", params)
                print("Status code:", r.status_code)
                print("Response text:", r.text[:1000])

            r.raise_for_status()
            return r.json()

        except requests.exceptions.RequestException:
            if attempt == max_retries - 1:
                raise
            time.sleep(sleep_base * (attempt + 1))


def paginate(path: str, params: Dict[str, Any], result_key: str) -> List[Dict[str, Any]]:
    rows = []
    cursor = None

    while True:
        query = dict(params)
        if cursor:
            query["cursor"] = cursor

        data = kalshi_get(path, query)
        rows.extend(data.get(result_key, []))

        cursor = data.get("cursor")
        if not cursor:
            break

    return rows


def get_historical_cutoff() -> Dict[str, Any]:
    return kalshi_get("/historical/cutoff")


def get_economics_series() -> pd.DataFrame:
    """
    Step 1:
    Get all series_ticker values for the Economics category.
    """
    data = kalshi_get(
        "/series",
        {
            "category": CATEGORY,
            "include_product_metadata": "true",
            "include_volume": "true",
        },
    )

    df = pd.DataFrame(data.get("series", []))

    if df.empty:
        return df

    df = df.rename(columns={
        "ticker": "series_ticker",
        "title": "series_title",
        "frequency": "series_frequency",
        "category": "series_category",
        "product_metadata": "series_product_metadata",
        "volume_fp": "series_volume_fp",
    })

    return df


def get_historical_markets_for_series(series_ticker: str) -> List[Dict[str, Any]]:
    """
    Step 2:
    Get historical markets for a single series_ticker.

    IMPORTANT:
    Do NOT pass mve_filter here.
    /historical/markets treats series_ticker and mve_filter as mutually exclusive.
    """
    params = {
        "series_ticker": series_ticker,
        "limit": 1000,
    }

    return paginate("/historical/markets", params, "markets")


def get_events_for_series(series_ticker: str) -> List[Dict[str, Any]]:
    """
    Step 3:
    Get event metadata. Events remain accessible from /events even when
    their markets are historical.
    """
    params = {
        "series_ticker": series_ticker,
        "limit": 200,
        "with_nested_markets": "false",
    }

    return paginate("/events", params, "events")


def filter_by_date_window(df: pd.DataFrame, date_field: str = DATE_FIELD) -> pd.DataFrame:
    if df.empty:
        return df

    if date_field not in df.columns:
        raise ValueError(
            f"Column {date_field!r} not found. Available columns include: "
            f"{list(df.columns)[:40]}"
        )

    dt = pd.to_datetime(df[date_field], utc=True, errors="coerce")

    start = pd.Timestamp(START_DATE, tz="UTC")
    end = pd.Timestamp(END_DATE, tz="UTC") + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)

    return df[(dt >= start) & (dt <= end)].copy()


def build_economics_closed_historical_df():
    cutoff = get_historical_cutoff()
    market_cutoff = pd.to_datetime(cutoff["market_settled_ts"], utc=True)

    print("Kalshi historical cutoff:")
    print("  market_settled_ts:", market_cutoff)
    print()
    print(f"Target category: {CATEGORY}")
    print(f"Target window: {START_DATE} through {END_DATE}")
    print(f"Date field used for filtering: {DATE_FIELD}")
    print()

    series_df = get_economics_series()

    if series_df.empty:
        raise RuntimeError("No Economics series found.")

    print(f"Found {len(series_df)} Economics series.")
    print("Example series tickers:")
    print(series_df["series_ticker"].head(20).to_list())
    print()

    market_dfs = []
    event_rows = []
    failed_series = []

    for i, row in series_df.iterrows():
        series_ticker = row["series_ticker"]
        series_title = row.get("series_title", "")

        print(f"[{i + 1}/{len(series_df)}] {series_ticker} — {series_title}")

        try:
            markets = get_historical_markets_for_series(series_ticker)

            if markets:
                mdf = pd.json_normalize(markets)
                mdf["series_ticker"] = series_ticker
                mdf["data_source"] = "historical"
                market_dfs.append(mdf)

            events = get_events_for_series(series_ticker)

            for event in events:
                event["series_ticker"] = series_ticker

            event_rows.extend(events)

        except requests.exceptions.HTTPError as e:
            failed_series.append({
                "series_ticker": series_ticker,
                "series_title": series_title,
                "error": str(e),
            })
            print(f"  Skipped because of error: {e}")

        time.sleep(0.1)

    failed_df = pd.DataFrame(failed_series)

    if not market_dfs:
        return pd.DataFrame(), series_df, failed_df

    markets_df = pd.concat(market_dfs, ignore_index=True)

    # Historical endpoint may include more than your requested window,
    # so filter locally.
    markets_df = filter_by_date_window(markets_df, DATE_FIELD)

    if markets_df.empty:
        return markets_df, series_df, failed_df

    # Keep closed/settled markets only if the status field exists.
    if "status" in markets_df.columns:
        markets_df = markets_df[
            markets_df["status"].isin(["closed", "settled"])
        ].copy()

    # Deduplicate by market ticker.
    if "ticker" in markets_df.columns:
        markets_df = markets_df.drop_duplicates("ticker", keep="first")

    # Join series metadata.
    series_keep_cols = [
        c for c in [
            "series_ticker",
            "series_title",
            "series_frequency",
            "series_category",
            "tags",
            "series_product_metadata",
            "series_volume_fp",
        ]
        if c in series_df.columns
    ]

    markets_df = markets_df.merge(
        series_df[series_keep_cols],
        on="series_ticker",
        how="left",
    )

    # Join event metadata.
    events_df = pd.json_normalize(event_rows) if event_rows else pd.DataFrame()

    if not events_df.empty and "event_ticker" in events_df.columns:
        event_keep_cols = [
            c for c in [
                "event_ticker",
                "title",
                "sub_title",
                "category",
                "strike_date",
                "strike_period",
                "product_metadata",
                "last_updated_ts",
            ]
            if c in events_df.columns
        ]

        event_meta = events_df[event_keep_cols].drop_duplicates("event_ticker")

        markets_df = markets_df.merge(
            event_meta,
            on="event_ticker",
            how="left",
            suffixes=("_market", "_event"),
        )

    # Friendly column names.
    markets_df = markets_df.rename(columns={
        "ticker": "market_ticker",
        "title_market": "market_question",
        "title": "market_question",
        "subtitle": "market_subtitle",
        "title_event": "event_title",
        "sub_title": "event_sub_title",
        "category": "event_category",
    })

    markets_df["requested_category"] = CATEGORY
    markets_df["requested_start_date"] = START_DATE
    markets_df["requested_end_date"] = END_DATE
    markets_df["historical_cutoff_market_settled_ts"] = market_cutoff

    # Useful columns first.
    first_cols = [
        "requested_category",
        "data_source",
        "series_ticker",
        "series_title",
        "series_frequency",
        "series_category",
        "event_ticker",
        "event_title",
        "event_sub_title",
        "market_ticker",
        "market_question",
        "market_subtitle",
        "yes_sub_title",
        "no_sub_title",
        "status",
        "open_time",
        "close_time",
        "expiration_time",
        "expected_expiration_time",
        "settlement_ts",
        "settlement_value_dollars",
        "last_price_dollars",
        "yes_bid_dollars",
        "yes_ask_dollars",
        "no_bid_dollars",
        "no_ask_dollars",
        "volume_fp",
        "volume_24h_fp",
        "open_interest_fp",
        "liquidity_dollars",
        "rules_primary",
        "rules_secondary",
        "requested_start_date",
        "requested_end_date",
        "historical_cutoff_market_settled_ts",
    ]

    first_cols = [c for c in first_cols if c in markets_df.columns]
    other_cols = [c for c in markets_df.columns if c not in first_cols]

    markets_df = markets_df[first_cols + other_cols].reset_index(drop=True)

    # Unique series tickers that actually had markets in the requested window.
    series_tickers_df = (
        markets_df[[
            "series_ticker",
            "series_title",
            "series_frequency",
            "series_category",
        ]]
        .drop_duplicates()
        .sort_values("series_ticker")
        .reset_index(drop=True)
    )

    return markets_df, series_tickers_df, failed_df


df_markets, df_series_tickers, df_failed_series = build_economics_closed_historical_df()

print()
print("Final output")
print("Rows / markets:", len(df_markets))

if not df_markets.empty:
    print("Unique series_tickers:", df_markets["series_ticker"].nunique())
    print("Unique event_tickers:", df_markets["event_ticker"].nunique())
    print("Unique market_tickers:", df_markets["market_ticker"].nunique())

print("Failed series:", len(df_failed_series))

df_markets.head()

Kalshi historical cutoff:
  market_settled_ts: 2026-04-16 00:00:00+00:00

Target category: Economics
Target window: 2024-01-01 through 2026-04-14
Date field used for filtering: close_time

Found 578 Economics series.
Example series tickers:
['KXUTILITYSOCAL', 'KXNDEBTSHRINK', 'KXAAAGASMAXNY', 'LCPIYOY', 'KXHELIUMIMP', 'KXNFPDELAY', 'KXACPICORE', 'KXCREDITRATING', 'KXAUNABCONF', 'NGDPQ', 'KXPSAVERT', 'KXCPIUSEDCAR', 'KXSPRMAX', 'PESO', 'KXFEDMEET', 'KXGDPUSMAX', 'KXDEBTGROWTH26', 'KXLCPIYOY', 'KXTRUFTSA', 'KXGDPYEAR']

[1/578] KXUTILITYSOCAL — SoCal electricity prices
[2/578] KXNDEBTSHRINK — national debt shrinking
[3/578] KXAAAGASMAXNY — NEW YORK highest gas price yearly
[4/578] LCPIYOY — Inflation surge this year
Failed URL: https://external-api.kalshi.com/trade-api/v2/events
Params: {'series_ticker': 'LCPIYOY', 'limit': 200, 'with_nested_markets': 'false'}
Status code: 429
Response text: {"error":{"code":"too_many_requests","message":"too many requests"}}
[5/578] KXHELIUMIMP — US hel

,requested_category,data_source,series_ticker,series_title,series_frequency,series_category,event_ticker,event_title,event_sub_title,market_ticker,...,custom_strike.Expo_Date,custom_strike.Interest Rate,custom_strike.Count,tags,series_product_metadata,series_volume_fp,event_category,strike_date,strike_period,last_updated_ts


**16 June 2026 Work**

***Historical Cutoff***

A closed market means trading has ended -- People can no longer buy or sell contracts in that market.

A settled market means the final result has been resolved.

There can be a short period where a market is closed but not yet settled.

Kalshi moves older ***settled*** markets into the historical endpoint.

For series and events data - no historical endpoint in API needed




**Kalshi Hierarchy**

Category: X

Series:
KXCPI

Event:
CPI report for March 2025

Markets:


*   Will CPI be above 2.5%?
*   Will CPI be above 3.0%?



Orders:
Buy YES at 40 cents
Sell YES at 45 cents

Trades:
YES traded at 42 cents
YES traded at 43 cents


***Nested markets***

        
            "events":

            "event_ticker": "KXCPI-25MAR",
            "series_ticker": "KXCPI",
            "title": "CPI inflation for March 2025",
            "markets":
                {
                    "ticker": "KXCPI-25MAR-B2.5",
                    "title": "Will CPI be above 2.5%?",
                    "status": "settled",
                    "open_time": "2025-03-01T00:00:00Z",
                    "close_time": "2025-03-12T13:30:00Z"
                },
                {
                    "ticker": "KXCPI-25MAR-B3.0",
                    "title": "Will CPI be above 3.0%?",
                    "status": "settled",
                    "open_time": "2025-03-01T00:00:00Z",
                    "close_time": "2025-03-12T13:30:00Z"
                }
            
        
    

Kalshi’s historical cutoff for markets is based on settlement time, not close time. Kalshi’s docs define market_settled_ts as the cutoff: markets that settled before that timestamp must be accessed through /historical/markets; markets that settled after that cutoff should be available through /markets.

The warning is that this nested markets list is not guaranteed to include old historical markets. Kalshi says that historical markets settled before the historical cutoff are not included when using with_nested_markets=true on /events.


event: KXCPI-24JAN

It may have had 10 markets. But if those markets settled before Kalshi’s historical cutoff, then this request:

GET /events?series_ticker=KXCPI&with_nested_markets=true

might return the event, but with zero markets or only some newer market


If the market settled after the historical cutoff, then it should still be available through the normal live/recent endpoint:

GET /markets

Market settled before cutoff  → /historical/markets

Market settled after cutoff   → /markets

A closed market means trading has ended.

People can no longer buy or sell contracts.

A settled market means the final result has been resolved.

Kalshi knows whether YES or NO won, and payouts can happen.

Category
  → Series
      → Event
          → Market
              → Orders
              → Trades

Category: X / Economics

Series (a series is a recurring family of related events): KXCPI

Event (an event is one specific occurrence inside a series): CPI report for March 2025

Markets (one specific tradable question inside an event):
  - Will CPI be above 2.5%?
  - Will CPI be above 3.0%?

Orders (An order is someone trying to buy or sell):
  - Buy YES at 40 cents
  - Sell YES at 45 cents

Trades (A trade is when an order actually matches and executes):
  - YES traded at 42 cents
  - YES traded at 43 cents

Nested markets means that Kalshi includes the market objects directly inside each event response. Normally, when you call /events?series_ticker=KXCPI, Kalshi returns event-level information, such as the event_ticker, series_ticker, and event title.

{
  "event_ticker": "KXCPI-25MAR",

  "series_ticker": "KXCPI",

  "title": "CPI inflation for March 2025"
}



If you add with_nested_markets=true, like /events?series_ticker=KXCPI&with_nested_markets=true, each event can also include a markets field containing the markets that belong to that event.


If you add with_nested_markets=true, Kalshi also includes the markets inside each event.

Example:

GET /events?series_ticker=KXCPI&with_nested_markets=true

Then the event response may include a markets field:

{
  "event_ticker": "KXCPI-25MAR",

  "title": "CPI inflation for March 2025",

  "markets":
    
      "ticker": "KXCPI-25MAR-B2.5",

      "title": "Will CPI be above 2.5%?"
    
    
      "ticker": "KXCPI-25MAR-B3.0",

      "title": "Will CPI be above 3.0%?"
    
  
}


Important warning: with_nested_markets=true does not include historical markets that settled before Kalshi’s historical cutoff. Those older settled markets have moved to:

GET /historical/markets

Therefore, /events?with_nested_markets=true may return an event but still miss some older markets. To reliably find all markets for a series across a historical date range, use both:

GET /markets
GET /historical/markets

Orders are requests to buy or sell contracts. For example, someone might place an order saying: “I want to buy 10 YES contracts at 40 cents” or “I want to sell 5 YES contracts at 45 cents.” An order is just an offer or instruction. It may sit there waiting, get canceled, or eventually get matched.

Trades are completed transactions. A trade happens when a buy order and a sell order match at a price. For example, if one person wants to buy YES at 42 cents and another person is willing to sell YES at 42 cents, the orders match and a trade occurs. Kalshi describes a trade as a completed transaction between two users on a specific market, including the market ticker, price, quantity, and timestamp.


Order = someone wants to buy or sell
Trade = the buy/sell actually happened

Kalshi operates like an order book exchange, not like a store with a fixed pile of contracts already sitting there.

Kalshi creates the market question, for example:

Will CPI be above 3.0%?

Then traders create the available offers by placing orders. An order might be:

I want to buy YES at 40 cents.
I want to sell YES at 45 cents.

Those orders sit in the order book until someone accepts them or places a matching order. Kalshi’s order book shows active bid orders for YES and NO sides. Because Kalshi markets are binary, a YES bid at one price is economically related to a NO offer at the opposite price. For example, Kalshi notes that a YES bid at 7 cents is equivalent to a NO ask at 93 cents.

So there usually is not an initial fixed amount of offers created by Kalshi. Instead, liquidity comes from users and market makers placing orders.

Buying a side = you want that side to win.

Selling a side = you want that side to lose.

Buy YES and sell NO both mean you are leaning YES.

Buy NO and sell YES both mean you are leaning NO.

_______________________________________________________________

Will CPI be above 3.0%?

There are two outcomes:

YES = CPI is above 3.0%

NO = CPI is not above 3.0%

________________________________

Buying YES = betting YES happens.

You make money if the answer is YES.

Example: you buy YES because you think CPI will be above 3.0%.
You buy 1 YES contract for $0.40.

__

Winning case:

CPI is above 3.0%.
YES wins.
Your YES contract pays $1.
$1.00 - $0.40 = $0.60 profit

Losing case:

CPI is not above 3.0%.
NO wins.
Your YES contract pays $0.
loss is $0.40

_______________________________________________________________


Selling YES = betting YES does not happen.

You make money if the answer is NO.

Example: you sell YES because you think CPI will not be above 3.0%.

_______________________________________________________________


Buying NO = betting NO happens.

You make money if the answer is NO.

Example: you buy NO because you think CPI will not be above 3.0%.

_______________________________________________________________


Selling NO = betting NO does not happen.

You make money if the answer is YES.

Example: you sell NO because you think CPI will be above 3.0%.

why both sell and buy exist: So why choose one over the other? Because one side may have a better available price or more liquidity. Maybe there is someone willing to sell YES at 0.42 dollars but someone else willing to buy NO at 0.61 dollars. You would choose the better deal.

# Cutoff

In [ ]:
import requests
import pandas as pd

BASE_URL = "https://external-api.kalshi.com/trade-api/v2"

resp = requests.get(f"{BASE_URL}/historical/cutoff")
resp.raise_for_status()

cutoff = resp.json()
cutoff

{'market_settled_ts': '2026-04-24T00:00:00Z',
 'orders_updated_ts': '2026-04-24T00:00:00Z',
 'trades_created_ts': '2026-04-24T00:00:00Z'}

PLAN:

If you want all markets/events for a series_ticker in a date range, do not rely only on nested markets.

Use this safer process:

1. Query /markets for live/recent markets
2. Query /historical/markets for older settled markets
3. Combine them
4. Deduplicate by market ticker
5. Filter by open_time and close_time
6. Extract event_ticker values
7. Optionally fetch event details from /events

In one sentence:

Use /events to describe events, but use /markets plus /historical/markets to reliably find all markets across your time period.

So when you get series from:

GET /series?category=Economics

you are getting Economics series definitions, not necessarily live tradable markets.

A series may be:

currently active with open markets

still listed but no open markets right now

old/inactive

recurring over time

one-off (happens once) or limited (repeats only during a specific period)

# Live Markets Code -  takes too much time to run

In [ ]:
import time
import requests
import pandas as pd
from urllib.parse import urljoin

BASE_URL = "https://external-api.kalshi.com/trade-api/v2"

session = requests.Session()
session.headers.update({
    "User-Agent": "kalshi-live-markets-colab/1.0"
})


def get_json(path, params=None, max_retries=10, timeout=30):
    url = urljoin(BASE_URL + "/", path.lstrip("/"))
    params = params or {}

    for attempt in range(max_retries):
        resp = session.get(url, params=params, timeout=timeout)

        if resp.status_code == 429:
            sleep_s = min(5 * (attempt + 1), 60)
            print(f"Rate limited. Sleeping {sleep_s}s...")
            time.sleep(sleep_s)
            continue

        if 500 <= resp.status_code < 600:
            sleep_s = min(5 * (attempt + 1), 60)
            print(f"Server error {resp.status_code}. Sleeping {sleep_s}s...")
            time.sleep(sleep_s)
            continue

        resp.raise_for_status()
        return resp.json()

    raise RuntimeError(f"Failed after {max_retries} retries: {url}")


def paginate(path, collection_key, params=None, limit=1000, sleep_s=0.25):
    params = dict(params or {})
    params["limit"] = limit

    rows = []
    cursor = None

    while True:
        if cursor:
            params["cursor"] = cursor
        else:
            params.pop("cursor", None)

        data = get_json(path, params=params)
        rows.extend(data.get(collection_key, []))

        cursor = data.get("cursor")
        if not cursor:
            break

        time.sleep(sleep_s)

    return rows


# 1. Get all currently open/live markets
live_markets = paginate(
    "/markets",
    "markets",
    params={
        "status": "open"
    },
    limit=1000
)

df_live_markets = pd.DataFrame(live_markets)

print("Number of live/open markets:", len(df_live_markets))
display(df_live_markets.head())
print(df_live_markets.columns.tolist())

KeyboardInterrupt: 

# Economics Series Only

1. Get Economics series.
2. Get events for each Economics series.
3. For each event, get historical markets.
4. Extract the market tickers.

In [ ]:
import time
import requests
import pandas as pd
from urllib.parse import urljoin

BASE_URL = "https://external-api.kalshi.com/trade-api/v2"

session = requests.Session()
session.headers.update({
    "User-Agent": "kalshi-economics-series-colab/1.0"
})


def get_json(path, params=None, max_retries=10, timeout=30):
    url = urljoin(BASE_URL + "/", path.lstrip("/"))
    params = params or {}

    for attempt in range(max_retries):
        resp = session.get(url, params=params, timeout=timeout)

        if resp.status_code == 429:
            sleep_s = min(5 * (attempt + 1), 60)
            print(f"Rate limited. Sleeping {sleep_s}s...")
            time.sleep(sleep_s)
            continue

        resp.raise_for_status()
        return resp.json()

    raise RuntimeError(f"Failed after {max_retries} retries: {url}")


def paginate(path, collection_key, params=None, limit=1000, sleep_s=0.25):
    params = dict(params or {})
    params["limit"] = limit

    rows = []
    cursor = None

    while True:
        if cursor:
            params["cursor"] = cursor
        else:
            params.pop("cursor", None)

        data = get_json(path, params=params)
        rows.extend(data.get(collection_key, []))

        cursor = data.get("cursor")
        if not cursor:
            break

        time.sleep(sleep_s)

    return rows

In [ ]:
economics_series = paginate(
    "/series",
    "series",
    params={
        "category": "Economics"
    },
    limit=1000
)

df_economics_series = pd.DataFrame(economics_series)

print("Economics series found:", len(df_economics_series))
display(df_economics_series.head())
print(df_economics_series.columns.tolist())

Economics series found: 590


,additional_prohibitions,category,contract_terms_url,contract_url,fee_multiplier,fee_type,frequency,last_updated_ts,settlement_sources,tags,ticker,title
0,[Persons who are employed by any of the Source...,Economics,https://kalshi-public-docs.s3.amazonaws.com/co...,https://kalshi-public-docs.s3.us-east-1.amazon...,1,quadratic,monthly,2026-02-26T08:50:31.901806Z,"[{'name': 'Eurostat', 'url': 'https://ec.europ...",None,KXHICP,Euro Area inflation
1,[Persons who are employed by any of the Source...,Economics,https://kalshi-public-docs.s3.amazonaws.com/co...,https://kalshi-public-docs.s3.us-east-1.amazon...,1,quadratic,annual,2026-02-26T08:50:31.901806Z,"[{'name': 'Bureau of Economic Analysis', 'url'...",[Growth],GDPUSMAX,US GDP peak
2,[Persons who are employed by any of the Source...,Economics,https://kalshi-public-docs.s3.amazonaws.com/co...,https://kalshi-public-docs.s3.us-east-1.amazon...,1,quadratic,one_off,2026-02-26T08:50:31.901806Z,"[{'name': 'Bureau of Labor Statistics', 'url':...",[Inflation],KXCPIYOYBANK,Inflation
3,[Persons who are employed by any of the Source...,Economics,https://kalshi-public-docs.s3.amazonaws.com/co...,https://kalshi-public-docs.s3.us-east-1.amazon...,1,quadratic,custom,2026-06-21T19:10:34.801666Z,[{'name': 'Bureau of Labor Statistics- Employm...,[Growth],KXTRADEDEFICIT,Trade Deficit by Date
4,[Persons who are employed by any of the Source...,Economics,https://kalshi-public-docs.s3.amazonaws.com/co...,https://kalshi-public-docs.s3.us-east-1.amazon...,1,quadratic,monthly,2026-02-26T08:50:31.901806Z,"[{'name': 'Bureau of Labor Statistics', 'url':...",[Inflation],CPIAPPAREL,CPI on apparel


['additional_prohibitions', 'category', 'contract_terms_url', 'contract_url', 'fee_multiplier', 'fee_type', 'frequency', 'last_updated_ts', 'settlement_sources', 'tags', 'ticker', 'title']


In [ ]:
import numpy as np
arr = df_economics_series.ticker.values
print(type(df_economics_series.ticker.values))

<class 'numpy.ndarray'>


In [ ]:
values, counts = np.unique(arr, return_counts=True)
print("Unique values:", values)  # [1 2 3 4]
print("Occurrences:", counts)
print(len(counts))

Unique values: ['AAAGASD' 'AAAGASM' 'AAAGASMAX' 'AAAGASMAXCA' 'AAAGASMAXTX' 'AAAGASMIN'
 'AAAGASMINCA' 'AAAGASMINTX' 'AAAGASW' 'AAAGASY' 'ACPI' 'ACPICORE'
 'ACPICORE-' 'ARGINFLATIONM' 'ARINFLATIONM' 'BANKRUPTS' 'BANKRUPTSY'
 'BIGBANKLAYOFF' 'BIGTECHLAYOFF' 'BOE' 'CHINAUSGDP' 'CNGDP' 'CNY' 'COIN'
 'COSTCOHOTDOG' 'COSTCOMEMBER' 'CPI' 'CPIAPPAREL' 'CPIAR' 'CPICN'
 'CPICORE' 'CPICOREYOY' 'CPIDELAY' 'CPIEU' 'CPIFOOD' 'CPIGAS' 'CPISHELTER'
 'CPIUSEDCAR' 'CPIYOY' 'CREDEF' 'CREDEFMAX' 'CREDEFMINMAX' 'CREDITCDEF'
 'CREDITRATING' 'DIESEL' 'DIESELM' 'DOTPLOT' 'ECB' 'ECONPATH' 'EPOP'
 'ETHETF' 'FED' 'FEDDECISION' 'FEDFACILITY' 'FEDHIKE' 'FEDMEET'
 'FEDRATEMIN' 'FRM' 'FRMMAX' 'FRMMIN' 'FTAPER' 'FXEURO' 'FXPESO' 'GAS'
 'GAS-MONTH' 'GASD' 'GASMAX' 'GASMIN' 'GDP' 'GDPCN' 'GDPEU' 'GDPUSMAX'
 'GDPUSMIN' 'GDPW' 'HOME' 'HOMEUS' 'HOMEUSY' 'HOUSELENGTH' 'HOUSESTART'
 'HPI' 'JOBLESS' 'JPY' 'KX25U3SEP' 'KX2YFOMC' 'KX30YUSTW' 'KX3MTBILL'
 'KXAAAGASD' 'KXAAAGASED' 'KXAAAGASM' 'KXAAAGASMAX' 'KXAAAGASMAXCA'
 'KXA

In [ ]:
categories_array = df_economics_series.category.values
values, counts = np.unique(categories_array, return_counts=True)
print("Unique values:", values)  # [1 2 3 4]
print("Occurrences:", counts)


Unique values: ['Companies' 'Economics' 'Financials' 'Politics' 'Science and Technology'
 'World']
Occurrences: [  1 574   5   3   1   6]


## Getting Events for the economics series


In [ ]:
import time
import pandas as pd

# If you already have a Python list, use it here:
# economics_series_tickers = ["KXCPI", "KXFED", "KXGDP"]

# If your tickers are in a DataFrame from the previous step, use this:
economics_series_tickers = (
    df_economics_series["ticker"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .tolist()
)

print("Economics series tickers:", len(economics_series_tickers))

Economics series tickers: 590


In [ ]:
import time
import requests
import pandas as pd
from urllib.parse import urljoin

BASE_URL = "https://external-api.kalshi.com/trade-api/v2"

session = requests.Session()
session.headers.update({
    "User-Agent": "kalshi-economics-events-colab/1.0"
})

# Use your existing list if you already have one:
# economics_series_tickers = ["KXCPI", "KXFED", "KXGDP"]

# Or use this if your Economics series are in df_economics_series:
economics_series_tickers = (
    df_economics_series["ticker"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .tolist()
)


def get_json(path, params=None, max_retries=10, timeout=30):
    url = urljoin(BASE_URL + "/", path.lstrip("/"))
    params = params or {}

    for attempt in range(max_retries):
        resp = session.get(url, params=params, timeout=timeout)

        if resp.status_code == 429:
            sleep_s = min(5 * (attempt + 1), 60)
            print(f"Rate limited. Sleeping {sleep_s}s...")
            time.sleep(sleep_s)
            continue

        if 500 <= resp.status_code < 600:
            sleep_s = min(5 * (attempt + 1), 60)
            print(f"Server error {resp.status_code}. Sleeping {sleep_s}s...")
            time.sleep(sleep_s)
            continue

        if resp.status_code >= 400:
            print("Bad request URL:", resp.url)
            print("Kalshi response:", resp.text)
            resp.raise_for_status()

        return resp.json()

    raise RuntimeError(f"Failed after {max_retries} retries: {url}")


def paginate(path, collection_key, params=None, limit=200, sleep_s=0.25):
    params = dict(params or {})
    params["limit"] = limit

    rows = []
    cursor = None

    while True:
        if cursor:
            params["cursor"] = cursor
        else:
            params.pop("cursor", None)

        data = get_json(path, params=params)
        rows.extend(data.get(collection_key, []))

        cursor = data.get("cursor")
        if not cursor:
            break

        time.sleep(sleep_s)

    return rows


all_economics_events = []
failed_series = []

print("Economics series tickers:", len(economics_series_tickers))

for i, series_ticker in enumerate(economics_series_tickers, start=1):
    print(f"[{i}/{len(economics_series_tickers)}] Getting events for {series_ticker}...")

    try:
        events = paginate(
            "/events",
            "events",
            params={
                "series_ticker": series_ticker,
                "with_nested_markets": False
            },
            limit=200,
            sleep_s=0.25
        )

        for event in events:
            event["series_ticker_searched"] = series_ticker
            event["category"] = "Economics"

        all_economics_events.extend(events)

    except requests.exceptions.HTTPError as e:
        print(f"Skipping {series_ticker}: {e}")
        failed_series.append(series_ticker)
        continue

    time.sleep(0.25)


df_economics_events = pd.DataFrame(all_economics_events)

print("Total event rows:", len(df_economics_events))
print("Failed series:", len(failed_series))

if not df_economics_events.empty:
    event_id_col = "event_ticker" if "event_ticker" in df_economics_events.columns else "ticker"

    df_economics_events = (
        df_economics_events
        .drop_duplicates(subset=[event_id_col])
        .sort_values(event_id_col)
        .reset_index(drop=True)
    )

    print("Unique Economics events:", len(df_economics_events))
    display(df_economics_events.head(50))

    df_economics_events.to_csv(
        "kalshi_economics_events.csv",
        index=False
    )

if failed_series:
    df_failed_series = pd.DataFrame({"failed_series_ticker": failed_series})
    display(df_failed_series)
    df_failed_series.to_csv(
        "kalshi_failed_economics_series_for_events.csv",
        index=False
    )

Economics series tickers: 590
[1/590] Getting events for AAAGASD...
[2/590] Getting events for AAAGASM...
[3/590] Getting events for AAAGASMAX...
[4/590] Getting events for AAAGASMAXCA...
[5/590] Getting events for AAAGASMAXTX...
[6/590] Getting events for AAAGASMIN...
[7/590] Getting events for AAAGASMINCA...
[8/590] Getting events for AAAGASMINTX...
[9/590] Getting events for AAAGASW...
[10/590] Getting events for AAAGASY...
[11/590] Getting events for ACPI...
[12/590] Getting events for ACPICORE...
[13/590] Getting events for ACPICORE-...
[14/590] Getting events for ARGINFLATIONM...
[15/590] Getting events for ARINFLATIONM...
[16/590] Getting events for BANKRUPTS...
[17/590] Getting events for BANKRUPTSY...
[18/590] Getting events for BIGBANKLAYOFF...
[19/590] Getting events for BIGTECHLAYOFF...
[20/590] Getting events for BOE...
[21/590] Getting events for CHINAUSGDP...
[22/590] Getting events for CNGDP...
[23/590] Getting events for CNY...
[24/590] Getting events for COIN...
[25/5

,available_on_brokers,category,collateral_return_type,event_ticker,last_updated_ts,mutually_exclusive,series_ticker,settlement_sources,strike_period,sub_title,title,series_ticker_searched,strike_date,product_metadata
0,False,Economics,,AAAGASD-23OCT03-US,2026-02-26T08:50:31.901806Z,False,KXAAAGASD,"[{'name': 'AAA', 'url': 'https://gasprices.aaa...",,"On Oct 3, 2023","US gas price up on Oct 3, 2023?",KXAAAGASD,2023-10-03T14:00:00Z,NaN
1,False,Economics,,AAAGASD-23OCT04-US,2026-02-26T08:50:31.901806Z,False,KXAAAGASD,"[{'name': 'AAA', 'url': 'https://gasprices.aaa...",,"On Oct 4, 2023","US gas price up on Oct 4, 2023?",KXAAAGASD,2023-10-04T14:00:00Z,NaN
2,False,Economics,,AAAGASD-23OCT05-US,2026-02-26T08:50:31.901806Z,False,KXAAAGASD,"[{'name': 'AAA', 'url': 'https://gasprices.aaa...",,"On Oct 5, 2023","US gas price up on Oct 5, 2023?",KXAAAGASD,2023-10-05T14:00:00Z,NaN
3,False,Economics,,AAAGASD-23OCT06-US,2026-02-26T08:50:31.901806Z,False,KXAAAGASD,"[{'name': 'AAA', 'url': 'https://gasprices.aaa...",,"On Oct 6, 2023","US gas price up on Oct 6, 2023?",KXAAAGASD,2023-10-06T14:00:00Z,NaN
4,False,Economics,,AAAGASD-23OCT07-US,2026-02-26T08:50:31.901806Z,False,KXAAAGASD,"[{'name': 'AAA', 'url': 'https://gasprices.aaa...",,"On Oct 7, 2023","US gas price up on Oct 7, 2023?",KXAAAGASD,2023-10-07T14:00:00Z,NaN
5,False,Economics,,AAAGASD-23OCT10-US,2026-02-26T08:50:31.901806Z,False,KXAAAGASD,"[{'name': 'AAA', 'url': 'https://gasprices.aaa...",,"On Oct 10, 2023","US gas price up on Oct 10, 2023?",KXAAAGASD,2023-10-10T14:00:00Z,NaN
6,False,Economics,,AAAGASD-23OCT11-US,2026-02-26T08:50:31.901806Z,False,KXAAAGASD,"[{'name': 'AAA', 'url': 'https://gasprices.aaa...",,"On Oct 11, 2023","US gas price up on Oct 11, 2023?",KXAAAGASD,2023-10-11T14:00:00Z,NaN
7,False,Economics,,AAAGASD-23OCT12-US,2026-02-26T08:50:31.901806Z,False,KXAAAGASD,"[{'name': 'AAA', 'url': 'https://gasprices.aaa...",,"On Oct 12, 2023","US gas price up on Oct 12, 2023?",KXAAAGASD,2023-10-12T14:00:00Z,NaN
8,False,Economics,,AAAGASD-23OCT13-US,2026-02-26T08:50:31.901806Z,False,KXAAAGASD,"[{'name': 'AAA', 'url': 'https://gasprices.aaa...",,"On Oct 13, 2023","US gas price up on Oct 13, 2023?",KXAAAGASD,2023-10-13T14:00:00Z,NaN
9,False,Economics,,AAAGASD-23OCT17-US,2026-02-26T08:50:31.901806Z,False,KXAAAGASD,"[{'name': 'AAA', 'url': 'https://gasprices.aaa...",,"On Oct 17, 2023","US gas price up on Oct 17, 2023?",KXAAAGASD,2023-10-17T14:00:00Z,NaN


In [ ]:
df_economics_events.shape

(2840, 14)

## For each event, filter and get historical markets only.

In [ ]:
#for an event - historical markets

import time
import requests
import pandas as pd
from urllib.parse import urljoin

BASE_URL = "https://external-api.kalshi.com/trade-api/v2"

session = requests.Session()
session.headers.update({
    "User-Agent": "kalshi-historical-markets-by-event-colab/1.0"
})


def get_json(path, params=None, max_retries=10, timeout=30):
    url = urljoin(BASE_URL + "/", path.lstrip("/"))
    params = params or {}

    for attempt in range(max_retries):
        resp = session.get(url, params=params, timeout=timeout)

        if resp.status_code == 429:
            sleep_s = min(5 * (attempt + 1), 60)
            print(f"Rate limited. Sleeping {sleep_s}s...")
            time.sleep(sleep_s)
            continue

        if 500 <= resp.status_code < 600:
            sleep_s = min(5 * (attempt + 1), 60)
            print(f"Server error {resp.status_code}. Sleeping {sleep_s}s...")
            time.sleep(sleep_s)
            continue

        if resp.status_code >= 400:
            print("Bad request URL:", resp.url)
            print("Kalshi response:", resp.text)
            resp.raise_for_status()

        return resp.json()

    raise RuntimeError(f"Failed after {max_retries} retries: {url}")


def paginate(path, collection_key, params=None, limit=200, sleep_s=0.25):
    params = dict(params or {})
    params["limit"] = limit

    rows = []
    cursor = None

    while True:
        if cursor:
            params["cursor"] = cursor
        else:
            params.pop("cursor", None)

        data = get_json(path, params=params)
        rows.extend(data.get(collection_key, []))

        cursor = data.get("cursor")
        if not cursor:
            break

        time.sleep(sleep_s)

    return rows


# df_economics_events should already exist from your previous step.
event_id_col = "event_ticker" if "event_ticker" in df_economics_events.columns else "ticker"

event_rows = (
    df_economics_events
    .dropna(subset=[event_id_col])
    .drop_duplicates(subset=[event_id_col])
    .reset_index(drop=True)
)

all_historical_markets = []
failed_events = []

print("Events to check:", len(event_rows))

for i, row in event_rows.iterrows():
    event_ticker = row[event_id_col]
    event_title = row.get("title")
    series_ticker = row.get("series_ticker", row.get("series_ticker_searched"))
    series_title = row.get("series_title")
    category = row.get("category", "Economics")

    print(f"[{i+1}/{len(event_rows)}] Getting historical markets for {event_ticker}...")

    try:
        historical_markets = paginate(
            "/historical/markets",
            "markets",
            params={
                "event_ticker": event_ticker
            },
            limit=200,
            sleep_s=0.25
        )

        for market in historical_markets:
            market["event_ticker_searched"] = event_ticker
            market["event_title"] = event_title
            market["series_ticker_searched"] = series_ticker
            market["series_title"] = series_title
            market["category"] = category

        all_historical_markets.extend(historical_markets)

    except requests.exceptions.HTTPError as e:
        print(f"Skipping {event_ticker}: {e}")
        failed_events.append(event_ticker)
        continue

    time.sleep(0.25)


df_historical_economics_markets = pd.DataFrame(all_historical_markets)

print("Historical market rows:", len(df_historical_economics_markets))
print("Failed events:", len(failed_events))

if not df_historical_economics_markets.empty:
    df_historical_economics_markets = (
        df_historical_economics_markets
        .drop_duplicates(subset=["ticker"])
        .sort_values("ticker")
        .reset_index(drop=True)
    )

    print("Unique historical market tickers:", len(df_historical_economics_markets))
    display(df_historical_economics_markets.head(50))

    df_historical_economics_markets.to_csv(
        "kalshi_historical_economics_markets_by_event.csv",
        index=False
    )

    df_historical_economics_market_tickers = df_historical_economics_markets[[
        c for c in [
            "ticker",
            "event_ticker",
            "event_ticker_searched",
            "event_title",
            "series_ticker",
            "series_ticker_searched",
            "series_title",
            "category",
            "title",
            "subtitle",
            "status",
            "open_time",
            "close_time",
            "expiration_time",
            "settlement_ts",
            "volume",
            "open_interest",
            "liquidity",
            "yes_bid",
            "yes_ask",
            "no_bid",
            "no_ask",
            "last_price"
        ]
        if c in df_historical_economics_markets.columns
    ]].copy()

    display(df_historical_economics_market_tickers.head(50))

    df_historical_economics_market_tickers.to_csv(
        "kalshi_historical_economics_market_tickers_by_event.csv",
        index=False
    )

if failed_events:
    df_failed_events = pd.DataFrame({"failed_event_ticker": failed_events})
    display(df_failed_events)
    df_failed_events.to_csv(
        "kalshi_failed_events_for_historical_markets.csv",
        index=False
    )

Events to check: 2840
[1/2840] Getting historical markets for AAAGASD-23OCT03-US...
[2/2840] Getting historical markets for AAAGASD-23OCT04-US...
[3/2840] Getting historical markets for AAAGASD-23OCT05-US...
[4/2840] Getting historical markets for AAAGASD-23OCT06-US...
[5/2840] Getting historical markets for AAAGASD-23OCT07-US...
[6/2840] Getting historical markets for AAAGASD-23OCT10-US...
[7/2840] Getting historical markets for AAAGASD-23OCT11-US...
[8/2840] Getting historical markets for AAAGASD-23OCT12-US...
[9/2840] Getting historical markets for AAAGASD-23OCT13-US...
[10/2840] Getting historical markets for AAAGASD-23OCT17-US...
[11/2840] Getting historical markets for AAAGASD-23OCT18-US...
[12/2840] Getting historical markets for AAAGASD-23OCT20-US...
[13/2840] Getting historical markets for AAAGASD-23SEP18-US...
[14/2840] Getting historical markets for AAAGASD-23SEP19-US...
[15/2840] Getting historical markets for AAAGASD-23SEP20-US...
[16/2840] Getting historical markets for A

,can_close_early,close_time,created_time,event_ticker,expected_expiration_time,expiration_time,expiration_value,floor_strike,fractional_trading_enabled,last_price_dollars,...,event_ticker_searched,event_title,series_ticker_searched,series_title,category,early_close_condition,fee_waiver_expiration_time,cap_strike,custom_strike,occurrence_datetime
0,True,2023-10-03T13:51:29.310828Z,2023-10-02T13:35:13.614692Z,AAAGASD-23OCT03-US,2023-10-04T14:00:00Z,2023-10-10T14:00:00Z,3.798,3.814,True,0.0000,...,AAAGASD-23OCT03-US,"US gas price up on Oct 3, 2023?",KXAAAGASD,None,Economics,NaN,NaN,NaN,NaN,NaN
1,True,2023-10-04T13:55:00Z,2023-10-03T13:45:56.854722Z,AAAGASD-23OCT04-US,2023-10-05T14:00:00Z,2023-10-11T14:00:00Z,3.785,3.798,True,0.0000,...,AAAGASD-23OCT04-US,"US gas price up on Oct 4, 2023?",KXAAAGASD,None,Economics,NaN,NaN,NaN,NaN,NaN
2,True,2023-10-05T13:55:00Z,2023-10-04T13:52:19.965144Z,AAAGASD-23OCT05-US,2023-10-06T14:00:00Z,2023-10-12T14:00:00Z,3.768,3.785,True,0.1100,...,AAAGASD-23OCT05-US,"US gas price up on Oct 5, 2023?",KXAAAGASD,None,Economics,NaN,NaN,NaN,NaN,NaN
3,True,2023-10-06T13:55:00Z,2023-10-05T13:53:23.418309Z,AAAGASD-23OCT06-US,2023-10-07T14:00:00Z,2023-10-13T14:00:00Z,3.746,3.768,True,0.0300,...,AAAGASD-23OCT06-US,"US gas price up on Oct 6, 2023?",KXAAAGASD,None,Economics,NaN,NaN,NaN,NaN,NaN
4,True,2023-10-07T13:55:00Z,2023-10-06T13:58:10.71795Z,AAAGASD-23OCT07-US,2023-10-08T14:00:00Z,2023-10-14T14:00:00Z,3.722,3.746,True,0.0000,...,AAAGASD-23OCT07-US,"US gas price up on Oct 7, 2023?",KXAAAGASD,None,Economics,NaN,NaN,NaN,NaN,NaN
5,True,2023-10-10T13:55:00Z,2023-10-09T14:04:05.744661Z,AAAGASD-23OCT10-US,2023-10-11T14:00:00Z,2023-10-17T14:00:00Z,3.682,3.704,True,0.1000,...,AAAGASD-23OCT10-US,"US gas price up on Oct 10, 2023?",KXAAAGASD,None,Economics,NaN,NaN,NaN,NaN,NaN
6,True,2023-10-11T13:55:00Z,2023-10-10T13:41:52.799917Z,AAAGASD-23OCT11-US,2023-10-12T14:00:00Z,2023-10-18T14:00:00Z,3.663,3.682,True,0.1200,...,AAAGASD-23OCT11-US,"US gas price up on Oct 11, 2023?",KXAAAGASD,None,Economics,NaN,NaN,NaN,NaN,NaN
7,True,2023-10-12T13:55:00Z,2023-10-11T13:57:37.582919Z,AAAGASD-23OCT12-US,2023-10-13T14:00:00Z,2023-10-19T14:00:00Z,3.646,3.663,True,0.0000,...,AAAGASD-23OCT12-US,"US gas price up on Oct 12, 2023?",KXAAAGASD,None,Economics,NaN,NaN,NaN,NaN,NaN
8,True,2023-10-13T13:55:00Z,2023-10-12T13:48:27.1847Z,AAAGASD-23OCT13-US,2023-10-14T14:00:00Z,2023-10-20T14:00:00Z,3.628,3.646,True,0.0000,...,AAAGASD-23OCT13-US,"US gas price up on Oct 13, 2023?",KXAAAGASD,None,Economics,NaN,NaN,NaN,NaN,NaN
9,True,2023-10-17T13:55:00Z,2023-10-16T15:05:20.510785Z,AAAGASD-23OCT17-US,2023-10-18T14:00:00Z,2023-10-24T14:00:00Z,3.584,3.600,True,0.5000,...,AAAGASD-23OCT17-US,"US gas price up on Oct 17, 2023?",KXAAAGASD,None,Economics,NaN,NaN,NaN,NaN,NaN


,ticker,event_ticker,event_ticker_searched,event_title,series_ticker_searched,series_title,category,title,subtitle,status,open_time,close_time,expiration_time,settlement_ts
0,AAAGASD-23OCT03-US-3.814,AAAGASD-23OCT03-US,AAAGASD-23OCT03-US,"US gas price up on Oct 3, 2023?",KXAAAGASD,None,Economics,Will average **gas prices** be above $3.814?,>$3.814,finalized,2023-10-02T14:00:00Z,2023-10-03T13:51:29.310828Z,2023-10-10T14:00:00Z,2023-10-03T15:01:00.388905Z
1,AAAGASD-23OCT04-US-3.798,AAAGASD-23OCT04-US,AAAGASD-23OCT04-US,"US gas price up on Oct 4, 2023?",KXAAAGASD,None,Economics,Will average **gas prices** be above $3.798?,>$3.798,finalized,2023-10-03T14:00:00Z,2023-10-04T13:55:00Z,2023-10-11T14:00:00Z,2023-10-04T15:15:00.329165Z
2,AAAGASD-23OCT05-US-3.785,AAAGASD-23OCT05-US,AAAGASD-23OCT05-US,"US gas price up on Oct 5, 2023?",KXAAAGASD,None,Economics,Will average **gas prices** be above $3.785?,>$3.785,finalized,2023-10-04T14:00:00Z,2023-10-05T13:55:00Z,2023-10-12T14:00:00Z,2023-10-05T15:03:00.543954Z
3,AAAGASD-23OCT06-US-3.768,AAAGASD-23OCT06-US,AAAGASD-23OCT06-US,"US gas price up on Oct 6, 2023?",KXAAAGASD,None,Economics,Will average **gas prices** be above $3.768?,>$3.768,finalized,2023-10-05T14:00:00Z,2023-10-06T13:55:00Z,2023-10-13T14:00:00Z,2023-10-06T18:52:00.869807Z
4,AAAGASD-23OCT07-US-3.746,AAAGASD-23OCT07-US,AAAGASD-23OCT07-US,"US gas price up on Oct 7, 2023?",KXAAAGASD,None,Economics,Will average **gas prices** be above $3.746?,>$3.746,finalized,2023-10-06T14:00:00Z,2023-10-07T13:55:00Z,2023-10-14T14:00:00Z,2023-10-09T16:19:00.490268Z
5,AAAGASD-23OCT10-US-3.704,AAAGASD-23OCT10-US,AAAGASD-23OCT10-US,"US gas price up on Oct 10, 2023?",KXAAAGASD,None,Economics,Will average **gas prices** be above $3.704?,>$3.704,finalized,2023-10-09T14:05:00Z,2023-10-10T13:55:00Z,2023-10-17T14:00:00Z,2023-10-11T23:22:00.365814Z
6,AAAGASD-23OCT11-US-3.682,AAAGASD-23OCT11-US,AAAGASD-23OCT11-US,"US gas price up on Oct 11, 2023?",KXAAAGASD,None,Economics,Will average **gas prices** be above $3.682?,>$3.682,finalized,2023-10-10T14:00:00Z,2023-10-11T13:55:00Z,2023-10-18T14:00:00Z,2023-10-11T23:22:00.381093Z
7,AAAGASD-23OCT12-US-3.663,AAAGASD-23OCT12-US,AAAGASD-23OCT12-US,"US gas price up on Oct 12, 2023?",KXAAAGASD,None,Economics,Will average **gas prices** be above $3.663?,>$3.663,finalized,2023-10-11T14:00:00Z,2023-10-12T13:55:00Z,2023-10-19T14:00:00Z,2023-10-12T16:22:00.630388Z
8,AAAGASD-23OCT13-US-3.646,AAAGASD-23OCT13-US,AAAGASD-23OCT13-US,"US gas price up on Oct 13, 2023?",KXAAAGASD,None,Economics,Will average **gas prices** be above $3.646?,>$3.646,finalized,2023-10-12T14:00:00Z,2023-10-13T13:55:00Z,2023-10-20T14:00:00Z,2023-10-13T22:07:00.812829Z
9,AAAGASD-23OCT17-US-3.600,AAAGASD-23OCT17-US,AAAGASD-23OCT17-US,"US gas price up on Oct 17, 2023?",KXAAAGASD,None,Economics,Will average **gas prices** be above $3.600?,>$3.600,finalized,2023-10-16T15:10:00Z,2023-10-17T13:55:00Z,2023-10-24T14:00:00Z,2023-10-17T22:19:00.332769Z


In [ ]:
df_historical_economics_market_tickers.shape

(12606, 14)

In [ ]:
df_historical_economics_market_tickers.columns

Index(['ticker', 'event_ticker', 'event_ticker_searched', 'event_title',
       'series_ticker_searched', 'series_title', 'category', 'title',
       'subtitle', 'status', 'open_time', 'close_time', 'expiration_time',
       'settlement_ts'],
      dtype='object')

In [ ]:
df_historical_economics_market_tickers.tail(50)

,ticker,event_ticker,event_ticker_searched,event_title,series_ticker_searched,series_title,category,title,subtitle,status,open_time,close_time,expiration_time,settlement_ts
12440,U3-24MAR-T3.9,U3-24MAR,U3-24MAR,Unemployment in Mar 2024?,KXU3,None,Economics,Will the unemployment rate (U-3) be above 3.9%...,3.9%:: Feb 2024 result: 3.9,finalized,2024-02-23T15:00:00Z,2024-04-05T12:25:00Z,2024-04-12T14:00:00Z,2024-04-05T16:41:01.355403Z
12441,U3-24MAR-T4.0,U3-24MAR,U3-24MAR,Unemployment in Mar 2024?,KXU3,None,Economics,Will the unemployment rate (U-3) be above 4.0%...,4.0%,finalized,2024-02-23T15:00:00Z,2024-04-05T12:25:00Z,2024-04-12T14:00:00Z,2024-04-05T16:41:01.379918Z
12442,U3-24MAR-T4.1,U3-24MAR,U3-24MAR,Unemployment in Mar 2024?,KXU3,None,Economics,Will the unemployment rate (U-3) be above 4.1%...,4.1%,finalized,2024-02-23T15:00:00Z,2024-04-05T12:25:00Z,2024-04-12T14:00:00Z,2024-04-05T16:41:01.439552Z
12443,U3-24MAR-T4.2,U3-24MAR,U3-24MAR,Unemployment in Mar 2024?,KXU3,None,Economics,Will the unemployment rate (U-3) be above 4.2%...,4.2%,finalized,2024-02-23T15:00:00Z,2024-04-05T12:25:00Z,2024-04-12T14:00:00Z,2024-04-05T16:41:01.425181Z
12444,U3-24MAY-T3.3,U3-24MAY,U3-24MAY,Unemployment in May 2024?,KXU3,None,Economics,Will the unemployment rate (U-3) be above 3.3%...,3.3%,finalized,2024-04-19T14:00:00Z,2024-06-07T12:25:00Z,2024-06-14T14:00:00Z,2024-06-07T16:03:00.847618Z
12445,U3-24MAY-T3.4,U3-24MAY,U3-24MAY,Unemployment in May 2024?,KXU3,None,Economics,Will the unemployment rate (U-3) be above 3.4%...,3.4%,finalized,2024-04-19T14:00:00Z,2024-06-07T12:25:00Z,2024-06-14T14:00:00Z,2024-06-07T16:07:00.548103Z
12446,U3-24MAY-T3.5,U3-24MAY,U3-24MAY,Unemployment in May 2024?,KXU3,None,Economics,Will the unemployment rate (U-3) be above 3.5%...,3.5%,finalized,2024-04-19T14:00:00Z,2024-06-07T12:25:00Z,2024-06-14T14:00:00Z,2024-06-07T16:07:00.56307Z
12447,U3-24MAY-T3.6,U3-24MAY,U3-24MAY,Unemployment in May 2024?,KXU3,None,Economics,Will the unemployment rate (U-3) be above 3.6%...,3.6%,finalized,2024-04-19T14:00:00Z,2024-06-07T12:25:00Z,2024-06-14T14:00:00Z,2024-06-07T16:07:00.572593Z
12448,U3-24MAY-T3.7,U3-24MAY,U3-24MAY,Unemployment in May 2024?,KXU3,None,Economics,Will the unemployment rate (U-3) be above 3.7%...,3.7%,finalized,2024-04-19T14:00:00Z,2024-06-07T12:25:00Z,2024-06-14T14:00:00Z,2024-06-07T16:07:01.148081Z
12449,U3-24MAY-T3.8,U3-24MAY,U3-24MAY,Unemployment in May 2024?,KXU3,None,Economics,Will the unemployment rate (U-3) be above 3.8%...,3.8%,finalized,2024-04-19T14:00:00Z,2024-06-07T12:25:00Z,2024-06-14T14:00:00Z,2024-06-07T16:03:00.868117Z


In [ ]:
df_historical_economics_market_tickers.open_time.min()

'2021-06-30T14:00:00Z'

In [ ]:
df_historical_economics_market_tickers.close_time.min()

'2021-07-01T23:00:00Z'

In [ ]:
df_historical_economics_market_tickers.close_time.max()

'2026-04-23T14:32:14Z'

In [ ]:
df_historical_economics_market_tickers.open_time.max()

'2026-04-22T12:40:00Z'

In [ ]:
df_historical_economics_market_tickers.ticker.values

array(['AAAGASD-23OCT03-US-3.814', 'AAAGASD-23OCT04-US-3.798',
       'AAAGASD-23OCT05-US-3.785', ..., 'USEDCAR-23JUL-T4',
       'WHEAT-22APR-T7.50', 'WRECSS-26-GER'], dtype=object)

# Specific Attributes

In [ ]:
import time
import pandas as pd

# Your NumPy array of historical market tickers
historical_market_tickers = df_historical_economics_market_tickers.ticker.values

wanted_attrs = [
    "ticker",
    "event_ticker",
    "series_ticker",
    "yes_sub_title",
    "no_sub_title",
    "created_time",
    "open_time",
    "close_time",
    "status",
    "result",
    "notional_value_dollars",
    "rules_primary",
    "rules_secondary",
    "title",
    "volume_fp",
    "volume_24h_fp",
]

market_rows = []
failed_tickers = []

for i, ticker in enumerate(historical_market_tickers, start=1):
    ticker = str(ticker).strip()

    print(f"[{i}/{len(historical_market_tickers)}] Getting historical market {ticker}...")

    try:
        data = get_json(f"/historical/markets/{ticker}")
        market = data.get("market", data)

        row = {attr: market.get(attr) for attr in wanted_attrs}

        # Fallback: if series_ticker is not returned by the endpoint,
        # try to recover it from your existing DataFrame.
        if not row.get("series_ticker") and "series_ticker_searched" in df_historical_economics_market_tickers.columns:
            match = df_historical_economics_market_tickers[
                df_historical_economics_market_tickers["ticker"].astype(str).str.strip() == ticker
            ]

            if not match.empty:
                row["series_ticker"] = match.iloc[0].get("series_ticker_searched")

        market_rows.append(row)

    except Exception as e:
        print(f"Skipping {ticker}: {e}")
        failed_tickers.append({
            "ticker": ticker,
            "error": str(e)
        })

    time.sleep(0.25)

df_historical_market_selected_attributes = pd.DataFrame(
    market_rows,
    columns=wanted_attrs
)

display(df_historical_market_selected_attributes)

print("Markets extracted:", len(df_historical_market_selected_attributes))
print("Failed tickers:", len(failed_tickers))

df_historical_market_selected_attributes.to_csv(
    "kalshi_historical_market_selected_attributes.csv",
    index=False
)

if failed_tickers:
    df_failed_historical_market_tickers = pd.DataFrame(failed_tickers)
    display(df_failed_historical_market_tickers)

    df_failed_historical_market_tickers.to_csv(
        "kalshi_failed_historical_market_tickers.csv",
        index=False
    )

[1/12606] Getting historical market AAAGASD-23OCT03-US-3.814...
[2/12606] Getting historical market AAAGASD-23OCT04-US-3.798...
[3/12606] Getting historical market AAAGASD-23OCT05-US-3.785...
[4/12606] Getting historical market AAAGASD-23OCT06-US-3.768...
[5/12606] Getting historical market AAAGASD-23OCT07-US-3.746...
[6/12606] Getting historical market AAAGASD-23OCT10-US-3.704...
[7/12606] Getting historical market AAAGASD-23OCT11-US-3.682...
[8/12606] Getting historical market AAAGASD-23OCT12-US-3.663...
[9/12606] Getting historical market AAAGASD-23OCT13-US-3.646...
[10/12606] Getting historical market AAAGASD-23OCT17-US-3.600...
[11/12606] Getting historical market AAAGASD-23OCT18-US-3.584...
[12/12606] Getting historical market AAAGASD-23OCT20-US-3.565...
[13/12606] Getting historical market AAAGASD-23SEP18-US-3.866...
[14/12606] Getting historical market AAAGASD-23SEP19-US-3.881...
[15/12606] Getting historical market AAAGASD-23SEP20-US-3.880...
[16/12606] Getting historical mark

# Candlesticks for a historical market

https://docs.kalshi.com/api-reference/historical/get-historical-market$0



this info can be extracted, given a ticker:

{
  "market": {
    "ticker": "<string>",

    "event_ticker": "<string>",

    "yes_sub_title": "<string>",

    "no_sub_title": "<string>",

    "created_time": "2023-11-07T05:31:56Z",

    "updated_time": "2023-11-07T05:31:56Z",

    "open_time": "2023-11-07T05:31:56Z",

    "close_time": "2023-11-07T05:31:56Z",

    "latest_expiration_time": "2023-11-07T05:31:56Z",

    "settlement_timer_seconds": 123,

    "yes_bid_dollars": "0.5600",

    "yes_bid_size_fp": "10.00",

    "yes_ask_dollars": "0.5600",

    "yes_ask_size_fp": "10.00",

    "no_bid_dollars": "0.5600",

    "no_ask_dollars": "0.5600",

    "last_price_dollars": "0.5600",

    "volume_fp": "10.00",

    "volume_24h_fp": "10.00",

    "can_close_early": true,

    "fractional_trading_enabled": true,

    "open_interest_fp": "10.00",

    "notional_value_dollars": "0.5600",

    "previous_yes_bid_dollars": "0.5600",

    "previous_yes_ask_dollars": "0.5600",

    "previous_price_dollars": "0.5600",

    "liquidity_dollars": "0.5600",

    "expiration_value": "<string>",

    "rules_primary": "<string>",

    "rules_secondary": "<string>",

    "price_level_structure": "<string>",

    "price_ranges": [
      {
        "start": "<string>",
        "end": "<string>",
        "step": "<string>"
      }
    ],
    "title": "<string>",
    "subtitle": "<string>",
    "expected_expiration_time": "2023-11-07T05:31:56Z",
    "expiration_time": "2023-11-07T05:31:56Z",
    "response_price_units": "usd_cent",
    "settlement_value_dollars": "0.5600",
    "settlement_ts": "2023-11-07T05:31:56Z",
    "occurrence_datetime": "2023-11-07T05:31:56Z",
    "fee_waiver_expiration_time": "2023-11-07T05:31:56Z",
    "early_close_condition": "<string>",
    "floor_strike": 123,
    "cap_strike": 123,
    "functional_strike": "<string>",
    "custom_strike": {},
    "mve_collection_ticker": "<string>",
    "mve_selected_legs": [
      {
        "event_ticker": "<string>",
        "market_ticker": "<string>",
        "side": "<string>",
        "yes_settlement_value_dollars": "0.5600"
      }
    ],
    "primary_participant_key": "<string>",
    "is_provisional": true,
    "exchange_index": 0
  }
}


In [ ]:
import requests
import pandas as pd
from urllib.parse import urljoin
import time

BASE_URL = "https://external-api.kalshi.com/trade-api/v2"

session = requests.Session()
session.headers.update({
    "User-Agent": "kalshi-daily-candlesticks-colab/1.0"
})

def get_json(path, params=None, max_retries=10, timeout=30):
    url = urljoin(BASE_URL + "/", path.lstrip("/"))
    params = params or {}

    for attempt in range(max_retries):
        resp = session.get(url, params=params, timeout=timeout)

        if resp.status_code == 429:
            sleep_s = min(5 * (attempt + 1), 60)
            print(f"Rate limited. Sleeping {sleep_s}s...")
            time.sleep(sleep_s)
            continue

        if resp.status_code >= 400:
            print("Bad request URL:", resp.url)
            print("Kalshi response:", resp.text)
            resp.raise_for_status()

        return resp.json()

    raise RuntimeError(f"Failed after {max_retries} retries: {url}")


# Put your historical market ticker here
market_ticker = "WRECSS-26-GER"

# Get market details so we can use its open/close dates
market_data = get_json(f"/historical/markets/{market_ticker}")
market = market_data.get("market", market_data)

open_time = pd.to_datetime(market.get("open_time"), utc=True, errors="coerce")
close_time = pd.to_datetime(market.get("close_time"), utc=True, errors="coerce")

start_ts = int(open_time.timestamp())
end_ts = int(close_time.timestamp())

# Get daily candlesticks
candles_data = get_json(
    f"/historical/markets/{market_ticker}/candlesticks",
    params={
        "start_ts": start_ts,
        "end_ts": end_ts,
        "period_interval": 1440
    }
)

df_daily_candlesticks = pd.DataFrame(candles_data.get("candlesticks", []))

display(df_daily_candlesticks)

print("Market ticker:", market_ticker)
print("Open time:", open_time)
print("Close time:", close_time)
print("Daily candlesticks:", len(df_daily_candlesticks))

,end_period_ts,open_interest,price,volume,yes_ask,yes_bid
0,1713931200,1000.00,"{'close': '0.1400', 'high': '0.1400', 'low': '...",1000.00,"{'close': '0.5000', 'high': '1.0000', 'low': '...","{'close': '0.0400', 'high': '0.0400', 'low': '..."
1,1714017600,1040.00,"{'close': '0.2400', 'high': '0.2400', 'low': '...",40.00,"{'close': '0.2400', 'high': '0.5000', 'low': '...","{'close': '0.1700', 'high': '0.1700', 'low': '..."
2,1714104000,1120.00,"{'close': '0.2400', 'high': '0.2400', 'low': '...",80.00,"{'close': '0.2400', 'high': '0.3600', 'low': '...","{'close': '0.1700', 'high': '0.1700', 'low': '..."
3,1714363200,1120.00,"{'close': '0.1700', 'high': '0.1700', 'low': '...",1.00,"{'close': '0.2400', 'high': '0.2400', 'low': '...","{'close': '0.1700', 'high': '0.1700', 'low': '..."
4,1714881600,1139.00,"{'close': '0.2400', 'high': '0.2400', 'low': '...",19.00,"{'close': '0.2400', 'high': '0.2400', 'low': '...","{'close': '0.1800', 'high': '0.1800', 'low': '..."
...,...,...,...,...,...,...
407,1761796800,3053.00,"{'close': '0.5200', 'high': '0.5200', 'low': '...",3.00,"{'close': '0.5200', 'high': '0.5200', 'low': '...","{'close': '0.4700', 'high': '0.4700', 'low': '..."
408,1761883200,5125.00,"{'close': '0.6700', 'high': '0.6700', 'low': '...",2072.00,"{'close': '0.6400', 'high': '0.8400', 'low': '...","{'close': '0.5800', 'high': '0.5800', 'low': '..."
409,1761969600,6035.00,"{'close': '0.9200', 'high': '0.9600', 'low': '...",5349.00,"{'close': '0.9200', 'high': '0.9700', 'low': '...","{'close': '0.8600', 'high': '0.9300', 'low': '..."
410,1762056000,6008.00,"{'close': '0.8500', 'high': '0.9300', 'low': '...",57.00,"{'close': '0.8900', 'high': '0.9300', 'low': '...","{'close': '0.8500', 'high': '0.8800', 'low': '..."


Market ticker: WRECSS-26-GER
Open time: 2024-04-23 12:00:00+00:00
Close time: 2025-11-03 18:12:29.373203+00:00
Daily candlesticks: 412


In [ ]:
#Getting Economics category tickers for all events CODE FROM EARLIER - Check
import requests
import pandas as pd
import time
from typing import Optional, Dict, Any, List

BASE_URL = "https://external-api.kalshi.com/trade-api/v2"
CATEGORY = "Economics"


def kalshi_get(path: str, params: Optional[Dict[str, Any]] = None, max_retries: int = 3) -> Dict[str, Any]:
    """
    Simple GET helper with basic retry handling.
    Public market-data endpoints do not require authentication.
    """
    url = f"{BASE_URL}{path}"
    params = params or {}

    for attempt in range(max_retries):
        try:
            response = requests.get(url, params=params, timeout=30)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            if attempt == max_retries - 1:
                raise
            time.sleep(1.5 * (attempt + 1))


def get_series_by_category(category: str = "Economics") -> List[Dict[str, Any]]:
    """
    Get all Kalshi series in a category.
    """
    data = kalshi_get(
        "/series",
        params={
            "category": category,
            "include_product_metadata": "true",
            "include_volume": "true",
        },
    )
    return data.get("series", [])


def get_events_for_series(
    series_ticker: str,
    status: Optional[str] = None,
    with_nested_markets: bool = True,
) -> List[Dict[str, Any]]:
    """
    Get all events for one series ticker, handling cursor pagination.

    status options:
      None      -> all statuses
      "unopened"
      "open"
      "closed"
      "settled"
    """
    all_events = []
    cursor = None

    while True:
        params = {
            "series_ticker": series_ticker,
            "limit": 200,
            "with_nested_markets": str(with_nested_markets).lower(),
        }

        if status is not None:
            params["status"] = status

        if cursor:
            params["cursor"] = cursor

        data = kalshi_get("/events", params=params)

        events = data.get("events", [])
        all_events.extend(events)

        cursor = data.get("cursor")
        if not cursor:
            break

    return all_events


def economics_events_markets_df(
    category: str = "Economics",
    status: Optional[str] = None,
) -> pd.DataFrame:
    """
    Returns one row per market.

    If an event has no nested markets returned, it still creates one row
    with market fields set to None so you do not lose the event.
    """
    series_list = get_series_by_category(category)
    rows = []

    print(f"Found {len(series_list)} series in category={category!r}")

    for i, series in enumerate(series_list, start=1):
        series_ticker = series.get("ticker")
        series_title = series.get("title")

        print(f"[{i}/{len(series_list)}] Fetching events for {series_ticker}: {series_title}")

        events = get_events_for_series(
            series_ticker=series_ticker,
            status=status,
            with_nested_markets=True,
        )

        for event in events:
            markets = event.get("markets") or []

            event_base = {
                "category": event.get("category"),
                "series_ticker": event.get("series_ticker"),
                "series_title": series.get("title"),
                "series_frequency": series.get("frequency"),
                "series_tags": series.get("tags"),
                "event_ticker": event.get("event_ticker"),
                "event_title": event.get("title"),
                "event_sub_title": event.get("sub_title"),
                "event_strike_date": event.get("strike_date"),
                "event_strike_period": event.get("strike_period"),
                "event_last_updated_ts": event.get("last_updated_ts"),
                "event_product_metadata": event.get("product_metadata"),
            }

            if not markets:
                rows.append({
                    **event_base,
                    "market_ticker": None,
                    "market_question": None,
                    "market_title": None,
                    "market_subtitle": None,
                    "yes_sub_title": None,
                    "no_sub_title": None,
                    "market_status_open_time": None,
                    "market_close_time": None,
                    "market_expiration_time": None,
                    "yes_bid_dollars": None,
                    "yes_ask_dollars": None,
                    "no_bid_dollars": None,
                    "no_ask_dollars": None,
                    "last_price_dollars": None,
                    "volume_fp": None,
                    "volume_24h_fp": None,
                    "open_interest_fp": None,
                    "liquidity_dollars": None,
                    "rules_primary": None,
                    "rules_secondary": None,
                })
            else:
                for market in markets:
                    rows.append({
                        **event_base,
                        "market_ticker": market.get("ticker"),
                        # Kalshi's user-facing market question is usually the market title.
                        "market_question": market.get("title"),
                        "market_title": market.get("title"),
                        "market_subtitle": market.get("subtitle"),
                        "yes_sub_title": market.get("yes_sub_title"),
                        "no_sub_title": market.get("no_sub_title"),
                        "market_status_open_time": market.get("open_time"),
                        "market_close_time": market.get("close_time"),
                        "market_expiration_time": market.get("expiration_time"),
                        "yes_bid_dollars": market.get("yes_bid_dollars"),
                        "yes_ask_dollars": market.get("yes_ask_dollars"),
                        "no_bid_dollars": market.get("no_bid_dollars"),
                        "no_ask_dollars": market.get("no_ask_dollars"),
                        "last_price_dollars": market.get("last_price_dollars"),
                        "volume_fp": market.get("volume_fp"),
                        "volume_24h_fp": market.get("volume_24h_fp"),
                        "open_interest_fp": market.get("open_interest_fp"),
                        "liquidity_dollars": market.get("liquidity_dollars"),
                        "rules_primary": market.get("rules_primary"),
                        "rules_secondary": market.get("rules_secondary"),
                    })

    df = pd.DataFrame(rows)

    # Optional cleanup: remove duplicate market rows if pagination/API overlap ever occurs.
    if not df.empty:
        df = df.drop_duplicates(
            subset=["series_ticker", "event_ticker", "market_ticker"],
            keep="first",
        ).reset_index(drop=True)

    return df


# Use status=None for all events regardless of status.
# Use status="open" if you only want currently open economics markets.
df = economics_events_markets_df(
    category="Economics",
    status=None,
)

print("Rows:", len(df))
print("Unique series:", df["series_ticker"].nunique() if not df.empty else 0)
print("Unique events:", df["event_ticker"].nunique() if not df.empty else 0)
print("Unique markets:", df["market_ticker"].nunique() if not df.empty else 0)

Found 589 series in category='Economics'
[1/589] Fetching events for KXSHELTERCPI: US shelter CPI in [month]
[2/589] Fetching events for KXACPI: US annual inflation
[3/589] Fetching events for U3MAX: Unemployment spike
[4/589] Fetching events for KXPPICPI: PPI YoY exceeds CPI YoY for [time period]
[5/589] Fetching events for KXGOVTSPEND: Government budget increases
[6/589] Fetching events for KXJPCPIYOY: Japan Inflation Rate YoY
[7/589] Fetching events for KXPAYROLLS: Jobs numbers
[8/589] Fetching events for KXTECHLAYOFF: Tech layoffs up
[9/589] Fetching events for KXCBDECISIONEU: EU CENTRAL BANK POLICY INTEREST RATE
[10/589] Fetching events for GDPW: Global GDP growth
[11/589] Fetching events for PPISEMI: Semiconductor PPI
[12/589] Fetching events for KXHOMEUSY: US housing prices up
[13/589] Fetching events for KXECONSTATCORECPIYOY: year over year core inflation
[14/589] Fetching events for KXMORTGAGERATE: Mortgage Rate
[15/589] Fetching events for LOWESTRATE: Fed funds lowest rate
[1

# GET /historical/markets/{ticker}

In [ ]:
import requests
import pandas as pd
import json
from urllib.parse import urljoin

BASE_URL = "https://external-api.kalshi.com/trade-api/v2"

market_ticker = "WRECSS-26-GER"

session = requests.Session()
session.headers.update({
    "User-Agent": "kalshi-historical-market-attributes-test/1.0"
})

url = urljoin(BASE_URL + "/", f"historical/markets/{market_ticker}")

resp = session.get(url, timeout=30)

print("URL:", resp.url)
print("Status code:", resp.status_code)

try:
    data = resp.json()
except Exception:
    data = None
    print(resp.text)

if resp.status_code >= 400:
    print("Kalshi error response:")
    print(json.dumps(data, indent=2))
else:
    print("Raw response:")
    print(json.dumps(data, indent=2)[:5000])

    # Kalshi may return {"market": {...}}
    market = data.get("market", data)

    # Print all available attributes
    print("\nAttributes returned:")
    for key in sorted(market.keys()):
        print(key)

    # Put the market into a one-row DataFrame
    df_historical_market_attributes = pd.DataFrame([market])

    display(df_historical_market_attributes)

    print("\nNumber of attributes:", len(df_historical_market_attributes.columns))
    print("\nColumns:")
    print(df_historical_market_attributes.columns.tolist())

URL: https://external-api.kalshi.com/trade-api/v2/historical/markets/WRECSS-26-GER
Status code: 200
Raw response:
{
  "market": {
    "can_close_early": true,
    "close_time": "2025-11-03T18:12:29.373203Z",
    "created_time": "2024-04-22T23:39:53.037748Z",
    "custom_strike": {
      "Country": "Germany"
    },
    "early_close_condition": "If this event occurs, the market will close the following 10:00 AM ET.",
    "event_ticker": "WRECSS-26",
    "expected_expiration_time": "2026-12-31T15:00:00Z",
    "expiration_time": "2027-12-31T15:00:00Z",
    "expiration_value": "Yes",
    "fractional_trading_enabled": true,
    "last_price_dollars": "0.8500",
    "latest_expiration_time": "2027-12-31T15:00:00Z",
    "liquidity_dollars": "0.0000",
    "market_type": "binary",
    "no_ask_dollars": "1.0000",
    "no_bid_dollars": "0.0000",
    "no_sub_title": "Germany",
    "notional_value_dollars": "1.0000",
    "open_interest_fp": "0.00",
    "open_time": "2024-04-23T12:00:00Z",
    "previou

,can_close_early,close_time,created_time,custom_strike,early_close_condition,event_ticker,expected_expiration_time,expiration_time,expiration_value,fractional_trading_enabled,...,ticker,title,updated_time,volume_24h_fp,volume_fp,yes_ask_dollars,yes_ask_size_fp,yes_bid_dollars,yes_bid_size_fp,yes_sub_title
0,True,2025-11-03T18:12:29.373203Z,2024-04-22T23:39:53.037748Z,{'Country': 'Germany'},"If this event occurs, the market will close th...",WRECSS-26,2026-12-31T15:00:00Z,2027-12-31T15:00:00Z,Yes,True,...,WRECSS-26-GER,Which countries will have recessions?,2026-03-23T22:26:48.576397Z,0.00,22433.00,1.0000,0.00,0.0000,0.00,Germany



Number of attributes: 45

Columns:
['can_close_early', 'close_time', 'created_time', 'custom_strike', 'early_close_condition', 'event_ticker', 'expected_expiration_time', 'expiration_time', 'expiration_value', 'fractional_trading_enabled', 'last_price_dollars', 'latest_expiration_time', 'liquidity_dollars', 'market_type', 'no_ask_dollars', 'no_bid_dollars', 'no_sub_title', 'notional_value_dollars', 'open_interest_fp', 'open_time', 'previous_price_dollars', 'previous_yes_ask_dollars', 'previous_yes_bid_dollars', 'price_level_structure', 'price_ranges', 'response_price_units', 'result', 'rules_primary', 'rules_secondary', 'settlement_timer_seconds', 'settlement_ts', 'settlement_value_dollars', 'status', 'strike_type', 'subtitle', 'ticker', 'title', 'updated_time', 'volume_24h_fp', 'volume_fp', 'yes_ask_dollars', 'yes_ask_size_fp', 'yes_bid_dollars', 'yes_bid_size_fp', 'yes_sub_title']


In [ ]:
import requests
import pandas as pd
import json
from urllib.parse import urljoin

BASE_URL = "https://external-api.kalshi.com/trade-api/v2"
market_ticker = "WRECSS-26-GER"

session = requests.Session()
session.headers.update({
    "User-Agent": "kalshi-historical-market-attributes-test/1.0"
})

# Call GET /historical/markets/{ticker}
url = urljoin(BASE_URL + "/", f"historical/markets/{market_ticker}")

resp = session.get(url, timeout=30)

print("URL:", resp.url)
print("Status code:", resp.status_code)

try:
    data = resp.json()
except Exception:
    print("Non-JSON response:")
    print(resp.text)
    raise

if resp.status_code >= 400:
    print("Kalshi error response:")
    print(json.dumps(data, indent=2))
    raise ValueError(f"{market_ticker} was not found in /historical/markets/{{ticker}}")

# Kalshi usually returns {"market": {...}}
market = data.get("market", data)

print("\nRaw response:")
print(json.dumps(data, indent=2)[:10000])

print("\nAttributes returned:")
for key in sorted(market.keys()):
    print(key)

# DataFrame 1: one row with all market attributes as columns
df_historical_market = pd.DataFrame([market])

display(df_historical_market)

print("\nNumber of attributes:", len(df_historical_market.columns))
print("Columns:")
print(df_historical_market.columns.tolist())

# DataFrame 2: easier-to-read attribute/value table
df_historical_market_attributes = pd.DataFrame([
    {
        "attribute": key,
        "value": value
    }
    for key, value in market.items()
])

display(df_historical_market_attributes)

# Save both
df_historical_market.to_csv(
    f"historical_market_{market_ticker}_all_attributes.csv",
    index=False
)

df_historical_market_attributes.to_csv(
    f"historical_market_{market_ticker}_attribute_values.csv",
    index=False
)

URL: https://external-api.kalshi.com/trade-api/v2/historical/markets/WRECSS-26-GER
Status code: 200

Raw response:
{
  "market": {
    "can_close_early": true,
    "close_time": "2025-11-03T18:12:29.373203Z",
    "created_time": "2024-04-22T23:39:53.037748Z",
    "custom_strike": {
      "Country": "Germany"
    },
    "early_close_condition": "If this event occurs, the market will close the following 10:00 AM ET.",
    "event_ticker": "WRECSS-26",
    "expected_expiration_time": "2026-12-31T15:00:00Z",
    "expiration_time": "2027-12-31T15:00:00Z",
    "expiration_value": "Yes",
    "fractional_trading_enabled": true,
    "last_price_dollars": "0.8500",
    "latest_expiration_time": "2027-12-31T15:00:00Z",
    "liquidity_dollars": "0.0000",
    "market_type": "binary",
    "no_ask_dollars": "1.0000",
    "no_bid_dollars": "0.0000",
    "no_sub_title": "Germany",
    "notional_value_dollars": "1.0000",
    "open_interest_fp": "0.00",
    "open_time": "2024-04-23T12:00:00Z",
    "previo

,can_close_early,close_time,created_time,custom_strike,early_close_condition,event_ticker,expected_expiration_time,expiration_time,expiration_value,fractional_trading_enabled,...,ticker,title,updated_time,volume_24h_fp,volume_fp,yes_ask_dollars,yes_ask_size_fp,yes_bid_dollars,yes_bid_size_fp,yes_sub_title
0,True,2025-11-03T18:12:29.373203Z,2024-04-22T23:39:53.037748Z,{'Country': 'Germany'},"If this event occurs, the market will close th...",WRECSS-26,2026-12-31T15:00:00Z,2027-12-31T15:00:00Z,Yes,True,...,WRECSS-26-GER,Which countries will have recessions?,2026-03-23T22:26:48.576397Z,0.00,22433.00,1.0000,0.00,0.0000,0.00,Germany



Number of attributes: 45
Columns:
['can_close_early', 'close_time', 'created_time', 'custom_strike', 'early_close_condition', 'event_ticker', 'expected_expiration_time', 'expiration_time', 'expiration_value', 'fractional_trading_enabled', 'last_price_dollars', 'latest_expiration_time', 'liquidity_dollars', 'market_type', 'no_ask_dollars', 'no_bid_dollars', 'no_sub_title', 'notional_value_dollars', 'open_interest_fp', 'open_time', 'previous_price_dollars', 'previous_yes_ask_dollars', 'previous_yes_bid_dollars', 'price_level_structure', 'price_ranges', 'response_price_units', 'result', 'rules_primary', 'rules_secondary', 'settlement_timer_seconds', 'settlement_ts', 'settlement_value_dollars', 'status', 'strike_type', 'subtitle', 'ticker', 'title', 'updated_time', 'volume_24h_fp', 'volume_fp', 'yes_ask_dollars', 'yes_ask_size_fp', 'yes_bid_dollars', 'yes_bid_size_fp', 'yes_sub_title']


,attribute,value
0,can_close_early,True
1,close_time,2025-11-03T18:12:29.373203Z
2,created_time,2024-04-22T23:39:53.037748Z
3,custom_strike,{'Country': 'Germany'}
4,early_close_condition,"If this event occurs, the market will close th..."
5,event_ticker,WRECSS-26
6,expected_expiration_time,2026-12-31T15:00:00Z
7,expiration_time,2027-12-31T15:00:00Z
8,expiration_value,Yes
9,fractional_trading_enabled,True


# GET /historical/markets/{ticker}/candlesticks

In [ ]:
import requests
import pandas as pd
import json
from urllib.parse import urljoin

BASE_URL = "https://external-api.kalshi.com/trade-api/v2"

market_ticker = "WRECSS-26-GER"

session = requests.Session()
session.headers.update({
    "User-Agent": "kalshi-historical-candlesticks-test/1.0"
})


def get_json_raw(path, params=None):
    url = urljoin(BASE_URL + "/", path.lstrip("/"))
    resp = session.get(url, params=params or {}, timeout=30)

    print("URL:", resp.url)
    print("Status code:", resp.status_code)

    try:
        data = resp.json()
    except Exception:
        data = None
        print(resp.text)
        return resp, data

    if resp.status_code >= 400:
        print("Kalshi error response:")
        print(json.dumps(data, indent=2))
    else:
        print("Top-level keys:", list(data.keys()) if isinstance(data, dict) else type(data))

    return resp, data


# 1. Get historical market details first
market_resp, market_data = get_json_raw(f"/historical/markets/{market_ticker}")

if market_resp.status_code >= 400:
    raise ValueError("Market was not found in /historical/markets/{ticker}. Cannot request historical candlesticks.")

market = market_data.get("market", market_data)

open_time = pd.to_datetime(market.get("open_time"), utc=True, errors="coerce")
close_time = pd.to_datetime(
    market.get("close_time") or market.get("expiration_time") or market.get("settlement_ts"),
    utc=True,
    errors="coerce"
)

start_ts = int(open_time.timestamp())
end_ts = int(close_time.timestamp())

print("Market title:", market.get("title"))
print("Open time:", open_time)
print("End time:", close_time)

# 2. Get daily historical candlesticks
candles_resp, candles_data = get_json_raw(
    f"/historical/markets/{market_ticker}/candlesticks",
    params={
        "start_ts": start_ts,
        "end_ts": end_ts,
        "period_interval": 1440
    }
)

if candles_resp.status_code < 400:
    print("Raw candlestick response:")
    print(json.dumps(candles_data, indent=2)[:5000])

    df_historical_candlesticks = pd.DataFrame(
        candles_data.get("candlesticks", [])
    )

    print("Candlestick rows:", len(df_historical_candlesticks))
    print("Candlestick columns:", df_historical_candlesticks.columns.tolist())

    display(df_historical_candlesticks)

    df_historical_candlesticks.to_csv(
        f"kalshi_historical_daily_candlesticks_{market_ticker}.csv",
        index=False
    )

URL: https://external-api.kalshi.com/trade-api/v2/historical/markets/WRECSS-26-GER
Status code: 200
Top-level keys: ['market']
Market title: Which countries will have recessions?
Open time: 2024-04-23 12:00:00+00:00
End time: 2025-11-03 18:12:29.373203+00:00
URL: https://external-api.kalshi.com/trade-api/v2/historical/markets/WRECSS-26-GER/candlesticks?start_ts=1713873600&end_ts=1762193549&period_interval=1440
Status code: 200
Top-level keys: ['candlesticks', 'ticker']
Raw candlestick response:
{
  "candlesticks": [
    {
      "end_period_ts": 1713931200,
      "open_interest": "1000.00",
      "price": {
        "close": "0.1400",
        "high": "0.1400",
        "low": "0.1200",
        "mean": "0.1320",
        "open": "0.1200",
        "previous": null
      },
      "volume": "1000.00",
      "yes_ask": {
        "close": "0.5000",
        "high": "1.0000",
        "low": "0.1200",
        "open": "1.0000"
      },
      "yes_bid": {
        "close": "0.0400",
        "high": "0

,end_period_ts,open_interest,price,volume,yes_ask,yes_bid
0,1713931200,1000.00,"{'close': '0.1400', 'high': '0.1400', 'low': '...",1000.00,"{'close': '0.5000', 'high': '1.0000', 'low': '...","{'close': '0.0400', 'high': '0.0400', 'low': '..."
1,1714017600,1040.00,"{'close': '0.2400', 'high': '0.2400', 'low': '...",40.00,"{'close': '0.2400', 'high': '0.5000', 'low': '...","{'close': '0.1700', 'high': '0.1700', 'low': '..."
2,1714104000,1120.00,"{'close': '0.2400', 'high': '0.2400', 'low': '...",80.00,"{'close': '0.2400', 'high': '0.3600', 'low': '...","{'close': '0.1700', 'high': '0.1700', 'low': '..."
3,1714363200,1120.00,"{'close': '0.1700', 'high': '0.1700', 'low': '...",1.00,"{'close': '0.2400', 'high': '0.2400', 'low': '...","{'close': '0.1700', 'high': '0.1700', 'low': '..."
4,1714881600,1139.00,"{'close': '0.2400', 'high': '0.2400', 'low': '...",19.00,"{'close': '0.2400', 'high': '0.2400', 'low': '...","{'close': '0.1800', 'high': '0.1800', 'low': '..."
...,...,...,...,...,...,...
407,1761796800,3053.00,"{'close': '0.5200', 'high': '0.5200', 'low': '...",3.00,"{'close': '0.5200', 'high': '0.5200', 'low': '...","{'close': '0.4700', 'high': '0.4700', 'low': '..."
408,1761883200,5125.00,"{'close': '0.6700', 'high': '0.6700', 'low': '...",2072.00,"{'close': '0.6400', 'high': '0.8400', 'low': '...","{'close': '0.5800', 'high': '0.5800', 'low': '..."
409,1761969600,6035.00,"{'close': '0.9200', 'high': '0.9600', 'low': '...",5349.00,"{'close': '0.9200', 'high': '0.9700', 'low': '...","{'close': '0.8600', 'high': '0.9300', 'low': '..."
410,1762056000,6008.00,"{'close': '0.8500', 'high': '0.9300', 'low': '...",57.00,"{'close': '0.8900', 'high': '0.9300', 'low': '...","{'close': '0.8500', 'high': '0.8800', 'low': '..."


# Trying to get Candlesticks + Volume Data

In [1]:
import time
import json
import requests
import pandas as pd
from urllib.parse import urljoin

BASE_URL = "https://external-api.kalshi.com/trade-api/v2"

MARKET_TICKER = "KXELECTIONMOVZOHRAN-25-ZMAM-B7.5"
EVENT_TICKER = "KXELECTIONMOVZOHRAN-25"

session = requests.Session()
session.headers.update({
    "User-Agent": "kalshi-closed-market-full-data-colab/1.0"
})


def get_json(path, params=None, max_retries=10, timeout=30):
    url = urljoin(BASE_URL + "/", path.lstrip("/"))
    params = params or {}

    for attempt in range(max_retries):
        resp = session.get(url, params=params, timeout=timeout)

        if resp.status_code == 429:
            sleep_s = min(5 * (attempt + 1), 60)
            print(f"Rate limited. Sleeping {sleep_s}s...")
            time.sleep(sleep_s)
            continue

        if 500 <= resp.status_code < 600:
            sleep_s = min(5 * (attempt + 1), 60)
            print(f"Server error {resp.status_code}. Sleeping {sleep_s}s...")
            time.sleep(sleep_s)
            continue

        try:
            data = resp.json()
        except Exception:
            data = {"raw_text": resp.text}

        return {
            "ok": resp.ok,
            "status_code": resp.status_code,
            "url": resp.url,
            "data": data
        }

    raise RuntimeError(f"Failed after {max_retries} retries: {url}")


def paginate(path, collection_key, params=None, limit=200, sleep_s=0.25):
    params = dict(params or {})
    params["limit"] = limit

    rows = []
    cursor = None

    while True:
        if cursor:
            params["cursor"] = cursor
        else:
            params.pop("cursor", None)

        result = get_json(path, params=params)

        if not result["ok"]:
            print("Bad URL:", result["url"])
            print("Response:", json.dumps(result["data"], indent=2)[:3000])
            break

        data = result["data"]
        rows.extend(data.get(collection_key, []))

        cursor = data.get("cursor")
        if not cursor:
            break

        time.sleep(sleep_s)

    return rows


def extract_object(data, key):
    if isinstance(data, dict):
        return data.get(key, data)
    return {}


def to_number(x):
    return pd.to_numeric(x, errors="coerce")


# ------------------------------------------------------------
# 1. Get market details: try historical first, then live/recent
# ------------------------------------------------------------
historical_market_resp = get_json(f"/historical/markets/{MARKET_TICKER}")

if historical_market_resp["ok"]:
    market_source = "historical"
    market = extract_object(historical_market_resp["data"], "market")
else:
    print("Not found in historical endpoint. Trying live/recent endpoint...")
    print("Historical response:", json.dumps(historical_market_resp["data"], indent=2)[:2000])

    live_market_resp = get_json(f"/markets/{MARKET_TICKER}")

    if not live_market_resp["ok"]:
        print("Live/recent response:", json.dumps(live_market_resp["data"], indent=2)[:2000])
        raise ValueError("Market was not found in historical or live/recent market endpoints.")

    market_source = "live"
    market = extract_object(live_market_resp["data"], "market")

df_market_details = pd.DataFrame([market])
print("Market source:", market_source)
print("Market detail columns:", df_market_details.columns.tolist())
display(df_market_details)

df_market_details.to_csv(f"market_details_{MARKET_TICKER}.csv", index=False)


# ------------------------------------------------------------
# 2. Get event details
# ------------------------------------------------------------
event_resp = get_json(f"/events/{EVENT_TICKER}")

if event_resp["ok"]:
    event = extract_object(event_resp["data"], "event")
    df_event_details = pd.DataFrame([event])
else:
    print("Event details error:", json.dumps(event_resp["data"], indent=2)[:2000])
    event = {}
    df_event_details = pd.DataFrame()

display(df_event_details)

if not df_event_details.empty:
    df_event_details.to_csv(f"event_details_{EVENT_TICKER}.csv", index=False)


# ------------------------------------------------------------
# 3. Get event metadata, if available
# ------------------------------------------------------------
event_metadata_resp = get_json(f"/events/{EVENT_TICKER}/metadata")

if event_metadata_resp["ok"]:
    event_metadata = event_metadata_resp["data"]
    df_event_metadata = pd.DataFrame([event_metadata])
else:
    print("Event metadata not available or errored:")
    print(json.dumps(event_metadata_resp["data"], indent=2)[:2000])
    df_event_metadata = pd.DataFrame()

display(df_event_metadata)

if not df_event_metadata.empty:
    df_event_metadata.to_csv(f"event_metadata_{EVENT_TICKER}.csv", index=False)


# ------------------------------------------------------------
# 4. Get all markets in this event: live/recent + historical
# ------------------------------------------------------------
live_event_markets = paginate(
    "/markets",
    "markets",
    params={
        "event_ticker": EVENT_TICKER
    },
    limit=200
)

for m in live_event_markets:
    m["_source"] = "live"

historical_event_markets = paginate(
    "/historical/markets",
    "markets",
    params={
        "event_ticker": EVENT_TICKER
    },
    limit=200
)

for m in historical_event_markets:
    m["_source"] = "historical"

markets_by_ticker = {}

for m in live_event_markets + historical_event_markets:
    t = m.get("ticker")
    if t:
        markets_by_ticker[t] = m

all_event_markets = list(markets_by_ticker.values())

df_event_markets = pd.DataFrame(all_event_markets)

print("Live/recent event markets:", len(live_event_markets))
print("Historical event markets:", len(historical_event_markets))
print("Unique event markets:", len(df_event_markets))

display(df_event_markets)

df_event_markets.to_csv(f"all_markets_for_event_{EVENT_TICKER}.csv", index=False)


# ------------------------------------------------------------
# 5. Determine time range and series ticker
# ------------------------------------------------------------
series_ticker = (
    market.get("series_ticker")
    or event.get("series_ticker")
)

if not series_ticker:
    # Fallback guess from event ticker: KXELECTIONMOVZOHRAN-25 -> KXELECTIONMOVZOHRAN
    series_ticker = EVENT_TICKER.rsplit("-", 1)[0]

print("Series ticker:", series_ticker)

open_time = pd.to_datetime(market.get("open_time"), utc=True, errors="coerce")

end_time = pd.to_datetime(
    market.get("close_time") or market.get("expiration_time") or market.get("settlement_ts"),
    utc=True,
    errors="coerce"
)

if pd.isna(open_time):
    raise ValueError("Market open_time is missing, so start_ts cannot be created.")

if pd.isna(end_time):
    end_time = pd.Timestamp.now(tz="UTC")

start_ts = int(open_time.timestamp())
end_ts = int(end_time.timestamp())

print("Candlestick start:", open_time)
print("Candlestick end:", end_time)


# ------------------------------------------------------------
# 6. Get market candlesticks: daily + all valid intervals
# ------------------------------------------------------------
# Kalshi valid intervals:
# 1 = 1 minute
# 60 = 1 hour
# 1440 = 1 day

intervals = {
    1: "1min",
    60: "hourly",
    1440: "daily"
}

market_candle_dfs = {}

for period_interval, label in intervals.items():
    print(f"Getting market {label} candlesticks...")

    if market_source == "historical":
        candle_path = f"/historical/markets/{MARKET_TICKER}/candlesticks"
    else:
        candle_path = f"/series/{series_ticker}/markets/{MARKET_TICKER}/candlesticks"

    candle_resp = get_json(
        candle_path,
        params={
            "start_ts": start_ts,
            "end_ts": end_ts,
            "period_interval": period_interval
        }
    )

    print(label, "status:", candle_resp["status_code"])
    print(label, "URL:", candle_resp["url"])

    if candle_resp["ok"]:
        df_candles = pd.DataFrame(candle_resp["data"].get("candlesticks", []))
        df_candles["market_ticker"] = MARKET_TICKER
        df_candles["event_ticker"] = EVENT_TICKER
        df_candles["series_ticker"] = series_ticker
        df_candles["period_interval"] = period_interval
        df_candles["period_label"] = label

        market_candle_dfs[label] = df_candles

        print(label, "rows:", len(df_candles))
        print(label, "columns:", df_candles.columns.tolist())
        display(df_candles.head())

        df_candles.to_csv(
            f"market_{label}_candlesticks_{MARKET_TICKER}.csv",
            index=False
        )
    else:
        print(f"{label} candlestick error:")
        print(json.dumps(candle_resp["data"], indent=2)[:3000])

if market_candle_dfs:
    df_all_market_candlesticks = pd.concat(
        market_candle_dfs.values(),
        ignore_index=True
    )
else:
    df_all_market_candlesticks = pd.DataFrame()

display(df_all_market_candlesticks)
df_all_market_candlesticks.to_csv(
    f"all_market_candlesticks_{MARKET_TICKER}.csv",
    index=False
)

df_daily_market_candlesticks = market_candle_dfs.get("daily", pd.DataFrame())
display(df_daily_market_candlesticks)


# ------------------------------------------------------------
# 7. Add cumulative daily volume if daily candle volume exists
# ------------------------------------------------------------
df_daily_market_candlesticks_with_cumulative = df_daily_market_candlesticks.copy()

if not df_daily_market_candlesticks_with_cumulative.empty:
    volume_cols = [
        c for c in df_daily_market_candlesticks_with_cumulative.columns
        if "volume" in c.lower()
    ]

    print("Daily candlestick volume-like columns:", volume_cols)

    for col in volume_cols:
        numeric_col = f"{col}_numeric"
        cumulative_col = f"cumulative_{col}"

        df_daily_market_candlesticks_with_cumulative[numeric_col] = to_number(
            df_daily_market_candlesticks_with_cumulative[col]
        )

        df_daily_market_candlesticks_with_cumulative[cumulative_col] = (
            df_daily_market_candlesticks_with_cumulative[numeric_col]
            .fillna(0)
            .cumsum()
        )

display(df_daily_market_candlesticks_with_cumulative)

df_daily_market_candlesticks_with_cumulative.to_csv(
    f"daily_market_candlesticks_with_cumulative_{MARKET_TICKER}.csv",
    index=False
)


# ------------------------------------------------------------
# 8. Event-level daily candlesticks
# ------------------------------------------------------------
event_daily_resp = get_json(
    f"/series/{series_ticker}/events/{EVENT_TICKER}/candlesticks",
    params={
        "start_ts": start_ts,
        "end_ts": end_ts,
        "period_interval": 1440
    }
)

print("Event daily candlesticks status:", event_daily_resp["status_code"])
print("Event daily candlesticks URL:", event_daily_resp["url"])

if event_daily_resp["ok"]:
    df_event_daily_candlesticks = pd.DataFrame(
        event_daily_resp["data"].get("candlesticks", [])
    )
    df_event_daily_candlesticks["event_ticker"] = EVENT_TICKER
    df_event_daily_candlesticks["series_ticker"] = series_ticker

    display(df_event_daily_candlesticks)

    df_event_daily_candlesticks.to_csv(
        f"event_daily_candlesticks_{EVENT_TICKER}.csv",
        index=False
    )
else:
    print("Event daily candlesticks error:")
    print(json.dumps(event_daily_resp["data"], indent=2)[:3000])
    df_event_daily_candlesticks = pd.DataFrame()


# ------------------------------------------------------------
# 9. Event-level volume summary across all markets
# ------------------------------------------------------------
df_event_volume_summary = pd.DataFrame()

if not df_event_markets.empty:
    dfv = df_event_markets.copy()

    if "volume_fp" in dfv.columns:
        dfv["volume_contracts"] = to_number(dfv["volume_fp"])
    elif "volume" in dfv.columns:
        dfv["volume_contracts"] = to_number(dfv["volume"])
    else:
        dfv["volume_contracts"] = pd.NA

    if "volume_24h_fp" in dfv.columns:
        dfv["volume_24h_contracts"] = to_number(dfv["volume_24h_fp"])
    elif "volume_24h" in dfv.columns:
        dfv["volume_24h_contracts"] = to_number(dfv["volume_24h"])
    else:
        dfv["volume_24h_contracts"] = pd.NA

    if "notional_value_dollars" in dfv.columns:
        dfv["notional_value_dollars_numeric"] = to_number(dfv["notional_value_dollars"])
    else:
        dfv["notional_value_dollars_numeric"] = 1.0

    # This is NOT exact cash spent. It is notional contract volume.
    dfv["estimated_notional_volume_dollars"] = (
        dfv["volume_contracts"] * dfv["notional_value_dollars_numeric"]
    )

    df_event_volume_by_market = dfv[[
        c for c in [
            "ticker",
            "_source",
            "title",
            "status",
            "result",
            "volume_fp",
            "volume_contracts",
            "volume_24h_fp",
            "volume_24h_contracts",
            "notional_value_dollars",
            "estimated_notional_volume_dollars"
        ]
        if c in dfv.columns
    ]].copy()

    display(df_event_volume_by_market)

    df_event_volume_summary = pd.DataFrame([{
        "event_ticker": EVENT_TICKER,
        "series_ticker": series_ticker,
        "n_markets": len(dfv),
        "total_volume_contracts": dfv["volume_contracts"].sum(skipna=True),
        "total_24h_volume_contracts": dfv["volume_24h_contracts"].sum(skipna=True),
        "estimated_total_notional_volume_dollars": dfv["estimated_notional_volume_dollars"].sum(skipna=True),
        "note": "Dollar value is not exact cash paid; it is contract volume multiplied by notional value when available."
    }])

    display(df_event_volume_summary)

    df_event_volume_by_market.to_csv(
        f"event_volume_by_market_{EVENT_TICKER}.csv",
        index=False
    )

    df_event_volume_summary.to_csv(
        f"event_volume_summary_{EVENT_TICKER}.csv",
        index=False
    )

print("Done.")

Market source: historical
Market detail columns: ['can_close_early', 'cap_strike', 'close_time', 'created_time', 'early_close_condition', 'event_ticker', 'expected_expiration_time', 'expiration_time', 'expiration_value', 'floor_strike', 'fractional_trading_enabled', 'last_price_dollars', 'latest_expiration_time', 'liquidity_dollars', 'market_type', 'no_ask_dollars', 'no_bid_dollars', 'no_sub_title', 'notional_value_dollars', 'open_interest_fp', 'open_time', 'previous_price_dollars', 'previous_yes_ask_dollars', 'previous_yes_bid_dollars', 'price_level_structure', 'price_ranges', 'response_price_units', 'result', 'rules_primary', 'rules_secondary', 'settlement_timer_seconds', 'settlement_ts', 'settlement_value_dollars', 'status', 'strike_type', 'ticker', 'title', 'updated_time', 'volume_24h_fp', 'volume_fp', 'yes_ask_dollars', 'yes_ask_size_fp', 'yes_bid_dollars', 'yes_bid_size_fp', 'yes_sub_title']


,can_close_early,cap_strike,close_time,created_time,early_close_condition,event_ticker,expected_expiration_time,expiration_time,expiration_value,floor_strike,...,ticker,title,updated_time,volume_24h_fp,volume_fp,yes_ask_dollars,yes_ask_size_fp,yes_bid_dollars,yes_bid_size_fp,yes_sub_title
0,True,8.99,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,6,...,KXELECTIONMOVZOHRAN-25-ZMAM-B7.5,Margin of victory for Zohran Mamdani in the Ma...,2026-03-04T22:43:52.624601Z,0.00,3398478.00,1.0000,0.00,0.0000,0.00,6-8.99%


,available_on_brokers,category,collateral_return_type,event_ticker,last_updated_ts,mutually_exclusive,series_ticker,settlement_sources,strike_period,sub_title,title
0,True,Elections,MECNET,KXELECTIONMOVZOHRAN-25,0001-01-01T00:00:00Z,True,KXELECTIONMOVZOHRAN,[{'name': 'Board of Elections in the City of N...,,In 2025,Margin of victory for Zohran Mamdani in the NY...


,featured_image_url,image_url,market_details,settlement_sources
0,https://kalshi-fallback-images.s3.amazonaws.co...,https://kalshi-public-docs.s3.amazonaws.com/se...,"[{'color_code': '#AA00FF', 'image_url': 'https...",[{'name': 'Board of Elections in the City of N...


Live/recent event markets: 0
Historical event markets: 10
Unique event markets: 10


...

Event daily candlesticks status: 200
Event daily candlesticks URL: https://external-api.kalshi.com/trade-api/v2/series/KXELECTIONMOVZOHRAN/events/KXELECTIONMOVZOHRAN-25/candlesticks?start_ts=1755093600&end_ts=1764726288&period_interval=1440


,event_ticker,series_ticker


,ticker,_source,title,status,result,volume_fp,volume_contracts,volume_24h_fp,volume_24h_contracts,notional_value_dollars,estimated_notional_volume_dollars
0,KXELECTIONMOVZOHRAN-25-ZMAM-T24,historical,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,2715311.00,2715311.0,0.00,0.0,1.0000,2715311.0
1,KXELECTIONMOVZOHRAN-25-ZMAM-B7.5,historical,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,3398478.00,3398478.0,0.00,0.0,1.0000,3398478.0
2,KXELECTIONMOVZOHRAN-25-ZMAM-B22.5,historical,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,922621.00,922621.0,0.00,0.0,1.0000,922621.0
3,KXELECTIONMOVZOHRAN-25-ZMAM-B19.5,historical,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,819153.00,819153.0,0.00,0.0,1.0000,819153.0
4,KXELECTIONMOVZOHRAN-25-ZMAM-B16.5,historical,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,923736.00,923736.0,0.00,0.0,1.0000,923736.0
5,KXELECTIONMOVZOHRAN-25-ZMAM-B13.5,historical,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,1454986.00,1454986.0,0.00,0.0,1.0000,1454986.0
6,KXELECTIONMOVZOHRAN-25-ZMAM-B10.5,historical,Margin of victory for Zohran Mamdani in the Ma...,finalized,yes,2641908.00,2641908.0,0.00,0.0,1.0000,2641908.0
7,KXELECTIONMOVZOHRAN-25-ZMAM-T0,historical,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,1488079.00,1488079.0,0.00,0.0,1.0000,1488079.0
8,KXELECTIONMOVZOHRAN-25-ZMAM-B4.5,historical,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,1689089.00,1689089.0,0.00,0.0,1.0000,1689089.0
9,KXELECTIONMOVZOHRAN-25-ZMAM-B1.5,historical,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,1256254.00,1256254.0,0.00,0.0,1.0000,1256254.0


,event_ticker,series_ticker,n_markets,total_volume_contracts,total_24h_volume_contracts,estimated_total_notional_volume_dollars,note
0,KXELECTIONMOVZOHRAN-25,KXELECTIONMOVZOHRAN,10,17309615.0,0.0,17309615.0,Dollar value is not exact cash paid; it is con...


Done.


In [2]:
import time
import json
import requests
import pandas as pd
from urllib.parse import urljoin

BASE_URL = "https://external-api.kalshi.com/trade-api/v2"
EVENT_TICKER = "KXELECTIONMOVZOHRAN-25"

session = requests.Session()
session.headers.update({
    "User-Agent": "kalshi-event-markets-colab/1.0"
})


def get_json(path, params=None, max_retries=10, timeout=30):
    url = urljoin(BASE_URL + "/", path.lstrip("/"))
    params = params or {}

    for attempt in range(max_retries):
        resp = session.get(url, params=params, timeout=timeout)

        if resp.status_code == 429:
            sleep_s = min(5 * (attempt + 1), 60)
            print(f"Rate limited. Sleeping {sleep_s}s...")
            time.sleep(sleep_s)
            continue

        if 500 <= resp.status_code < 600:
            sleep_s = min(5 * (attempt + 1), 60)
            print(f"Server error {resp.status_code}. Sleeping {sleep_s}s...")
            time.sleep(sleep_s)
            continue

        try:
            data = resp.json()
        except Exception:
            data = {"raw_text": resp.text}

        if resp.status_code >= 400:
            print("Bad URL:", resp.url)
            print("Kalshi response:", json.dumps(data, indent=2)[:3000])
            resp.raise_for_status()

        return data

    raise RuntimeError(f"Failed after {max_retries} retries: {url}")


def paginate(path, collection_key, params=None, limit=200, sleep_s=0.25):
    params = dict(params or {})
    params["limit"] = limit

    rows = []
    cursor = None

    while True:
        if cursor:
            params["cursor"] = cursor
        else:
            params.pop("cursor", None)

        data = get_json(path, params=params)
        rows.extend(data.get(collection_key, []))

        cursor = data.get("cursor")
        if not cursor:
            break

        time.sleep(sleep_s)

    return rows


# 1. Get event details
event_data = get_json(f"/events/{EVENT_TICKER}")
event = event_data.get("event", event_data)

df_event_details = pd.DataFrame([event])
display(df_event_details)

series_ticker = event.get("series_ticker")
print("Series ticker:", series_ticker)


# 2. Get live/recent markets for this event
live_markets = []

try:
    live_markets = paginate(
        "/markets",
        "markets",
        params={
            "event_ticker": EVENT_TICKER
        },
        limit=200
    )

    for m in live_markets:
        m["_source"] = "live_or_recent"

except Exception as e:
    print("Could not fetch live/recent markets:", e)


# 3. Get historical markets for this event
historical_markets = []

try:
    historical_markets = paginate(
        "/historical/markets",
        "markets",
        params={
            "event_ticker": EVENT_TICKER
        },
        limit=200
    )

    for m in historical_markets:
        m["_source"] = "historical"

except Exception as e:
    print("Could not fetch historical markets:", e)


# 4. Combine and deduplicate by market ticker
markets_by_ticker = {}

for market in live_markets + historical_markets:
    ticker = market.get("ticker")
    if ticker:
        if ticker in markets_by_ticker:
            old_source = markets_by_ticker[ticker].get("_source", "")
            new_source = market.get("_source", "")
            market["_source"] = f"{old_source}+{new_source}"
        markets_by_ticker[ticker] = market

all_markets = list(markets_by_ticker.values())

df_event_markets = pd.DataFrame(all_markets)

print("Live/recent markets:", len(live_markets))
print("Historical markets:", len(historical_markets))
print("Unique markets total:", len(df_event_markets))

display(df_event_markets)

# 5. Cleaner ticker table
wanted_cols = [
    "ticker",
    "event_ticker",
    "series_ticker",
    "_source",
    "title",
    "subtitle",
    "yes_sub_title",
    "no_sub_title",
    "status",
    "result",
    "created_time",
    "open_time",
    "close_time",
    "expiration_time",
    "settlement_ts",
    "notional_value_dollars",
    "volume",
    "volume_fp",
    "volume_24h",
    "volume_24h_fp",
    "open_interest",
    "liquidity",
    "yes_bid",
    "yes_ask",
    "no_bid",
    "no_ask",
    "last_price",
    "rules_primary",
    "rules_secondary"
]

existing_cols = [c for c in wanted_cols if c in df_event_markets.columns]

df_event_market_summary = df_event_markets[existing_cols].copy()

display(df_event_market_summary)

# 6. Save files
df_event_details.to_csv(
    f"kalshi_event_details_{EVENT_TICKER}.csv",
    index=False
)

df_event_markets.to_csv(
    f"kalshi_all_markets_for_event_{EVENT_TICKER}.csv",
    index=False
)

df_event_market_summary.to_csv(
    f"kalshi_market_summary_for_event_{EVENT_TICKER}.csv",
    index=False
)

,available_on_brokers,category,collateral_return_type,event_ticker,last_updated_ts,mutually_exclusive,series_ticker,settlement_sources,strike_period,sub_title,title
0,True,Elections,MECNET,KXELECTIONMOVZOHRAN-25,0001-01-01T00:00:00Z,True,KXELECTIONMOVZOHRAN,[{'name': 'Board of Elections in the City of N...,,In 2025,Margin of victory for Zohran Mamdani in the NY...


Series ticker: KXELECTIONMOVZOHRAN
Live/recent markets: 0
Historical markets: 10
Unique markets total: 10


,can_close_early,close_time,created_time,early_close_condition,event_ticker,expected_expiration_time,expiration_time,expiration_value,floor_strike,fractional_trading_enabled,...,volume_24h_fp,volume_fp,yes_ask_dollars,yes_ask_size_fp,yes_bid_dollars,yes_bid_size_fp,yes_sub_title,_source,cap_strike,subtitle
0,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,24.0,True,...,0.00,2715311.00,1.0000,0.00,0.0000,0.00,24% or more,historical,NaN,NaN
1,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,6.0,True,...,0.00,3398478.00,1.0000,0.00,0.0000,0.00,6-8.99%,historical,8.99,NaN
2,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,21.0,True,...,0.00,922621.00,1.0000,0.00,0.0000,0.00,21-23.99%,historical,23.99,NaN
3,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,18.0,True,...,0.00,819153.00,1.0000,0.00,0.0000,0.00,18-20.99%,historical,20.99,NaN
4,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,15.0,True,...,0.00,923736.00,1.0000,0.00,0.0000,0.00,15-17.99%,historical,17.99,NaN
5,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,12.0,True,...,0.00,1454986.00,1.0000,0.00,0.0000,0.00,12-14.99%,historical,14.99,NaN
6,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,9.0,True,...,0.00,2641908.00,1.0000,0.00,0.0000,0.00,9-11.99%,historical,11.99,NaN
7,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121613Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,NaN,True,...,0.00,1488079.00,1.0000,0.00,0.0000,0.00,Less than 0%,historical,0.00,:: Loses
8,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121613Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,3.0,True,...,0.00,1689089.00,1.0000,0.00,0.0000,0.00,3-5.99%,historical,5.99,NaN
9,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121613Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,0.0,True,...,0.00,1256254.00,1.0000,0.00,0.0000,0.00,0-2.99%,historical,2.99,NaN


,ticker,event_ticker,_source,title,subtitle,yes_sub_title,no_sub_title,status,result,created_time,open_time,close_time,expiration_time,settlement_ts,notional_value_dollars,volume_fp,volume_24h_fp,rules_primary,rules_secondary
0,KXELECTIONMOVZOHRAN-25-ZMAM-T24,KXELECTIONMOVZOHRAN-25,historical,Margin of victory for Zohran Mamdani in the Ma...,NaN,24% or more,24% or more,finalized,no,2025-08-12T21:46:06.121614Z,2025-08-13T14:00:00Z,2025-12-03T01:44:48.719034Z,2026-11-04T15:00:00Z,2025-12-03T02:14:52.646486Z,1.0000,2715311.00,0.00,If the popular vote margin of victory is 24% o...,The margin of victory is calculated by taking ...
1,KXELECTIONMOVZOHRAN-25-ZMAM-B7.5,KXELECTIONMOVZOHRAN-25,historical,Margin of victory for Zohran Mamdani in the Ma...,NaN,6-8.99%,6-8.99%,finalized,no,2025-08-12T21:46:06.121614Z,2025-08-13T14:00:00Z,2025-12-03T01:44:48.719034Z,2026-11-04T15:00:00Z,2025-12-03T02:14:52.646486Z,1.0000,3398478.00,0.00,If the popular vote margin of victory is 6-8.9...,The margin of victory is calculated by taking ...
2,KXELECTIONMOVZOHRAN-25-ZMAM-B22.5,KXELECTIONMOVZOHRAN-25,historical,Margin of victory for Zohran Mamdani in the Ma...,NaN,21-23.99%,21-23.99%,finalized,no,2025-08-12T21:46:06.121614Z,2025-08-13T14:00:00Z,2025-12-03T01:44:48.719034Z,2026-11-04T15:00:00Z,2025-12-03T02:14:52.646486Z,1.0000,922621.00,0.00,If the popular vote margin of victory is 21-23...,The margin of victory is calculated by taking ...
3,KXELECTIONMOVZOHRAN-25-ZMAM-B19.5,KXELECTIONMOVZOHRAN-25,historical,Margin of victory for Zohran Mamdani in the Ma...,NaN,18-20.99%,18-20.99%,finalized,no,2025-08-12T21:46:06.121614Z,2025-08-13T14:00:00Z,2025-12-03T01:44:48.719034Z,2026-11-04T15:00:00Z,2025-12-03T02:14:52.646486Z,1.0000,819153.00,0.00,If the popular vote margin of victory is 18-20...,The margin of victory is calculated by taking ...
4,KXELECTIONMOVZOHRAN-25-ZMAM-B16.5,KXELECTIONMOVZOHRAN-25,historical,Margin of victory for Zohran Mamdani in the Ma...,NaN,15-17.99%,15-17.99%,finalized,no,2025-08-12T21:46:06.121614Z,2025-08-13T14:00:00Z,2025-12-03T01:44:48.719034Z,2026-11-04T15:00:00Z,2025-12-03T02:14:52.646486Z,1.0000,923736.00,0.00,If the popular vote margin of victory is 15-17...,The margin of victory is calculated by taking ...
5,KXELECTIONMOVZOHRAN-25-ZMAM-B13.5,KXELECTIONMOVZOHRAN-25,historical,Margin of victory for Zohran Mamdani in the Ma...,NaN,12-14.99%,12-14.99%,finalized,no,2025-08-12T21:46:06.121614Z,2025-08-13T14:00:00Z,2025-12-03T01:44:48.719034Z,2026-11-04T15:00:00Z,2025-12-03T02:14:52.646486Z,1.0000,1454986.00,0.00,If the popular vote margin of victory is 12-14...,The margin of victory is calculated by taking ...
6,KXELECTIONMOVZOHRAN-25-ZMAM-B10.5,KXELECTIONMOVZOHRAN-25,historical,Margin of victory for Zohran Mamdani in the Ma...,NaN,9-11.99%,9-11.99%,finalized,yes,2025-08-12T21:46:06.121614Z,2025-08-13T14:00:00Z,2025-12-03T01:44:48.719034Z,2026-11-04T15:00:00Z,2025-12-03T02:14:52.646486Z,1.0000,2641908.00,0.00,If the popular vote margin of victory is 9-11....,The margin of victory is calculated by taking ...
7,KXELECTIONMOVZOHRAN-25-ZMAM-T0,KXELECTIONMOVZOHRAN-25,historical,Margin of victory for Zohran Mamdani in the Ma...,:: Loses,Less than 0%,Less than 0%,finalized,no,2025-08-12T21:46:06.121613Z,2025-08-13T14:00:00Z,2025-12-03T01:44:48.719034Z,2026-11-04T15:00:00Z,2025-12-03T02:14:52.646486Z,1.0000,1488079.00,0.00,If the popular vote margin of victory is 0% or...,The margin of victory is calculated by taking ...
8,KXELECTIONMOVZOHRAN-25-ZMAM-B4.5,KXELECTIONMOVZOHRAN-25,historical,Margin of victory for Zohran Mamdani in the Ma...,NaN,3-5.99%,3-5.99%,finalized,no,2025-08-12T21:46:06.121613Z,2025-08-13T14:00:00Z,2025-12-03T01:44:48.719034Z,2026-11-04T15:00:00Z,2025-12-03T02:14:52.646486Z,1.0000,1689089.00,0.00,If the popular vote margin of victory is 3-5.9...,The margin of victory is calculated by taking ...
9,KXELECTIONMOVZOHRAN-25-ZMAM-B1.5,KXELECTIONMOVZOHRAN-25,historical,Margin of victory for Zohran Mamdani in the Ma...,NaN,0

In [4]:
import time
import json
import requests
import pandas as pd
from urllib.parse import urljoin

BASE_URL = "https://external-api.kalshi.com/trade-api/v2"

EVENT_TICKER = "KXELECTIONMOVZOHRAN-25"
SERIES_TICKER = "KXELECTIONMOVZOHRAN"

session = requests.Session()
session.headers.update({
    "User-Agent": "kalshi-event-candlesticks-colab/1.0"
})


def get_json(path, params=None, max_retries=10, timeout=30):
    url = urljoin(BASE_URL + "/", path.lstrip("/"))
    params = params or {}

    for attempt in range(max_retries):
        resp = session.get(url, params=params, timeout=timeout)

        if resp.status_code == 429:
            sleep_s = min(5 * (attempt + 1), 60)
            print(f"Rate limited. Sleeping {sleep_s}s...")
            time.sleep(sleep_s)
            continue

        try:
            data = resp.json()
        except Exception:
            data = {"raw_text": resp.text}

        if resp.status_code >= 400:
            print("Bad URL:", resp.url)
            print("Kalshi response:", json.dumps(data, indent=2)[:3000])
            resp.raise_for_status()

        return data

    raise RuntimeError(f"Failed after {max_retries} retries: {url}")


# 1. Get event details so we can use its dates
event_data = get_json(f"/events/{EVENT_TICKER}")
event = event_data.get("event", event_data)

df_event_details = pd.DataFrame([event])
display(df_event_details)

# If the API returns series_ticker, use it
SERIES_TICKER = event.get("series_ticker", SERIES_TICKER)

print("Series ticker:", SERIES_TICKER)
print("Event ticker:", EVENT_TICKER)

# 2. Choose date range
# Try event dates first; if unavailable, use a wide fallback range.
start_time = pd.to_datetime(
    event.get("open_time") or event.get("created_time"),
    utc=True,
    errors="coerce"
)

end_time = pd.to_datetime(
    event.get("close_time") or event.get("expiration_time") or event.get("settlement_ts"),
    utc=True,
    errors="coerce"
)

if pd.isna(start_time):
    # fallback: 1 year ago
    start_time = pd.Timestamp.now(tz="UTC") - pd.Timedelta(days=365)

if pd.isna(end_time):
    end_time = pd.Timestamp.now(tz="UTC")

start_ts = int(start_time.timestamp())
end_ts = int(end_time.timestamp())

print("Start:", start_time)
print("End:", end_time)

# 3. Get daily event candlesticks
# period_interval:
# 1    = 1 minute
# 60   = hourly
# 1440 = daily

event_candles_data = get_json(
    f"/series/{SERIES_TICKER}/events/{EVENT_TICKER}/candlesticks",
    params={
        "start_ts": start_ts,
        "end_ts": end_ts,
        "period_interval": 1440
    }
)

df_event_daily_candlesticks = pd.DataFrame(
    event_candles_data.get("candlesticks", [])
)

df_event_daily_candlesticks["event_ticker"] = EVENT_TICKER
df_event_daily_candlesticks["series_ticker"] = SERIES_TICKER
df_event_daily_candlesticks["period_interval"] = 1440
df_event_daily_candlesticks["period_label"] = "daily"

display(df_event_daily_candlesticks)

print("Daily event candlesticks:", len(df_event_daily_candlesticks))
print("Columns:", df_event_daily_candlesticks.columns.tolist())

df_event_daily_candlesticks.to_csv(
    f"kalshi_event_daily_candlesticks_{EVENT_TICKER}.csv",
    index=False
)

,available_on_brokers,category,collateral_return_type,event_ticker,last_updated_ts,mutually_exclusive,series_ticker,settlement_sources,strike_period,sub_title,title
0,True,Elections,MECNET,KXELECTIONMOVZOHRAN-25,0001-01-01T00:00:00Z,True,KXELECTIONMOVZOHRAN,[{'name': 'Board of Elections in the City of N...,,In 2025,Margin of victory for Zohran Mamdani in the NY...


Series ticker: KXELECTIONMOVZOHRAN
Event ticker: KXELECTIONMOVZOHRAN-25
Start: 2025-06-24 14:50:05.074690+00:00
End: 2026-06-24 14:50:05.074845+00:00


,event_ticker,series_ticker,period_interval,period_label


Daily event candlesticks: 0
Columns: ['event_ticker', 'series_ticker', 'period_interval', 'period_label']


In [6]:
import time
import json
import requests
import pandas as pd
from urllib.parse import urljoin

BASE_URL = "https://external-api.kalshi.com/trade-api/v2"

EVENT_TICKER = "KXELECTIONMOVZOHRAN-25"
SERIES_TICKER = "KXELECTIONMOVZOHRAN"

session = requests.Session()
session.headers.update({
    "User-Agent": "kalshi-event-candlesticks-colab/1.0"
})


def get_json(path, params=None, max_retries=10, timeout=30):
    url = urljoin(BASE_URL + "/", path.lstrip("/"))
    params = params or {}

    for attempt in range(max_retries):
        resp = session.get(url, params=params, timeout=timeout)

        if resp.status_code == 429:
            sleep_s = min(5 * (attempt + 1), 60)
            print(f"Rate limited. Sleeping {sleep_s}s...")
            time.sleep(sleep_s)
            continue

        try:
            data = resp.json()
        except Exception:
            data = {"raw_text": resp.text}

        if resp.status_code >= 400:
            print("Bad URL:", resp.url)
            print("Kalshi response:", json.dumps(data, indent=2)[:3000])
            resp.raise_for_status()

        print("URL:", resp.url)
        return data

    raise RuntimeError(f"Failed after {max_retries} retries: {url}")


def paginate(path, collection_key, params=None, limit=200, sleep_s=0.25):
    params = dict(params or {})
    params["limit"] = limit

    rows = []
    cursor = None

    while True:
        if cursor:
            params["cursor"] = cursor
        else:
            params.pop("cursor", None)

        data = get_json(path, params=params)
        rows.extend(data.get(collection_key, []))

        cursor = data.get("cursor")
        if not cursor:
            break

        time.sleep(sleep_s)

    return rows


# 1. Get event details
event_data = get_json(f"/events/{EVENT_TICKER}")
event = event_data.get("event", event_data)

df_event_details = pd.DataFrame([event])
display(df_event_details)

SERIES_TICKER = event.get("series_ticker", SERIES_TICKER)

print("Series ticker:", SERIES_TICKER)
print("Event ticker:", EVENT_TICKER)


# 2. Get all markets for this event to determine real date range
live_markets = paginate(
    "/markets",
    "markets",
    params={"event_ticker": EVENT_TICKER},
    limit=200
)

historical_markets = paginate(
    "/historical/markets",
    "markets",
    params={"event_ticker": EVENT_TICKER},
    limit=200
)

all_markets = live_markets + historical_markets
df_event_markets = pd.DataFrame(all_markets)

display(df_event_markets)

print("Live/recent markets:", len(live_markets))
print("Historical markets:", len(historical_markets))
print("Total market rows:", len(df_event_markets))


# 3. Build start_ts and end_ts from market dates
for col in ["open_time", "close_time", "expiration_time", "settlement_ts"]:
    if col in df_event_markets.columns:
        df_event_markets[col] = pd.to_datetime(
            df_event_markets[col],
            utc=True,
            errors="coerce"
        )

start_time = df_event_markets["open_time"].min()

end_time = pd.concat([
    df_event_markets[c]
    for c in ["close_time", "expiration_time", "settlement_ts"]
    if c in df_event_markets.columns
]).max()

start_ts = int(start_time.timestamp())
end_ts = int(end_time.timestamp())

print("Candlestick start:", start_time)
print("Candlestick end:", end_time)


# 4. Get daily event candlesticks
event_daily_data = get_json(
    f"/series/{SERIES_TICKER}/events/{EVENT_TICKER}/candlesticks",
    params={
        "start_ts": start_ts,
        "end_ts": end_ts,
        "period_interval": 1440
    }
)

df_event_daily_candlesticks = pd.DataFrame(
    event_daily_data.get("candlesticks", [])
)

df_event_daily_candlesticks["event_ticker"] = EVENT_TICKER
df_event_daily_candlesticks["series_ticker"] = SERIES_TICKER
df_event_daily_candlesticks["period_interval"] = 1440
df_event_daily_candlesticks["period_label"] = "daily"

display(df_event_daily_candlesticks)

print("Daily event candlesticks:", len(df_event_daily_candlesticks))
print("Columns:", df_event_daily_candlesticks.columns.tolist())

df_event_daily_candlesticks.to_csv(
    f"kalshi_event_daily_candlesticks_{EVENT_TICKER}.csv",
    index=False
)


# 5. Optional: get all valid event candlestick intervals
intervals = {
    1: "1min",
    60: "hourly",
    1440: "daily"
}

event_candlestick_dfs = {}

for period_interval, label in intervals.items():
    print(f"Getting {label} event candlesticks...")

    data = get_json(
        f"/series/{SERIES_TICKER}/events/{EVENT_TICKER}/candlesticks",
        params={
            "start_ts": start_ts,
            "end_ts": end_ts,
            "period_interval": period_interval
        }
    )

    df_tmp = pd.DataFrame(data.get("candlesticks", []))
    df_tmp["event_ticker"] = EVENT_TICKER
    df_tmp["series_ticker"] = SERIES_TICKER
    df_tmp["period_interval"] = period_interval
    df_tmp["period_label"] = label

    event_candlestick_dfs[label] = df_tmp

    print(label, "rows:", len(df_tmp))
    display(df_tmp.head())

    df_tmp.to_csv(
        f"kalshi_event_{label}_candlesticks_{EVENT_TICKER}.csv",
        index=False
    )

df_all_event_candlesticks = pd.concat(
    event_candlestick_dfs.values(),
    ignore_index=True
)

display(df_all_event_candlesticks)

df_all_event_candlesticks.to_csv(
    f"kalshi_all_event_candlesticks_{EVENT_TICKER}.csv",
    index=False
)

URL: https://external-api.kalshi.com/trade-api/v2/events/KXELECTIONMOVZOHRAN-25


,available_on_brokers,category,collateral_return_type,event_ticker,last_updated_ts,mutually_exclusive,series_ticker,settlement_sources,strike_period,sub_title,title
0,True,Elections,MECNET,KXELECTIONMOVZOHRAN-25,0001-01-01T00:00:00Z,True,KXELECTIONMOVZOHRAN,[{'name': 'Board of Elections in the City of N...,,In 2025,Margin of victory for Zohran Mamdani in the NY...


Series ticker: KXELECTIONMOVZOHRAN
Event ticker: KXELECTIONMOVZOHRAN-25
URL: https://external-api.kalshi.com/trade-api/v2/markets?event_ticker=KXELECTIONMOVZOHRAN-25&limit=200
URL: https://external-api.kalshi.com/trade-api/v2/historical/markets?event_ticker=KXELECTIONMOVZOHRAN-25&limit=200


,can_close_early,close_time,created_time,early_close_condition,event_ticker,expected_expiration_time,expiration_time,expiration_value,floor_strike,fractional_trading_enabled,...,updated_time,volume_24h_fp,volume_fp,yes_ask_dollars,yes_ask_size_fp,yes_bid_dollars,yes_bid_size_fp,yes_sub_title,cap_strike,subtitle
0,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,24.0,True,...,2026-03-04T22:43:52.878979Z,0.00,2715311.00,1.0000,0.00,0.0000,0.00,24% or more,NaN,NaN
1,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,6.0,True,...,2026-03-04T22:43:52.624601Z,0.00,3398478.00,1.0000,0.00,0.0000,0.00,6-8.99%,8.99,NaN
2,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,21.0,True,...,2026-03-04T22:43:52.829531Z,0.00,922621.00,1.0000,0.00,0.0000,0.00,21-23.99%,23.99,NaN
3,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,18.0,True,...,2026-03-04T22:43:52.77753Z,0.00,819153.00,1.0000,0.00,0.0000,0.00,18-20.99%,20.99,NaN
4,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,15.0,True,...,2026-03-04T22:43:52.739289Z,0.00,923736.00,1.0000,0.00,0.0000,0.00,15-17.99%,17.99,NaN
5,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,12.0,True,...,2026-03-04T22:43:52.700178Z,0.00,1454986.00,1.0000,0.00,0.0000,0.00,12-14.99%,14.99,NaN
6,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,9.0,True,...,2026-03-04T22:43:52.661638Z,0.00,2641908.00,1.0000,0.00,0.0000,0.00,9-11.99%,11.99,NaN
7,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121613Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,NaN,True,...,2026-03-04T22:43:52.494901Z,0.00,1488079.00,1.0000,0.00,0.0000,0.00,Less than 0%,0.00,:: Loses
8,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121613Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,3.0,True,...,2026-03-04T22:43:52.57737Z,0.00,1689089.00,1.0000,0.00,0.0000,0.00,3-5.99%,5.99,NaN
9,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121613Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,0.0,True,...,2026-03-04T22:43:52.539059Z,0.00,1256254.00,1.0000,0.00,0.0000,0.00,0-2.99%,2.99,NaN


Live/recent markets: 0
Historical markets: 10
Total market rows: 10
Candlestick start: 2025-08-13 14:00:00+00:00
Candlestick end: 2026-11-04 15:00:00+00:00
URL: https://external-api.kalshi.com/trade-api/v2/series/KXELECTIONMOVZOHRAN/events/KXELECTIONMOVZOHRAN-25/candlesticks?start_ts=1755093600&end_ts=1793804400&period_interval=1440


...

Getting hourly event candlesticks...
URL: https://external-api.kalshi.com/trade-api/v2/series/KXELECTIONMOVZOHRAN/events/KXELECTIONMOVZOHRAN-25/candlesticks?start_ts=1755093600&end_ts=1793804400&period_interval=60
hourly rows: 0


,event_ticker,series_ticker,period_interval,period_label


Getting daily event candlesticks...
Rate limited. Sleeping 5s...
URL: https://external-api.kalshi.com/trade-api/v2/series/KXELECTIONMOVZOHRAN/events/KXELECTIONMOVZOHRAN-25/candlesticks?start_ts=1755093600&end_ts=1793804400&period_interval=1440
daily rows: 0


,event_ticker,series_ticker,period_interval,period_label


,event_ticker,series_ticker,period_interval,period_label


In [7]:
# Get daily historical market candlesticks for every market in this event

all_market_daily_candles = []

for i, row in df_event_markets.iterrows():
    market_ticker = row["ticker"]

    open_time = pd.to_datetime(row.get("open_time"), utc=True, errors="coerce")
    close_time = pd.to_datetime(
        row.get("close_time") or row.get("expiration_time") or row.get("settlement_ts"),
        utc=True,
        errors="coerce"
    )

    if pd.isna(open_time) or pd.isna(close_time):
        print(f"Skipping {market_ticker}: missing open/close time")
        continue

    start_ts = int(open_time.timestamp())
    end_ts = int(close_time.timestamp())

    print(f"[{i+1}/{len(df_event_markets)}] Getting daily candles for {market_ticker}...")

    try:
        data = get_json(
            f"/historical/markets/{market_ticker}/candlesticks",
            params={
                "start_ts": start_ts,
                "end_ts": end_ts,
                "period_interval": 1440
            }
        )

        candles = data.get("candlesticks", [])

        for candle in candles:
            candle["market_ticker"] = market_ticker
            candle["event_ticker"] = EVENT_TICKER
            candle["series_ticker"] = SERIES_TICKER
            candle["market_title"] = row.get("title")
            candle["market_status"] = row.get("status")
            candle["market_result"] = row.get("result")
            candle["period_interval"] = 1440
            candle["period_label"] = "daily"

        all_market_daily_candles.extend(candles)

    except Exception as e:
        print(f"Skipping {market_ticker}: {e}")

    time.sleep(0.25)

df_event_market_daily_candlesticks = pd.DataFrame(all_market_daily_candles)

display(df_event_market_daily_candlesticks)

print("Total daily market candlesticks:", len(df_event_market_daily_candlesticks))
print("Columns:", df_event_market_daily_candlesticks.columns.tolist())

df_event_market_daily_candlesticks.to_csv(
    f"kalshi_market_daily_candlesticks_for_event_{EVENT_TICKER}.csv",
    index=False
)

[1/10] Getting daily candles for KXELECTIONMOVZOHRAN-25-ZMAM-T24...
URL: https://external-api.kalshi.com/trade-api/v2/historical/markets/KXELECTIONMOVZOHRAN-25-ZMAM-T24/candlesticks?start_ts=1755093600&end_ts=1764726288&period_interval=1440
[2/10] Getting daily candles for KXELECTIONMOVZOHRAN-25-ZMAM-B7.5...
URL: https://external-api.kalshi.com/trade-api/v2/historical/markets/KXELECTIONMOVZOHRAN-25-ZMAM-B7.5/candlesticks?start_ts=1755093600&end_ts=1764726288&period_interval=1440
[3/10] Getting daily candles for KXELECTIONMOVZOHRAN-25-ZMAM-B22.5...
URL: https://external-api.kalshi.com/trade-api/v2/historical/markets/KXELECTIONMOVZOHRAN-25-ZMAM-B22.5/candlesticks?start_ts=1755093600&end_ts=1764726288&period_interval=1440
[4/10] Getting daily candles for KXELECTIONMOVZOHRAN-25-ZMAM-B19.5...
URL: https://external-api.kalshi.com/trade-api/v2/historical/markets/KXELECTIONMOVZOHRAN-25-ZMAM-B19.5/candlesticks?start_ts=1755093600&end_ts=1764726288&period_interval=1440
[5/10] Getting daily candl

,end_period_ts,open_interest,price,volume,yes_ask,yes_bid,market_ticker,event_ticker,series_ticker,market_title,market_status,market_result,period_interval,period_label
0,1755144000,5154.00,"{'close': '0.4100', 'high': '0.7400', 'low': '...",5433.00,"{'close': '0.4500', 'high': '1.0000', 'low': '...","{'close': '0.4100', 'high': '0.4700', 'low': '...",KXELECTIONMOVZOHRAN-25-ZMAM-T24,KXELECTIONMOVZOHRAN-25,KXELECTIONMOVZOHRAN,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,1440,daily
1,1755230400,6394.00,"{'close': '0.4000', 'high': '0.7000', 'low': '...",1858.00,"{'close': '0.4800', 'high': '0.7900', 'low': '...","{'close': '0.4000', 'high': '0.4500', 'low': '...",KXELECTIONMOVZOHRAN-25-ZMAM-T24,KXELECTIONMOVZOHRAN-25,KXELECTIONMOVZOHRAN,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,1440,daily
2,1755316800,6429.00,"{'close': '0.3900', 'high': '0.4000', 'low': '...",90.00,"{'close': '0.4700', 'high': '0.4800', 'low': '...","{'close': '0.4100', 'high': '0.4100', 'low': '...",KXELECTIONMOVZOHRAN-25-ZMAM-T24,KXELECTIONMOVZOHRAN-25,KXELECTIONMOVZOHRAN,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,1440,daily
3,1755403200,6429.00,"{'close': '0.4800', 'high': '0.4800', 'low': '...",4.00,"{'close': '0.4700', 'high': '0.4800', 'low': '...","{'close': '0.4200', 'high': '0.4200', 'low': '...",KXELECTIONMOVZOHRAN-25-ZMAM-T24,KXELECTIONMOVZOHRAN-25,KXELECTIONMOVZOHRAN,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,1440,daily
4,1755489600,6840.00,"{'close': '0.4100', 'high': '0.4200', 'low': '...",468.00,"{'close': '0.4600', 'high': '0.4800', 'low': '...","{'close': '0.4100', 'high': '0.4200', 'low': '...",KXELECTIONMOVZOHRAN-25-ZMAM-T24,KXELECTIONMOVZOHRAN-25,KXELECTIONMOVZOHRAN,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,1440,daily
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1105,1764306000,921513.00,"{'close': '0.0100', 'high': '0.0100', 'low': '...",348.00,"{'close': '0.0100', 'high': '0.0100', 'low': '...","{'close': '0.0000', 'high': '0.0000', 'low': '...",KXELECTIONMOVZOHRAN-25-ZMAM-B1.5,KXELECTIONMOVZOHRAN-25,KXELECTIONMOVZOHRAN,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,1440,daily
1106,1764392400,921514.00,"{'close': '0.0100', 'high': '0.0100', 'low': '...",1.00,"{'close': '0.0100', 'high': '0.0100', 'low': '...","{'close': '0.0000', 'high': '0.0000', 'low': '...",KXELECTIONMOVZOHRAN-25-ZMAM-B1.5,KXELECTIONMOVZOHRAN-25,KXELECTIONMOVZOHRAN,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,1440,daily
1107,1764478800,921514.00,"{'close': None, 'high': None, 'low': None, 'me...",0.00,"{'close': '0.0100', 'high': '0.0100', 'low': '...","{'close': '0.0000', 'high': '0.0000', 'low': '...",KXELECTIONMOVZOHRAN-25-ZMAM-B1.5,KXELECTIONMOVZOHRAN-25,KXELECTIONMOVZOHRAN,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,1440,daily
1108,1764565200,921514.00,"{'close': None, 'high': None, 'low': None, 'me...",0.00,"{'close': '0.0100', 'high': '0.0100', 'low': '...","{'close': '0.0000', 'high': '0.0000', 'low': '...",KXELECTIONMOVZOHRAN-25-ZMAM-B1.5,KXELECTIONMOVZOHRAN-25,KXELECTIONMOVZOHRAN,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,1440,daily


Total daily market candlesticks: 1110
Columns: ['end_period_ts', 'open_interest', 'price', 'volume', 'yes_ask', 'yes_bid', 'market_ticker', 'event_ticker', 'series_ticker', 'market_title', 'market_status', 'market_result', 'period_interval', 'period_label']


In [10]:
import requests
import pandas as pd

EVENT_TICKER = "KXELECTIONMOVZOHRAN-25"
SERIES_TICKER = "KXELECTIONMOVZOHRAN"

START_TS = 1755093600   # 2025-08-13 14:00:00 UTC
END_TS = 1764727200     # 2025-12-03 02:00:00 UTC
PERIOD_INTERVAL = 1440  # 1 = minute, 60 = hour, 1440 = day

BASE_URL = "https://external-api.kalshi.com/trade-api/v2"

url = (
    f"{BASE_URL}/series/{SERIES_TICKER}"
    f"/events/{EVENT_TICKER}/candlesticks"
)

params = {
    "start_ts": START_TS,
    "end_ts": END_TS,
    "period_interval": PERIOD_INTERVAL,
}

response = requests.get(
    url,
    params=params,
    headers={"Accept": "application/json"},
    timeout=30,
)

response.raise_for_status()
data = response.json()

rows = []

for market_ticker, candles in zip(
    data["market_tickers"],
    data["market_candlesticks"]
):
    for candle in candles:
        row = {
            "market_ticker": market_ticker,
            "end_period_ts": candle.get("end_period_ts"),
            "end_period_datetime": pd.to_datetime(
                candle.get("end_period_ts"),
                unit="s",
                utc=True
            ),
            "volume": candle.get("volume_fp"),
            "open_interest": candle.get("open_interest_fp"),
        }

        for group in ["yes_bid", "yes_ask", "price"]:
            values = candle.get(group, {})
            for key, value in values.items():
                row[f"{group}_{key}"] = value

        rows.append(row)

df = pd.DataFrame(rows)

df

""


In [14]:
import time
import json
import requests
import pandas as pd
from urllib.parse import urljoin

BASE_URL = "https://external-api.kalshi.com/trade-api/v2"

EVENT_TICKER = "KXELECTIONMOVZOHRAN-25"
SERIES_TICKER = "KXELECTIONMOVZOHRAN"

session = requests.Session()
session.headers.update({
    "User-Agent": "kalshi-all-event-market-data-colab/1.0"
})


def get_json(path, params=None, max_retries=10, timeout=30):
    url = urljoin(BASE_URL + "/", path.lstrip("/"))
    params = params or {}

    for attempt in range(max_retries):
        resp = session.get(url, params=params, timeout=timeout)

        if resp.status_code == 429:
            sleep_s = min(5 * (attempt + 1), 60)
            print(f"Rate limited. Sleeping {sleep_s}s...")
            time.sleep(sleep_s)
            continue

        try:
            data = resp.json()
        except Exception:
            data = {"raw_text": resp.text}

        if resp.status_code >= 400:
            print("Bad URL:", resp.url)
            print("Kalshi response:", json.dumps(data, indent=2)[:3000])
            resp.raise_for_status()

        return data

    raise RuntimeError(f"Failed after {max_retries} retries: {url}")


def paginate(path, collection_key, params=None, limit=200, sleep_s=0.25):
    params = dict(params or {})
    params["limit"] = limit

    rows = []
    cursor = None

    while True:
        if cursor:
            params["cursor"] = cursor
        else:
            params.pop("cursor", None)

        data = get_json(path, params=params)
        rows.extend(data.get(collection_key, []))

        cursor = data.get("cursor")
        if not cursor:
            break

        time.sleep(sleep_s)

    return rows


def to_number(x):
    return pd.to_numeric(x, errors="coerce")


# 1. Event details
event_data = get_json(f"/events/{EVENT_TICKER}")
event = event_data.get("event", event_data)

df_event_details = pd.DataFrame([event])
display(df_event_details)

SERIES_TICKER = event.get("series_ticker", SERIES_TICKER)

df_event_details.to_csv(
    f"kalshi_event_details_{EVENT_TICKER}.csv",
    index=False
)


# 2. Event metadata
try:
    event_metadata = get_json(f"/events/{EVENT_TICKER}/metadata")
    df_event_metadata = pd.DataFrame([event_metadata])
    display(df_event_metadata)

    df_event_metadata.to_csv(
        f"kalshi_event_metadata_{EVENT_TICKER}.csv",
        index=False
    )
except Exception as e:
    print("Could not fetch event metadata:", e)
    df_event_metadata = pd.DataFrame()


# 3. All markets for event: live/recent + historical
live_markets = []

try:
    live_markets = paginate(
        "/markets",
        "markets",
        params={"event_ticker": EVENT_TICKER},
        limit=200
    )

    for m in live_markets:
        m["_source"] = "live_or_recent"

except Exception as e:
    print("Could not fetch live/recent markets:", e)


historical_markets = []

try:
    historical_markets = paginate(
        "/historical/markets",
        "markets",
        params={"event_ticker": EVENT_TICKER},
        limit=200
    )

    for m in historical_markets:
        m["_source"] = "historical"

except Exception as e:
    print("Could not fetch historical markets:", e)


markets_by_ticker = {}

for m in live_markets + historical_markets:
    ticker = m.get("ticker")
    if ticker:
        markets_by_ticker[ticker] = m

df_event_markets = pd.DataFrame(list(markets_by_ticker.values()))

print("Live/recent markets:", len(live_markets))
print("Historical markets:", len(historical_markets))
print("Unique markets:", len(df_event_markets))

display(df_event_markets)

df_event_markets.to_csv(
    f"kalshi_all_markets_for_event_{EVENT_TICKER}.csv",
    index=False
)


# 4. Full market details for each market ticker
full_market_rows = []
failed_market_details = []

for i, row in df_event_markets.iterrows():
    market_ticker = row["ticker"]
    source = row.get("_source", "")

    print(f"[{i+1}/{len(df_event_markets)}] Fetching full market details for {market_ticker}...")

    try:
        if source == "historical":
            data = get_json(f"/historical/markets/{market_ticker}")
        else:
            data = get_json(f"/markets/{market_ticker}")

        market = data.get("market", data)
        market["_source"] = source
        full_market_rows.append(market)

    except Exception as e:
        print(f"Failed {market_ticker}: {e}")
        failed_market_details.append({
            "ticker": market_ticker,
            "source": source,
            "error": str(e)
        })

    time.sleep(0.25)

df_full_market_details = pd.DataFrame(full_market_rows)

display(df_full_market_details)

df_full_market_details.to_csv(
    f"kalshi_full_market_details_for_event_{EVENT_TICKER}.csv",
    index=False
)

if failed_market_details:
    df_failed_market_details = pd.DataFrame(failed_market_details)
    display(df_failed_market_details)

    df_failed_market_details.to_csv(
        f"kalshi_failed_market_details_{EVENT_TICKER}.csv",
        index=False
    )


# 5. Clean market summary
summary_cols = [
    "ticker",
    "event_ticker",
    "series_ticker",
    "_source",
    "title",
    "subtitle",
    "yes_sub_title",
    "no_sub_title",
    "status",
    "result",
    "created_time",
    "open_time",
    "close_time",
    "expiration_time",
    "settlement_ts",
    "notional_value_dollars",
    "volume",
    "volume_fp",
    "volume_24h",
    "volume_24h_fp",
    "open_interest",
    "liquidity",
    "yes_bid",
    "yes_ask",
    "no_bid",
    "no_ask",
    "last_price",
    "rules_primary",
    "rules_secondary"
]

existing_summary_cols = [
    c for c in summary_cols
    if c in df_full_market_details.columns
]

df_market_summary = df_full_market_details[existing_summary_cols].copy()

display(df_market_summary)

df_market_summary.to_csv(
    f"kalshi_market_summary_for_event_{EVENT_TICKER}.csv",
    index=False
)


# 6. Candlesticks for every market: 1min, hourly, daily
intervals = {
    1: "1min",
    60: "hourly",
    1440: "daily"
}

all_candles_by_interval = {
    "1min": [],
    "hourly": [],
    "daily": []
}

failed_candles = []

for i, row in df_full_market_details.iterrows():
    market_ticker = row["ticker"]
    source = row.get("_source", "")

    open_time = pd.to_datetime(row.get("open_time"), utc=True, errors="coerce")
    end_time = pd.to_datetime(
        row.get("close_time") or row.get("expiration_time") or row.get("settlement_ts"),
        utc=True,
        errors="coerce"
    )

    if pd.isna(open_time) or pd.isna(end_time):
        print(f"Skipping candles for {market_ticker}: missing date range")
        continue

    start_ts = int(open_time.timestamp())
    end_ts = int(end_time.timestamp())

    for period_interval, label in intervals.items():
        print(f"[{i+1}/{len(df_full_market_details)}] Getting {label} candles for {market_ticker}...")

        try:
            if source == "historical":
                path = f"/historical/markets/{market_ticker}/candlesticks"
            else:
                path = f"/series/{SERIES_TICKER}/markets/{market_ticker}/candlesticks"

            data = get_json(
                path,
                params={
                    "start_ts": start_ts,
                    "end_ts": end_ts,
                    "period_interval": period_interval
                }
            )

            candles = data.get("candlesticks", [])

            for candle in candles:
                candle["market_ticker"] = market_ticker
                candle["event_ticker"] = EVENT_TICKER
                candle["series_ticker"] = SERIES_TICKER
                candle["market_title"] = row.get("title")
                candle["market_status"] = row.get("status")
                candle["market_result"] = row.get("result")
                candle["period_interval"] = period_interval
                candle["period_label"] = label
                candle["_source"] = source

            all_candles_by_interval[label].extend(candles)

        except Exception as e:
            print(f"Failed candles for {market_ticker} {label}: {e}")
            failed_candles.append({
                "ticker": market_ticker,
                "period_label": label,
                "error": str(e)
            })

        time.sleep(0.25)


df_1min_candles = pd.DataFrame(all_candles_by_interval["1min"])
df_hourly_candles = pd.DataFrame(all_candles_by_interval["hourly"])
df_daily_candles = pd.DataFrame(all_candles_by_interval["daily"])

display(df_daily_candles)

df_1min_candles.to_csv(
    f"kalshi_1min_market_candlesticks_for_event_{EVENT_TICKER}.csv",
    index=False
)

df_hourly_candles.to_csv(
    f"kalshi_hourly_market_candlesticks_for_event_{EVENT_TICKER}.csv",
    index=False
)

df_daily_candles.to_csv(
    f"kalshi_daily_market_candlesticks_for_event_{EVENT_TICKER}.csv",
    index=False
)

if failed_candles:
    df_failed_candles = pd.DataFrame(failed_candles)
    display(df_failed_candles)

    df_failed_candles.to_csv(
        f"kalshi_failed_market_candlesticks_{EVENT_TICKER}.csv",
        index=False
    )


# 7. Event volume summary
dfv = df_full_market_details.copy()

if "volume_fp" in dfv.columns:
    dfv["volume_contracts"] = to_number(dfv["volume_fp"])
elif "volume" in dfv.columns:
    dfv["volume_contracts"] = to_number(dfv["volume"])
else:
    dfv["volume_contracts"] = pd.NA

if "volume_24h_fp" in dfv.columns:
    dfv["volume_24h_contracts"] = to_number(dfv["volume_24h_fp"])
elif "volume_24h" in dfv.columns:
    dfv["volume_24h_contracts"] = to_number(dfv["volume_24h"])
else:
    dfv["volume_24h_contracts"] = pd.NA

if "notional_value_dollars" in dfv.columns:
    dfv["notional_value_dollars_numeric"] = to_number(dfv["notional_value_dollars"])
else:
    dfv["notional_value_dollars_numeric"] = 1.0

dfv["estimated_notional_volume_dollars"] = (
    dfv["volume_contracts"] * dfv["notional_value_dollars_numeric"]
)

volume_by_market_cols = [
    "ticker",
    "_source",
    "title",
    "status",
    "result",
    "volume_fp",
    "volume_contracts",
    "volume_24h_fp",
    "volume_24h_contracts",
    "notional_value_dollars",
    "estimated_notional_volume_dollars"
]

existing_volume_cols = [
    c for c in volume_by_market_cols
    if c in dfv.columns
]

df_event_volume_by_market = dfv[existing_volume_cols].copy()

df_event_volume_summary = pd.DataFrame([{
    "event_ticker": EVENT_TICKER,
    "series_ticker": SERIES_TICKER,
    "n_markets": len(dfv),
    "total_volume_contracts": dfv["volume_contracts"].sum(skipna=True),
    "total_24h_volume_contracts": dfv["volume_24h_contracts"].sum(skipna=True),
    "estimated_total_notional_volume_dollars": dfv["estimated_notional_volume_dollars"].sum(skipna=True),
    "note": "This is not exact cash spent. Exact dollars would require trade-level price * quantity data."
}])

display(df_event_volume_by_market)
display(df_event_volume_summary)

df_event_volume_by_market.to_csv(
    f"kalshi_event_volume_by_market_{EVENT_TICKER}.csv",
    index=False
)

df_event_volume_summary.to_csv(
    f"kalshi_event_volume_summary_{EVENT_TICKER}.csv",
    index=False
)

print("Done. CSV files saved.")


,available_on_brokers,category,collateral_return_type,event_ticker,last_updated_ts,mutually_exclusive,series_ticker,settlement_sources,strike_period,sub_title,title
0,True,Elections,MECNET,KXELECTIONMOVZOHRAN-25,0001-01-01T00:00:00Z,True,KXELECTIONMOVZOHRAN,[{'name': 'Board of Elections in the City of N...,,In 2025,Margin of victory for Zohran Mamdani in the NY...


,featured_image_url,image_url,market_details,settlement_sources
0,https://kalshi-fallback-images.s3.amazonaws.co...,https://kalshi-public-docs.s3.amazonaws.com/se...,"[{'color_code': '#AA00FF', 'image_url': 'https...",[{'name': 'Board of Elections in the City of N...


Live/recent markets: 0
Historical markets: 10
Unique markets: 10


,can_close_early,close_time,created_time,early_close_condition,event_ticker,expected_expiration_time,expiration_time,expiration_value,floor_strike,fractional_trading_enabled,...,volume_24h_fp,volume_fp,yes_ask_dollars,yes_ask_size_fp,yes_bid_dollars,yes_bid_size_fp,yes_sub_title,_source,cap_strike,subtitle
0,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,24.0,True,...,0.00,2715311.00,1.0000,0.00,0.0000,0.00,24% or more,historical,NaN,NaN
1,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,6.0,True,...,0.00,3398478.00,1.0000,0.00,0.0000,0.00,6-8.99%,historical,8.99,NaN
2,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,21.0,True,...,0.00,922621.00,1.0000,0.00,0.0000,0.00,21-23.99%,historical,23.99,NaN
3,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,18.0,True,...,0.00,819153.00,1.0000,0.00,0.0000,0.00,18-20.99%,historical,20.99,NaN
4,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,15.0,True,...,0.00,923736.00,1.0000,0.00,0.0000,0.00,15-17.99%,historical,17.99,NaN
5,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,12.0,True,...,0.00,1454986.00,1.0000,0.00,0.0000,0.00,12-14.99%,historical,14.99,NaN
6,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,9.0,True,...,0.00,2641908.00,1.0000,0.00,0.0000,0.00,9-11.99%,historical,11.99,NaN
7,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121613Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,NaN,True,...,0.00,1488079.00,1.0000,0.00,0.0000,0.00,Less than 0%,historical,0.00,:: Loses
8,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121613Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,3.0,True,...,0.00,1689089.00,1.0000,0.00,0.0000,0.00,3-5.99%,historical,5.99,NaN
9,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121613Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,0.0,True,...,0.00,1256254.00,1.0000,0.00,0.0000,0.00,0-2.99%,historical,2.99,NaN


[1/10] Fetching full market details for KXELECTIONMOVZOHRAN-25-ZMAM-T24...
[2/10] Fetching full market details for KXELECTIONMOVZOHRAN-25-ZMAM-B7.5...
[3/10] Fetching full market details for KXELECTIONMOVZOHRAN-25-ZMAM-B22.5...
[4/10] Fetching full market details for KXELECTIONMOVZOHRAN-25-ZMAM-B19.5...
[5/10] Fetching full market details for KXELECTIONMOVZOHRAN-25-ZMAM-B16.5...
[6/10] Fetching full market details for KXELECTIONMOVZOHRAN-25-ZMAM-B13.5...
[7/10] Fetching full market details for KXELECTIONMOVZOHRAN-25-ZMAM-B10.5...
[8/10] Fetching full market details for KXELECTIONMOVZOHRAN-25-ZMAM-T0...
[9/10] Fetching full market details for KXELECTIONMOVZOHRAN-25-ZMAM-B4.5...
[10/10] Fetching full market details for KXELECTIONMOVZOHRAN-25-ZMAM-B1.5...


...

,end_period_ts,open_interest,price,volume,yes_ask,yes_bid,market_ticker,event_ticker,series_ticker,market_title,market_status,market_result,period_interval,period_label,_source
0,1755144000,5154.00,"{'close': '0.4100', 'high': '0.7400', 'low': '...",5433.00,"{'close': '0.4500', 'high': '1.0000', 'low': '...","{'close': '0.4100', 'high': '0.4700', 'low': '...",KXELECTIONMOVZOHRAN-25-ZMAM-T24,KXELECTIONMOVZOHRAN-25,KXELECTIONMOVZOHRAN,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,1440,daily,historical
1,1755230400,6394.00,"{'close': '0.4000', 'high': '0.7000', 'low': '...",1858.00,"{'close': '0.4800', 'high': '0.7900', 'low': '...","{'close': '0.4000', 'high': '0.4500', 'low': '...",KXELECTIONMOVZOHRAN-25-ZMAM-T24,KXELECTIONMOVZOHRAN-25,KXELECTIONMOVZOHRAN,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,1440,daily,historical
2,1755316800,6429.00,"{'close': '0.3900', 'high': '0.4000', 'low': '...",90.00,"{'close': '0.4700', 'high': '0.4800', 'low': '...","{'close': '0.4100', 'high': '0.4100', 'low': '...",KXELECTIONMOVZOHRAN-25-ZMAM-T24,KXELECTIONMOVZOHRAN-25,KXELECTIONMOVZOHRAN,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,1440,daily,historical
3,1755403200,6429.00,"{'close': '0.4800', 'high': '0.4800', 'low': '...",4.00,"{'close': '0.4700', 'high': '0.4800', 'low': '...","{'close': '0.4200', 'high': '0.4200', 'low': '...",KXELECTIONMOVZOHRAN-25-ZMAM-T24,KXELECTIONMOVZOHRAN-25,KXELECTIONMOVZOHRAN,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,1440,daily,historical
4,1755489600,6840.00,"{'close': '0.4100', 'high': '0.4200', 'low': '...",468.00,"{'close': '0.4600', 'high': '0.4800', 'low': '...","{'close': '0.4100', 'high': '0.4200', 'low': '...",KXELECTIONMOVZOHRAN-25-ZMAM-T24,KXELECTIONMOVZOHRAN-25,KXELECTIONMOVZOHRAN,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,1440,daily,historical
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1105,1764306000,921513.00,"{'close': '0.0100', 'high': '0.0100', 'low': '...",348.00,"{'close': '0.0100', 'high': '0.0100', 'low': '...","{'close': '0.0000', 'high': '0.0000', 'low': '...",KXELECTIONMOVZOHRAN-25-ZMAM-B1.5,KXELECTIONMOVZOHRAN-25,KXELECTIONMOVZOHRAN,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,1440,daily,historical
1106,1764392400,921514.00,"{'close': '0.0100', 'high': '0.0100', 'low': '...",1.00,"{'close': '0.0100', 'high': '0.0100', 'low': '...","{'close': '0.0000', 'high': '0.0000', 'low': '...",KXELECTIONMOVZOHRAN-25-ZMAM-B1.5,KXELECTIONMOVZOHRAN-25,KXELECTIONMOVZOHRAN,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,1440,daily,historical
1107,1764478800,921514.00,"{'close': None, 'high': None, 'low': None, 'me...",0.00,"{'close': '0.0100', 'high': '0.0100', 'low': '...","{'close': '0.0000', 'high': '0.0000', 'low': '...",KXELECTIONMOVZOHRAN-25-ZMAM-B1.5,KXELECTIONMOVZOHRAN-25,KXELECTIONMOVZOHRAN,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,1440,daily,historical
1108,1764565200,921514.00,"{'close': None, 'high': None, 'low': None, 'me...",0.00,"{'close': '0.0100', 'high': '0.0100', 'low': '...","{'close': '0.0000', 'high': '0.0000', 'low': '...",KXELECTIONMOVZOHRAN-25-ZMAM-B1.5,KXELECTIONMOVZOHRAN-25,KXELECTIONMOVZOHRAN,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,1440,daily,historical


,ticker,period_label,error
0,KXELECTIONMOVZOHRAN-25-ZMAM-T24,1min,400 Client Error: Bad Request for url: https:/...
1,KXELECTIONMOVZOHRAN-25-ZMAM-B7.5,1min,400 Client Error: Bad Request for url: https:/...
2,KXELECTIONMOVZOHRAN-25-ZMAM-B22.5,1min,400 Client Error: Bad Request for url: https:/...
3,KXELECTIONMOVZOHRAN-25-ZMAM-B19.5,1min,400 Client Error: Bad Request for url: https:/...
4,KXELECTIONMOVZOHRAN-25-ZMAM-B16.5,1min,400 Client Error: Bad Request for url: https:/...
5,KXELECTIONMOVZOHRAN-25-ZMAM-B13.5,1min,400 Client Error: Bad Request for url: https:/...
6,KXELECTIONMOVZOHRAN-25-ZMAM-B10.5,1min,400 Client Error: Bad Request for url: https:/...
7,KXELECTIONMOVZOHRAN-25-ZMAM-T0,1min,400 Client Error: Bad Request for url: https:/...
8,KXELECTIONMOVZOHRAN-25-ZMAM-B4.5,1min,400 Client Error: Bad Request for url: https:/...
9,KXELECTIONMOVZOHRAN-25-ZMAM-B1.5,1min,400 Client Error: Bad Request for url: https:/...


,ticker,_source,title,status,result,volume_fp,volume_contracts,volume_24h_fp,volume_24h_contracts,notional_value_dollars,estimated_notional_volume_dollars
0,KXELECTIONMOVZOHRAN-25-ZMAM-T24,historical,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,2715311.00,2715311.0,0.00,0.0,1.0000,2715311.0
1,KXELECTIONMOVZOHRAN-25-ZMAM-B7.5,historical,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,3398478.00,3398478.0,0.00,0.0,1.0000,3398478.0
2,KXELECTIONMOVZOHRAN-25-ZMAM-B22.5,historical,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,922621.00,922621.0,0.00,0.0,1.0000,922621.0
3,KXELECTIONMOVZOHRAN-25-ZMAM-B19.5,historical,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,819153.00,819153.0,0.00,0.0,1.0000,819153.0
4,KXELECTIONMOVZOHRAN-25-ZMAM-B16.5,historical,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,923736.00,923736.0,0.00,0.0,1.0000,923736.0
5,KXELECTIONMOVZOHRAN-25-ZMAM-B13.5,historical,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,1454986.00,1454986.0,0.00,0.0,1.0000,1454986.0
6,KXELECTIONMOVZOHRAN-25-ZMAM-B10.5,historical,Margin of victory for Zohran Mamdani in the Ma...,finalized,yes,2641908.00,2641908.0,0.00,0.0,1.0000,2641908.0
7,KXELECTIONMOVZOHRAN-25-ZMAM-T0,historical,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,1488079.00,1488079.0,0.00,0.0,1.0000,1488079.0
8,KXELECTIONMOVZOHRAN-25-ZMAM-B4.5,historical,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,1689089.00,1689089.0,0.00,0.0,1.0000,1689089.0
9,KXELECTIONMOVZOHRAN-25-ZMAM-B1.5,historical,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,1256254.00,1256254.0,0.00,0.0,1.0000,1256254.0


,event_ticker,series_ticker,n_markets,total_volume_contracts,total_24h_volume_contracts,estimated_total_notional_volume_dollars,note
0,KXELECTIONMOVZOHRAN-25,KXELECTIONMOVZOHRAN,10,17309615.0,0.0,17309615.0,This is not exact cash spent. Exact dollars wo...


Done. CSV files saved.


In [17]:
import zipfile
from google.colab import files
import glob

files.download("kalshi_full_market_details_for_event_KXELECTIONMOVZOHRAN-25.csv")
files.download("kalshi_market_summary_for_event_KXELECTIONMOVZOHRAN-25.csv")
files.download("kalshi_daily_market_candlesticks_for_event_KXELECTIONMOVZOHRAN-25.csv")
files.download("kalshi_event_volume_summary_KXELECTIONMOVZOHRAN-25.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
import time
import json
import re
import os
import zipfile
import requests
import pandas as pd
from urllib.parse import urljoin
from google.colab import files

BASE_URL = "https://external-api.kalshi.com/trade-api/v2"
EVENT_TICKER = "KXELECTIONMOVZOHRAN-25"

OUT_DIR = f"kalshi_markets_for_{EVENT_TICKER}"
os.makedirs(OUT_DIR, exist_ok=True)

session = requests.Session()
session.headers.update({
    "User-Agent": "kalshi-event-market-export-colab/1.0"
})


def safe_filename(x):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(x))


def get_json(path, params=None, max_retries=10, timeout=30):
    url = urljoin(BASE_URL + "/", path.lstrip("/"))
    params = params or {}

    for attempt in range(max_retries):
        resp = session.get(url, params=params, timeout=timeout)

        if resp.status_code == 429:
            sleep_s = min(5 * (attempt + 1), 60)
            print(f"Rate limited. Sleeping {sleep_s}s...")
            time.sleep(sleep_s)
            continue

        if resp.status_code >= 400:
            print("Bad URL:", resp.url)
            print("Kalshi response:", resp.text[:3000])
            resp.raise_for_status()

        return resp.json()

    raise RuntimeError(f"Failed after {max_retries} retries: {url}")


def paginate(path, collection_key, params=None, limit=200, sleep_s=0.25):
    params = dict(params or {})
    params["limit"] = limit

    rows = []
    cursor = None

    while True:
        if cursor:
            params["cursor"] = cursor
        else:
            params.pop("cursor", None)

        data = get_json(path, params=params)
        rows.extend(data.get(collection_key, []))

        cursor = data.get("cursor")
        if not cursor:
            break

        time.sleep(sleep_s)

    return rows


def flatten_dict(d, parent_key=""):
    out = {}

    for k, v in d.items():
        key = f"{parent_key}.{k}" if parent_key else k

        if isinstance(v, dict):
            out.update(flatten_dict(v, key))
        elif isinstance(v, list):
            out[key] = json.dumps(v)
        else:
            out[key] = v

    return out


# 1. Get event details
event_data = get_json(f"/events/{EVENT_TICKER}")
event = event_data.get("event", event_data)

df_event_details = pd.DataFrame([event])
df_event_details.to_csv(
    f"{OUT_DIR}/event_details_{EVENT_TICKER}.csv",
    index=False
)

display(df_event_details)


# 2. Get markets from both live/recent and historical endpoints
live_markets = paginate(
    "/markets",
    "markets",
    params={"event_ticker": EVENT_TICKER},
    limit=200
)

for m in live_markets:
    m["_source"] = "live_or_recent"

historical_markets = paginate(
    "/historical/markets",
    "markets",
    params={"event_ticker": EVENT_TICKER},
    limit=200
)

for m in historical_markets:
    m["_source"] = "historical"


# 3. Deduplicate by market ticker
markets_by_ticker = {}

for market in live_markets + historical_markets:
    ticker = market.get("ticker")
    if ticker:
        markets_by_ticker[ticker] = market

event_markets = list(markets_by_ticker.values())

print("Live/recent markets:", len(live_markets))
print("Historical markets:", len(historical_markets))
print("Unique markets:", len(event_markets))


# 4. Fetch full details for every market
full_market_rows = []
failed_markets = []

for i, market in enumerate(event_markets, start=1):
    ticker = market["ticker"]
    source = market.get("_source")

    print(f"[{i}/{len(event_markets)}] Fetching full details for {ticker}...")

    try:
        if source == "historical":
            data = get_json(f"/historical/markets/{ticker}")
            detail_endpoint = f"/historical/markets/{ticker}"
        else:
            data = get_json(f"/markets/{ticker}")
            detail_endpoint = f"/markets/{ticker}"

        full_market = data.get("market", data)

        full_market["_source"] = source
        full_market["_detail_endpoint"] = detail_endpoint
        full_market["_event_ticker_requested"] = EVENT_TICKER

        full_market_rows.append(full_market)

    except Exception as e:
        print(f"Failed {ticker}: {e}")
        failed_markets.append({
            "ticker": ticker,
            "source": source,
            "error": str(e)
        })

    time.sleep(0.25)


# 5. Combined full market CSV
df_full_market_details = pd.DataFrame(full_market_rows)

display(df_full_market_details)

df_full_market_details.to_csv(
    f"{OUT_DIR}/all_market_details_for_event_{EVENT_TICKER}.csv",
    index=False
)


# 6. One separate CSV per market
market_file_index = []

for market in full_market_rows:
    ticker = market.get("ticker")
    flat_market = flatten_dict(market)

    df_one_market = pd.DataFrame([flat_market])

    filename = f"market_{safe_filename(ticker)}.csv"
    filepath = f"{OUT_DIR}/{filename}"

    df_one_market.to_csv(filepath, index=False)

    market_file_index.append({
        "ticker": ticker,
        "csv_file": filename,
        "source": market.get("_source"),
        "status": market.get("status"),
        "result": market.get("result"),
        "title": market.get("title")
    })


df_market_file_index = pd.DataFrame(market_file_index)

display(df_market_file_index)

df_market_file_index.to_csv(
    f"{OUT_DIR}/market_file_index_{EVENT_TICKER}.csv",
    index=False
)


# 7. Failed markets CSV, if any
if failed_markets:
    df_failed_markets = pd.DataFrame(failed_markets)
    display(df_failed_markets)

    df_failed_markets.to_csv(
        f"{OUT_DIR}/failed_market_details_{EVENT_TICKER}.csv",
        index=False
    )


# 8. Zip all CSV files and download
zip_name = f"kalshi_all_market_csvs_{EVENT_TICKER}.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for filename in os.listdir(OUT_DIR):
        filepath = os.path.join(OUT_DIR, filename)
        z.write(filepath, arcname=filename)

print("Created zip:", zip_name)
files.download(zip_name)

,available_on_brokers,category,collateral_return_type,event_ticker,last_updated_ts,mutually_exclusive,series_ticker,settlement_sources,strike_period,sub_title,title
0,True,Elections,MECNET,KXELECTIONMOVZOHRAN-25,0001-01-01T00:00:00Z,True,KXELECTIONMOVZOHRAN,[{'name': 'Board of Elections in the City of N...,,In 2025,Margin of victory for Zohran Mamdani in the NY...


Live/recent markets: 0
Historical markets: 10
Unique markets: 10
[1/10] Fetching full details for KXELECTIONMOVZOHRAN-25-ZMAM-T24...
[2/10] Fetching full details for KXELECTIONMOVZOHRAN-25-ZMAM-B7.5...
[3/10] Fetching full details for KXELECTIONMOVZOHRAN-25-ZMAM-B22.5...
[4/10] Fetching full details for KXELECTIONMOVZOHRAN-25-ZMAM-B19.5...
[5/10] Fetching full details for KXELECTIONMOVZOHRAN-25-ZMAM-B16.5...
[6/10] Fetching full details for KXELECTIONMOVZOHRAN-25-ZMAM-B13.5...
[7/10] Fetching full details for KXELECTIONMOVZOHRAN-25-ZMAM-B10.5...
[8/10] Fetching full details for KXELECTIONMOVZOHRAN-25-ZMAM-T0...
[9/10] Fetching full details for KXELECTIONMOVZOHRAN-25-ZMAM-B4.5...
[10/10] Fetching full details for KXELECTIONMOVZOHRAN-25-ZMAM-B1.5...


,can_close_early,close_time,created_time,early_close_condition,event_ticker,expected_expiration_time,expiration_time,expiration_value,floor_strike,fractional_trading_enabled,...,yes_ask_dollars,yes_ask_size_fp,yes_bid_dollars,yes_bid_size_fp,yes_sub_title,_source,_detail_endpoint,_event_ticker_requested,cap_strike,subtitle
0,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,24.0,True,...,1.0000,0.00,0.0000,0.00,24% or more,historical,/historical/markets/KXELECTIONMOVZOHRAN-25-ZMA...,KXELECTIONMOVZOHRAN-25,NaN,NaN
1,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,6.0,True,...,1.0000,0.00,0.0000,0.00,6-8.99%,historical,/historical/markets/KXELECTIONMOVZOHRAN-25-ZMA...,KXELECTIONMOVZOHRAN-25,8.99,NaN
2,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,21.0,True,...,1.0000,0.00,0.0000,0.00,21-23.99%,historical,/historical/markets/KXELECTIONMOVZOHRAN-25-ZMA...,KXELECTIONMOVZOHRAN-25,23.99,NaN
3,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,18.0,True,...,1.0000,0.00,0.0000,0.00,18-20.99%,historical,/historical/markets/KXELECTIONMOVZOHRAN-25-ZMA...,KXELECTIONMOVZOHRAN-25,20.99,NaN
4,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,15.0,True,...,1.0000,0.00,0.0000,0.00,15-17.99%,historical,/historical/markets/KXELECTIONMOVZOHRAN-25-ZMA...,KXELECTIONMOVZOHRAN-25,17.99,NaN
5,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,12.0,True,...,1.0000,0.00,0.0000,0.00,12-14.99%,historical,/historical/markets/KXELECTIONMOVZOHRAN-25-ZMA...,KXELECTIONMOVZOHRAN-25,14.99,NaN
6,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,9.0,True,...,1.0000,0.00,0.0000,0.00,9-11.99%,historical,/historical/markets/KXELECTIONMOVZOHRAN-25-ZMA...,KXELECTIONMOVZOHRAN-25,11.99,NaN
7,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121613Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,NaN,True,...,1.0000,0.00,0.0000,0.00,Less than 0%,historical,/historical/markets/KXELECTIONMOVZOHRAN-25-ZMA...,KXELECTIONMOVZOHRAN-25,0.00,:: Loses
8,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121613Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,3.0,True,...,1.0000,0.00,0.0000,0.00,3-5.99%,historical,/historical/markets/KXELECTIONMOVZOHRAN-25-ZMA...,KXELECTIONMOVZOHRAN-25,5.99,NaN
9,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121613Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,0.0,True,...,1.0000,0.00,0.0000,0.00,0-2.99%,historical,/historical/markets/KXELECTIONMOVZOHRAN-25-ZMA...,KXELECTIONMOVZOHRAN-25,2.99,NaN


,ticker,csv_file,source,status,result,title
0,KXELECTIONMOVZOHRAN-25-ZMAM-T24,market_KXELECTIONMOVZOHRAN-25-ZMAM-T24.csv,historical,finalized,no,Margin of victory for Zohran Mamdani in the Ma...
1,KXELECTIONMOVZOHRAN-25-ZMAM-B7.5,market_KXELECTIONMOVZOHRAN-25-ZMAM-B7.5.csv,historical,finalized,no,Margin of victory for Zohran Mamdani in the Ma...
2,KXELECTIONMOVZOHRAN-25-ZMAM-B22.5,market_KXELECTIONMOVZOHRAN-25-ZMAM-B22.5.csv,historical,finalized,no,Margin of victory for Zohran Mamdani in the Ma...
3,KXELECTIONMOVZOHRAN-25-ZMAM-B19.5,market_KXELECTIONMOVZOHRAN-25-ZMAM-B19.5.csv,historical,finalized,no,Margin of victory for Zohran Mamdani in the Ma...
4,KXELECTIONMOVZOHRAN-25-ZMAM-B16.5,market_KXELECTIONMOVZOHRAN-25-ZMAM-B16.5.csv,historical,finalized,no,Margin of victory for Zohran Mamdani in the Ma...
5,KXELECTIONMOVZOHRAN-25-ZMAM-B13.5,market_KXELECTIONMOVZOHRAN-25-ZMAM-B13.5.csv,historical,finalized,no,Margin of victory for Zohran Mamdani in the Ma...
6,KXELECTIONMOVZOHRAN-25-ZMAM-B10.5,market_KXELECTIONMOVZOHRAN-25-ZMAM-B10.5.csv,historical,finalized,yes,Margin of victory for Zohran Mamdani in the Ma...
7,KXELECTIONMOVZOHRAN-25-ZMAM-T0,market_KXELECTIONMOVZOHRAN-25-ZMAM-T0.csv,historical,finalized,no,Margin of victory for Zohran Mamdani in the Ma...
8,KXELECTIONMOVZOHRAN-25-ZMAM-B4.5,market_KXELECTIONMOVZOHRAN-25-ZMAM-B4.5.csv,historical,finalized,no,Margin of victory for Zohran Mamdani in the Ma...
9,KXELECTIONMOVZOHRAN-25-ZMAM-B1.5,market_KXELECTIONMOVZOHRAN-25-ZMAM-B1.5.csv,historical,finalized,no,Margin of victory for Zohran Mamdani in the Ma...


Created zip: kalshi_all_market_csvs_KXELECTIONMOVZOHRAN-25.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
files.download(zip_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
import time
import os
import re
import zipfile
import pandas as pd
from google.colab import files

EVENT_TICKER = "KXELECTIONMOVZOHRAN-25"
SERIES_TICKER = "KXELECTIONMOVZOHRAN"

OUT_DIR = f"kalshi_daily_candlesticks_for_{EVENT_TICKER}"
os.makedirs(OUT_DIR, exist_ok=True)


def safe_filename(x):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(x))


# Use df_full_market_details if it exists; otherwise use df_event_markets
try:
    df_markets_for_candles = df_full_market_details.copy()
except NameError:
    df_markets_for_candles = df_event_markets.copy()


all_daily_candles = []
failed_daily_candles = []

for i, row in df_markets_for_candles.iterrows():
    market_ticker = row["ticker"]
    source = row.get("_source", "historical")

    open_time = pd.to_datetime(row.get("open_time"), utc=True, errors="coerce")
    end_time = pd.to_datetime(
        row.get("close_time") or row.get("expiration_time") or row.get("settlement_ts"),
        utc=True,
        errors="coerce"
    )

    if pd.isna(open_time) or pd.isna(end_time):
        print(f"Skipping {market_ticker}: missing open/end time")
        failed_daily_candles.append({
            "ticker": market_ticker,
            "reason": "missing open_time or end_time"
        })
        continue

    start_ts = int(open_time.timestamp())
    end_ts = int(end_time.timestamp())

    print(f"[{i+1}/{len(df_markets_for_candles)}] Getting daily candles for {market_ticker}...")

    try:
        if source == "historical":
            path = f"/historical/markets/{market_ticker}/candlesticks"
        else:
            path = f"/series/{SERIES_TICKER}/markets/{market_ticker}/candlesticks"

        data = get_json(
            path,
            params={
                "start_ts": start_ts,
                "end_ts": end_ts,
                "period_interval": 1440
            }
        )

        candles = data.get("candlesticks", [])

        market_candles = []

        for candle in candles:
            candle["market_ticker"] = market_ticker
            candle["event_ticker"] = EVENT_TICKER
            candle["series_ticker"] = SERIES_TICKER
            candle["market_title"] = row.get("title")
            candle["market_status"] = row.get("status")
            candle["market_result"] = row.get("result")
            candle["market_open_time"] = open_time
            candle["market_end_time"] = end_time
            candle["period_interval"] = 1440
            candle["period_label"] = "daily"
            candle["_source"] = source

            market_candles.append(candle)
            all_daily_candles.append(candle)

        df_one_market_daily = pd.DataFrame(market_candles)

        df_one_market_daily.to_csv(
            f"{OUT_DIR}/daily_candlesticks_{safe_filename(market_ticker)}.csv",
            index=False
        )

    except Exception as e:
        print(f"Failed {market_ticker}: {e}")
        failed_daily_candles.append({
            "ticker": market_ticker,
            "reason": str(e)
        })

    time.sleep(0.25)


df_all_daily_market_candlesticks = pd.DataFrame(all_daily_candles)

display(df_all_daily_market_candlesticks)

print("Total daily candlestick rows:", len(df_all_daily_market_candlesticks))
print("Columns:", df_all_daily_market_candlesticks.columns.tolist())

df_all_daily_market_candlesticks.to_csv(
    f"{OUT_DIR}/all_daily_market_candlesticks_{EVENT_TICKER}.csv",
    index=False
)

if failed_daily_candles:
    df_failed_daily_candles = pd.DataFrame(failed_daily_candles)
    display(df_failed_daily_candles)

    df_failed_daily_candles.to_csv(
        f"{OUT_DIR}/failed_daily_candlesticks_{EVENT_TICKER}.csv",
        index=False
    )


# Zip and download
zip_name = f"kalshi_daily_market_candlesticks_{EVENT_TICKER}.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for filename in os.listdir(OUT_DIR):
        filepath = os.path.join(OUT_DIR, filename)
        z.write(filepath, arcname=filename)

files.download(zip_name)

[1/10] Getting daily candles for KXELECTIONMOVZOHRAN-25-ZMAM-T24...
[2/10] Getting daily candles for KXELECTIONMOVZOHRAN-25-ZMAM-B7.5...
[3/10] Getting daily candles for KXELECTIONMOVZOHRAN-25-ZMAM-B22.5...
[4/10] Getting daily candles for KXELECTIONMOVZOHRAN-25-ZMAM-B19.5...
[5/10] Getting daily candles for KXELECTIONMOVZOHRAN-25-ZMAM-B16.5...
[6/10] Getting daily candles for KXELECTIONMOVZOHRAN-25-ZMAM-B13.5...
[7/10] Getting daily candles for KXELECTIONMOVZOHRAN-25-ZMAM-B10.5...
[8/10] Getting daily candles for KXELECTIONMOVZOHRAN-25-ZMAM-T0...
[9/10] Getting daily candles for KXELECTIONMOVZOHRAN-25-ZMAM-B4.5...
[10/10] Getting daily candles for KXELECTIONMOVZOHRAN-25-ZMAM-B1.5...


,end_period_ts,open_interest,price,volume,yes_ask,yes_bid,market_ticker,event_ticker,series_ticker,market_title,market_status,market_result,market_open_time,market_end_time,period_interval,period_label,_source
0,1755144000,5154.00,"{'close': '0.4100', 'high': '0.7400', 'low': '...",5433.00,"{'close': '0.4500', 'high': '1.0000', 'low': '...","{'close': '0.4100', 'high': '0.4700', 'low': '...",KXELECTIONMOVZOHRAN-25-ZMAM-T24,KXELECTIONMOVZOHRAN-25,KXELECTIONMOVZOHRAN,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,1440,daily,historical
1,1755230400,6394.00,"{'close': '0.4000', 'high': '0.7000', 'low': '...",1858.00,"{'close': '0.4800', 'high': '0.7900', 'low': '...","{'close': '0.4000', 'high': '0.4500', 'low': '...",KXELECTIONMOVZOHRAN-25-ZMAM-T24,KXELECTIONMOVZOHRAN-25,KXELECTIONMOVZOHRAN,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,1440,daily,historical
2,1755316800,6429.00,"{'close': '0.3900', 'high': '0.4000', 'low': '...",90.00,"{'close': '0.4700', 'high': '0.4800', 'low': '...","{'close': '0.4100', 'high': '0.4100', 'low': '...",KXELECTIONMOVZOHRAN-25-ZMAM-T24,KXELECTIONMOVZOHRAN-25,KXELECTIONMOVZOHRAN,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,1440,daily,historical
3,1755403200,6429.00,"{'close': '0.4800', 'high': '0.4800', 'low': '...",4.00,"{'close': '0.4700', 'high': '0.4800', 'low': '...","{'close': '0.4200', 'high': '0.4200', 'low': '...",KXELECTIONMOVZOHRAN-25-ZMAM-T24,KXELECTIONMOVZOHRAN-25,KXELECTIONMOVZOHRAN,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,1440,daily,historical
4,1755489600,6840.00,"{'close': '0.4100', 'high': '0.4200', 'low': '...",468.00,"{'close': '0.4600', 'high': '0.4800', 'low': '...","{'close': '0.4100', 'high': '0.4200', 'low': '...",KXELECTIONMOVZOHRAN-25-ZMAM-T24,KXELECTIONMOVZOHRAN-25,KXELECTIONMOVZOHRAN,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,1440,daily,historical
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1105,1764306000,921513.00,"{'close': '0.0100', 'high': '0.0100', 'low': '...",348.00,"{'close': '0.0100', 'high': '0.0100', 'low': '...","{'close': '0.0000', 'high': '0.0000', 'low': '...",KXELECTIONMOVZOHRAN-25-ZMAM-B1.5,KXELECTIONMOVZOHRAN-25,KXELECTIONMOVZOHRAN,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,1440,daily,historical
1106,1764392400,921514.00,"{'close': '0.0100', 'high': '0.0100', 'low': '...",1.00,"{'close': '0.0100', 'high': '0.0100', 'low': '...","{'close': '0.0000', 'high': '0.0000', 'low': '...",KXELECTIONMOVZOHRAN-25-ZMAM-B1.5,KXELECTIONMOVZOHRAN-25,KXELECTIONMOVZOHRAN,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,1440,daily,historical
1107,1764478800,921514.00,"{'close': None, 'high': None, 'low': None, 'me...",0.00,"{'close': '0.0100', 'high': '0.0100', 'low': '...","{'close': '0.0000', 'high': '0.0000', 'low': '...",KXELECTIONMOVZOHRAN-25-ZMAM-B1.5,KXELECTIONMOVZOHRAN-25,KXELECTIONMOVZOHRAN,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,1440,daily,historical
1108,1764565200,921514.00,"{'close': None, 'high': None, 'low': None, 'me...",0.00,"{'close': '0.0100', 'high': '0.0100', 'low': '...","{'close': '0.0000', 'high': '0.0000', 'low': '...",KXELECTIONMOVZOHRAN-25-ZMAM-B1.5,KXELECTIONMOVZOHRAN-25,KXELECTIONMOVZOHRAN,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,1440,daily,historical


Total daily candlestick rows: 1110
Columns: ['end_period_ts', 'open_interest', 'price', 'volume', 'yes_ask', 'yes_bid', 'market_ticker', 'event_ticker', 'series_ticker', 'market_title', 'market_status', 'market_result', 'market_open_time', 'market_end_time', 'period_interval', 'period_label', '_source']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
import json
import time
import requests
import pandas as pd
from urllib.parse import urljoin
from google.colab import files

BASE_URL = "https://external-api.kalshi.com/trade-api/v2"

MARKET_TICKER = "KXELECTIONMOVZOHRAN-25-ZMAM-T24"

session = requests.Session()
session.headers.update({
    "User-Agent": "kalshi-daily-historical-candlesticks-colab/1.0"
})


def get_json(path, params=None, max_retries=10, timeout=30):
    url = urljoin(BASE_URL + "/", path.lstrip("/"))
    params = params or {}

    for attempt in range(max_retries):
        resp = session.get(url, params=params, timeout=timeout)

        if resp.status_code == 429:
            sleep_s = min(5 * (attempt + 1), 60)
            print(f"Rate limited. Sleeping {sleep_s}s...")
            time.sleep(sleep_s)
            continue

        try:
            data = resp.json()
        except Exception:
            data = {"raw_text": resp.text}

        if resp.status_code >= 400:
            print("Bad URL:", resp.url)
            print("Kalshi response:", json.dumps(data, indent=2)[:3000])
            resp.raise_for_status()

        return data

    raise RuntimeError(f"Failed after {max_retries} retries: {url}")


def flatten_dict(d, parent_key=""):
    out = {}

    for k, v in d.items():
        key = f"{parent_key}.{k}" if parent_key else k

        if isinstance(v, dict):
            out.update(flatten_dict(v, key))
        elif isinstance(v, list):
            out[key] = json.dumps(v)
        else:
            out[key] = v

    return out


# 1. Get historical market details to determine full date range
market_data = get_json(f"/historical/markets/{MARKET_TICKER}")
market = market_data.get("market", market_data)

df_market_details = pd.DataFrame([market])
display(df_market_details)

open_time = pd.to_datetime(market.get("open_time"), utc=True, errors="coerce")

end_time = pd.to_datetime(
    market.get("close_time") or market.get("expiration_time") or market.get("settlement_ts"),
    utc=True,
    errors="coerce"
)

if pd.isna(open_time):
    raise ValueError("Market open_time is missing. Cannot create start_ts.")

if pd.isna(end_time):
    raise ValueError("Market close/expiration/settlement time is missing. Cannot create end_ts.")

start_ts = int(open_time.timestamp())
end_ts = int(end_time.timestamp())

print("Market ticker:", MARKET_TICKER)
print("Start:", open_time)
print("End:", end_time)


# 2. Get DAILY historical candlesticks
candles_data = get_json(
    f"/historical/markets/{MARKET_TICKER}/candlesticks",
    params={
        "start_ts": start_ts,
        "end_ts": end_ts,
        "period_interval": 1440
    }
)

candles = candles_data.get("candlesticks", [])

daily_rows = []

for candle in candles:
    row = flatten_dict(candle)

    row["market_ticker"] = MARKET_TICKER
    row["period_interval"] = 1440
    row["period_label"] = "daily"
    row["market_open_time"] = open_time
    row["market_end_time"] = end_time

    daily_rows.append(row)


# 3. Create DataFrame
df_daily_historical_candlesticks = pd.DataFrame(daily_rows)

display(df_daily_historical_candlesticks)

print("Daily candlestick rows:", len(df_daily_historical_candlesticks))
print("Columns:")
print(df_daily_historical_candlesticks.columns.tolist())


# 4. Save and download CSV
csv_filename = f"kalshi_daily_historical_candlesticks_{MARKET_TICKER}.csv"

df_daily_historical_candlesticks.to_csv(csv_filename, index=False)

files.download(csv_filename)

,can_close_early,close_time,created_time,early_close_condition,event_ticker,expected_expiration_time,expiration_time,expiration_value,floor_strike,fractional_trading_enabled,...,ticker,title,updated_time,volume_24h_fp,volume_fp,yes_ask_dollars,yes_ask_size_fp,yes_bid_dollars,yes_bid_size_fp,yes_sub_title
0,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,24,True,...,KXELECTIONMOVZOHRAN-25-ZMAM-T24,Margin of victory for Zohran Mamdani in the Ma...,2026-03-04T22:43:52.878979Z,0.00,2715311.00,1.0000,0.00,0.0000,0.00,24% or more


Market ticker: KXELECTIONMOVZOHRAN-25-ZMAM-T24
Start: 2025-08-13 14:00:00+00:00
End: 2025-12-03 01:44:48.719034+00:00


,end_period_ts,open_interest,price.close,price.high,price.low,price.mean,price.open,price.previous,volume,yes_ask.close,...,yes_ask.open,yes_bid.close,yes_bid.high,yes_bid.low,yes_bid.open,market_ticker,period_interval,period_label,market_open_time,market_end_time
0,1755144000,5154.00,0.4100,0.7400,0.1900,0.3164,0.1900,None,5433.00,0.4500,...,1.0000,0.4100,0.4700,0.0800,0.0800,KXELECTIONMOVZOHRAN-25-ZMAM-T24,1440,daily,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00
1,1755230400,6394.00,0.4000,0.7000,0.4000,0.4676,0.4100,0.4100,1858.00,0.4800,...,0.4500,0.4000,0.4500,0.3900,0.4100,KXELECTIONMOVZOHRAN-25-ZMAM-T24,1440,daily,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00
2,1755316800,6429.00,0.3900,0.4000,0.3900,0.3963,0.4000,0.4000,90.00,0.4700,...,0.4800,0.4100,0.4100,0.3900,0.4000,KXELECTIONMOVZOHRAN-25-ZMAM-T24,1440,daily,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00
3,1755403200,6429.00,0.4800,0.4800,0.4700,0.4750,0.4700,0.3900,4.00,0.4700,...,0.4700,0.4200,0.4200,0.4100,0.4100,KXELECTIONMOVZOHRAN-25-ZMAM-T24,1440,daily,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00
4,1755489600,6840.00,0.4100,0.4200,0.4100,0.4100,0.4200,0.4800,468.00,0.4600,...,0.4700,0.4100,0.4200,0.4100,0.4200,KXELECTIONMOVZOHRAN-25-ZMAM-T24,1440,daily,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,1764306000,1083612.00,0.0100,0.0100,0.0100,0.0100,0.0100,0.0100,904.00,0.0100,...,0.0100,0.0000,0.0000,0.0000,0.0000,KXELECTIONMOVZOHRAN-25-ZMAM-T24,1440,daily,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00
107,1764392400,1085458.00,0.0100,0.0100,0.0100,0.0100,0.0100,0.0100,1879.00,0.0100,...,0.0100,0.0000,0.0000,0.0000,0.0000,KXELECTIONMOVZOHRAN-25-ZMAM-T24,1440,daily,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00
108,1764478800,1085458.00,0.0100,0.0100,0.0100,0.0100,0.0100,0.0100,298.00,0.0100,...,0.0100,0.0000,0.0000,0.0000,0.0000,KXELECTIONMOVZOHRAN-25-ZMAM-T24,1440,daily,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00
109,1764565200,1085458.00,0.0100,0.0100,0.0100,0.0100,0.0100,0.0100,447.00,0.0100,...,0.0100,0.0000,0.0000,0.0000,0.0000,KXELECTIONMOVZOHRAN-25-ZMAM-T24,1440,daily,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00


Daily candlestick rows: 111
Columns:
['end_period_ts', 'open_interest', 'price.close', 'price.high', 'price.low', 'price.mean', 'price.open', 'price.previous', 'volume', 'yes_ask.close', 'yes_ask.high', 'yes_ask.low', 'yes_ask.open', 'yes_bid.close', 'yes_bid.high', 'yes_bid.low', 'yes_bid.open', 'market_ticker', 'period_interval', 'period_label', 'market_open_time', 'market_end_time']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
import json
import time
import os
import re
import zipfile
import requests
import pandas as pd
from urllib.parse import urljoin
from google.colab import files

BASE_URL = "https://external-api.kalshi.com/trade-api/v2"

market_tickers = [
    "KXELECTIONMOVZOHRAN-25-ZMAM-T24",
    "KXELECTIONMOVZOHRAN-25-ZMAM-B7.5",
    "KXELECTIONMOVZOHRAN-25-ZMAM-B22.5",
    "KXELECTIONMOVZOHRAN-25-ZMAM-B19.5",
    "KXELECTIONMOVZOHRAN-25-ZMAM-B16.5",
    "KXELECTIONMOVZOHRAN-25-ZMAM-B13.5",
    "KXELECTIONMOVZOHRAN-25-ZMAM-B10.5",
    "KXELECTIONMOVZOHRAN-25-ZMAM-T0",
    "KXELECTIONMOVZOHRAN-25-ZMAM-B4.5",
    "KXELECTIONMOVZOHRAN-25-ZMAM-B1.5",
]

OUT_DIR = "kalshi_daily_candlesticks_zmam_markets"
os.makedirs(OUT_DIR, exist_ok=True)

session = requests.Session()
session.headers.update({
    "User-Agent": "kalshi-bulk-daily-historical-candlesticks-colab/1.0"
})


def safe_filename(x):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(x))


def get_json(path, params=None, max_retries=10, timeout=30):
    url = urljoin(BASE_URL + "/", path.lstrip("/"))
    params = params or {}

    for attempt in range(max_retries):
        resp = session.get(url, params=params, timeout=timeout)

        if resp.status_code == 429:
            sleep_s = min(5 * (attempt + 1), 60)
            print(f"Rate limited. Sleeping {sleep_s}s...")
            time.sleep(sleep_s)
            continue

        try:
            data = resp.json()
        except Exception:
            data = {"raw_text": resp.text}

        if resp.status_code >= 400:
            print("Bad URL:", resp.url)
            print("Kalshi response:", json.dumps(data, indent=2)[:3000])
            resp.raise_for_status()

        return data

    raise RuntimeError(f"Failed after {max_retries} retries: {url}")


def flatten_dict(d, parent_key=""):
    out = {}

    for k, v in d.items():
        key = f"{parent_key}.{k}" if parent_key else k

        if isinstance(v, dict):
            out.update(flatten_dict(v, key))
        elif isinstance(v, list):
            out[key] = json.dumps(v)
        else:
            out[key] = v

    return out


all_daily_rows = []
market_detail_rows = []
failed_tickers = []

for i, market_ticker in enumerate(market_tickers, start=1):
    print(f"[{i}/{len(market_tickers)}] Processing {market_ticker}...")

    try:
        # 1. Get historical market details
        market_data = get_json(f"/historical/markets/{market_ticker}")
        market = market_data.get("market", market_data)

        market_detail_rows.append(market)

        open_time = pd.to_datetime(
            market.get("open_time"),
            utc=True,
            errors="coerce"
        )

        end_time = pd.to_datetime(
            market.get("close_time") or market.get("expiration_time") or market.get("settlement_ts"),
            utc=True,
            errors="coerce"
        )

        if pd.isna(open_time) or pd.isna(end_time):
            raise ValueError("Missing open_time or end_time")

        start_ts = int(open_time.timestamp())
        end_ts = int(end_time.timestamp())

        print("  Start:", open_time)
        print("  End:", end_time)

        # 2. Get daily candlesticks
        candles_data = get_json(
            f"/historical/markets/{market_ticker}/candlesticks",
            params={
                "start_ts": start_ts,
                "end_ts": end_ts,
                "period_interval": 1440
            }
        )

        candles = candles_data.get("candlesticks", [])
        print("  Daily candles:", len(candles))

        one_market_rows = []

        for candle in candles:
            row = flatten_dict(candle)

            row["market_ticker"] = market_ticker
            row["period_interval"] = 1440
            row["period_label"] = "daily"
            row["market_open_time"] = open_time
            row["market_end_time"] = end_time
            row["market_title"] = market.get("title")
            row["market_status"] = market.get("status")
            row["market_result"] = market.get("result")
            row["event_ticker"] = market.get("event_ticker")
            row["series_ticker"] = market.get("series_ticker")

            one_market_rows.append(row)
            all_daily_rows.append(row)

        # 3. Save individual market CSV
        df_one_market = pd.DataFrame(one_market_rows)

        one_market_csv = f"{OUT_DIR}/daily_candlesticks_{safe_filename(market_ticker)}.csv"
        df_one_market.to_csv(one_market_csv, index=False)

    except Exception as e:
        print(f"  Failed {market_ticker}: {e}")
        failed_tickers.append({
            "ticker": market_ticker,
            "error": str(e)
        })

    time.sleep(0.25)


# 4. Combined daily candlesticks DataFrame
df_all_daily_candlesticks = pd.DataFrame(all_daily_rows)

display(df_all_daily_candlesticks)

print("Total daily candlestick rows:", len(df_all_daily_candlesticks))
print("Columns:", df_all_daily_candlesticks.columns.tolist())

combined_csv = f"{OUT_DIR}/all_daily_candlesticks_zmam_markets.csv"
df_all_daily_candlesticks.to_csv(combined_csv, index=False)


# 5. Market details DataFrame
df_market_details = pd.DataFrame(market_detail_rows)

display(df_market_details)

market_details_csv = f"{OUT_DIR}/market_details_zmam_markets.csv"
df_market_details.to_csv(market_details_csv, index=False)


# 6. Failed ticker DataFrame
if failed_tickers:
    df_failed_tickers = pd.DataFrame(failed_tickers)
    display(df_failed_tickers)

    failed_csv = f"{OUT_DIR}/failed_zmam_market_tickers.csv"
    df_failed_tickers.to_csv(failed_csv, index=False)


# 7. Zip and download everything
zip_name = "kalshi_daily_candlesticks_zmam_markets.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for filename in os.listdir(OUT_DIR):
        filepath = os.path.join(OUT_DIR, filename)
        z.write(filepath, arcname=filename)

files.download(zip_name)

[1/10] Processing KXELECTIONMOVZOHRAN-25-ZMAM-T24...
  Start: 2025-08-13 14:00:00+00:00
  End: 2025-12-03 01:44:48.719034+00:00
  Daily candles: 111
[2/10] Processing KXELECTIONMOVZOHRAN-25-ZMAM-B7.5...
  Start: 2025-08-13 14:00:00+00:00
  End: 2025-12-03 01:44:48.719034+00:00
  Daily candles: 111
[3/10] Processing KXELECTIONMOVZOHRAN-25-ZMAM-B22.5...
  Start: 2025-08-13 14:00:00+00:00
  End: 2025-12-03 01:44:48.719034+00:00
  Daily candles: 111
[4/10] Processing KXELECTIONMOVZOHRAN-25-ZMAM-B19.5...
  Start: 2025-08-13 14:00:00+00:00
  End: 2025-12-03 01:44:48.719034+00:00
  Daily candles: 111
[5/10] Processing KXELECTIONMOVZOHRAN-25-ZMAM-B16.5...
  Start: 2025-08-13 14:00:00+00:00
  End: 2025-12-03 01:44:48.719034+00:00
  Daily candles: 111
[6/10] Processing KXELECTIONMOVZOHRAN-25-ZMAM-B13.5...
  Start: 2025-08-13 14:00:00+00:00
  End: 2025-12-03 01:44:48.719034+00:00
  Daily candles: 111
[7/10] Processing KXELECTIONMOVZOHRAN-25-ZMAM-B10.5...
  Start: 2025-08-13 14:00:00+00:00
  End: 

,end_period_ts,open_interest,price.close,price.high,price.low,price.mean,price.open,price.previous,volume,yes_ask.close,...,market_ticker,period_interval,period_label,market_open_time,market_end_time,market_title,market_status,market_result,event_ticker,series_ticker
0,1755144000,5154.00,0.4100,0.7400,0.1900,0.3164,0.1900,None,5433.00,0.4500,...,KXELECTIONMOVZOHRAN-25-ZMAM-T24,1440,daily,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,KXELECTIONMOVZOHRAN-25,None
1,1755230400,6394.00,0.4000,0.7000,0.4000,0.4676,0.4100,0.4100,1858.00,0.4800,...,KXELECTIONMOVZOHRAN-25-ZMAM-T24,1440,daily,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,KXELECTIONMOVZOHRAN-25,None
2,1755316800,6429.00,0.3900,0.4000,0.3900,0.3963,0.4000,0.4000,90.00,0.4700,...,KXELECTIONMOVZOHRAN-25-ZMAM-T24,1440,daily,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,KXELECTIONMOVZOHRAN-25,None
3,1755403200,6429.00,0.4800,0.4800,0.4700,0.4750,0.4700,0.3900,4.00,0.4700,...,KXELECTIONMOVZOHRAN-25-ZMAM-T24,1440,daily,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,KXELECTIONMOVZOHRAN-25,None
4,1755489600,6840.00,0.4100,0.4200,0.4100,0.4100,0.4200,0.4800,468.00,0.4600,...,KXELECTIONMOVZOHRAN-25-ZMAM-T24,1440,daily,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,KXELECTIONMOVZOHRAN-25,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1105,1764306000,921513.00,0.0100,0.0100,0.0100,0.0100,0.0100,0.0100,348.00,0.0100,...,KXELECTIONMOVZOHRAN-25-ZMAM-B1.5,1440,daily,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,KXELECTIONMOVZOHRAN-25,None
1106,1764392400,921514.00,0.0100,0.0100,0.0100,0.0100,0.0100,0.0100,1.00,0.0100,...,KXELECTIONMOVZOHRAN-25-ZMAM-B1.5,1440,daily,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,KXELECTIONMOVZOHRAN-25,None
1107,1764478800,921514.00,None,None,None,None,None,0.0100,0.00,0.0100,...,KXELECTIONMOVZOHRAN-25-ZMAM-B1.5,1440,daily,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,KXELECTIONMOVZOHRAN-25,None
1108,1764565200,921514.00,None,None,None,None,None,0.0100,0.00,0.0100,...,KXELECTIONMOVZOHRAN-25-ZMAM-B1.5,1440,daily,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,Margin of victory for Zohran Mamdani in the Ma...,finalized,no,KXELECTIONMOVZOHRAN-25,None


Total daily candlestick rows: 1110
Columns: ['end_period_ts', 'open_interest', 'price.close', 'price.high', 'price.low', 'price.mean', 'price.open', 'price.previous', 'volume', 'yes_ask.close', 'yes_ask.high', 'yes_ask.low', 'yes_ask.open', 'yes_bid.close', 'yes_bid.high', 'yes_bid.low', 'yes_bid.open', 'market_ticker', 'period_interval', 'period_label', 'market_open_time', 'market_end_time', 'market_title', 'market_status', 'market_result', 'event_ticker', 'series_ticker']


,can_close_early,close_time,created_time,early_close_condition,event_ticker,expected_expiration_time,expiration_time,expiration_value,floor_strike,fractional_trading_enabled,...,updated_time,volume_24h_fp,volume_fp,yes_ask_dollars,yes_ask_size_fp,yes_bid_dollars,yes_bid_size_fp,yes_sub_title,cap_strike,subtitle
0,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,24.0,True,...,2026-03-04T22:43:52.878979Z,0.00,2715311.00,1.0000,0.00,0.0000,0.00,24% or more,NaN,NaN
1,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,6.0,True,...,2026-03-04T22:43:52.624601Z,0.00,3398478.00,1.0000,0.00,0.0000,0.00,6-8.99%,8.99,NaN
2,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,21.0,True,...,2026-03-04T22:43:52.829531Z,0.00,922621.00,1.0000,0.00,0.0000,0.00,21-23.99%,23.99,NaN
3,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,18.0,True,...,2026-03-04T22:43:52.77753Z,0.00,819153.00,1.0000,0.00,0.0000,0.00,18-20.99%,20.99,NaN
4,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,15.0,True,...,2026-03-04T22:43:52.739289Z,0.00,923736.00,1.0000,0.00,0.0000,0.00,15-17.99%,17.99,NaN
5,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,12.0,True,...,2026-03-04T22:43:52.700178Z,0.00,1454986.00,1.0000,0.00,0.0000,0.00,12-14.99%,14.99,NaN
6,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121614Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,9.0,True,...,2026-03-04T22:43:52.661638Z,0.00,2641908.00,1.0000,0.00,0.0000,0.00,9-11.99%,11.99,NaN
7,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121613Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,NaN,True,...,2026-03-04T22:43:52.494901Z,0.00,1488079.00,1.0000,0.00,0.0000,0.00,Less than 0%,0.00,:: Loses
8,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121613Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,3.0,True,...,2026-03-04T22:43:52.57737Z,0.00,1689089.00,1.0000,0.00,0.0000,0.00,3-5.99%,5.99,NaN
9,True,2025-12-03T01:44:48.719034Z,2025-08-12T21:46:06.121613Z,This market will close and expire early if the...,KXELECTIONMOVZOHRAN-25,2025-11-04T15:00:00Z,2026-11-04T15:00:00Z,9-11.99%,0.0,True,...,2026-03-04T22:43:52.539059Z,0.00,1256254.00,1.0000,0.00,0.0000,0.00,0-2.99%,2.99,NaN


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
import json
import time
import os
import re
import zipfile
import requests
import pandas as pd
from urllib.parse import urljoin
from google.colab import files

BASE_URL = "https://external-api.kalshi.com/trade-api/v2"

market_tickers = [
    "KXELECTIONMOVZOHRAN-25-ZMAM-T24",
    "KXELECTIONMOVZOHRAN-25-ZMAM-B7.5",
    "KXELECTIONMOVZOHRAN-25-ZMAM-B22.5",
    "KXELECTIONMOVZOHRAN-25-ZMAM-B19.5",
    "KXELECTIONMOVZOHRAN-25-ZMAM-B16.5",
    "KXELECTIONMOVZOHRAN-25-ZMAM-B13.5",
    "KXELECTIONMOVZOHRAN-25-ZMAM-B10.5",
    "KXELECTIONMOVZOHRAN-25-ZMAM-T0",
    "KXELECTIONMOVZOHRAN-25-ZMAM-B4.5",
    "KXELECTIONMOVZOHRAN-25-ZMAM-B1.5",
]

OUT_DIR = "kalshi_zmam_daily_candlesticks_with_market_info"
os.makedirs(OUT_DIR, exist_ok=True)

session = requests.Session()
session.headers.update({
    "User-Agent": "kalshi-zmam-daily-candlesticks-with-info-colab/1.0"
})


def safe_filename(x):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(x))


def get_json(path, params=None, max_retries=10, timeout=30):
    url = urljoin(BASE_URL + "/", path.lstrip("/"))
    params = params or {}

    for attempt in range(max_retries):
        resp = session.get(url, params=params, timeout=timeout)

        if resp.status_code == 429:
            sleep_s = min(5 * (attempt + 1), 60)
            print(f"Rate limited. Sleeping {sleep_s}s...")
            time.sleep(sleep_s)
            continue

        try:
            data = resp.json()
        except Exception:
            data = {"raw_text": resp.text}

        if resp.status_code >= 400:
            print("Bad URL:", resp.url)
            print("Kalshi response:", json.dumps(data, indent=2)[:3000])
            resp.raise_for_status()

        return data

    raise RuntimeError(f"Failed after {max_retries} retries: {url}")


def flatten_dict(d, parent_key=""):
    out = {}

    for k, v in d.items():
        key = f"{parent_key}.{k}" if parent_key else k

        if isinstance(v, dict):
            out.update(flatten_dict(v, key))
        elif isinstance(v, list):
            out[key] = json.dumps(v)
        else:
            out[key] = v

    return out


all_daily_rows = []
market_detail_rows = []
failed_tickers = []

for i, market_ticker in enumerate(market_tickers, start=1):
    print(f"[{i}/{len(market_tickers)}] Processing {market_ticker}...")

    try:
        # 1. Get historical market details
        market_data = get_json(f"/historical/markets/{market_ticker}")
        market = market_data.get("market", market_data)

        market_title = market.get("title")
        event_ticker = market.get("event_ticker")
        series_ticker = market.get("series_ticker")

        market_detail_rows.append({
            "market_ticker": market_ticker,
            "market_title": market_title,
            "event_ticker": event_ticker,
            "series_ticker": series_ticker,
            "status": market.get("status"),
            "result": market.get("result"),
            "open_time": market.get("open_time"),
            "close_time": market.get("close_time"),
            "expiration_time": market.get("expiration_time"),
            "settlement_ts": market.get("settlement_ts"),
            "volume_fp": market.get("volume_fp"),
            "volume_24h_fp": market.get("volume_24h_fp"),
        })

        open_time = pd.to_datetime(
            market.get("open_time"),
            utc=True,
            errors="coerce"
        )

        end_time = pd.to_datetime(
            market.get("close_time") or market.get("expiration_time") or market.get("settlement_ts"),
            utc=True,
            errors="coerce"
        )

        if pd.isna(open_time) or pd.isna(end_time):
            raise ValueError("Missing open_time or end_time")

        start_ts = int(open_time.timestamp())
        end_ts = int(end_time.timestamp())

        print("  Title:", market_title)
        print("  Event:", event_ticker)
        print("  Series:", series_ticker)
        print("  Start:", open_time)
        print("  End:", end_time)

        # 2. Get daily candlesticks
        candles_data = get_json(
            f"/historical/markets/{market_ticker}/candlesticks",
            params={
                "start_ts": start_ts,
                "end_ts": end_ts,
                "period_interval": 1440
            }
        )

        candles = candles_data.get("candlesticks", [])
        print("  Daily candles:", len(candles))

        one_market_rows = []

        for candle in candles:
            row = flatten_dict(candle)

            row["market_ticker"] = market_ticker
            row["market_title"] = market_title
            row["event_ticker"] = event_ticker
            row["series_ticker"] = series_ticker
            row["market_status"] = market.get("status")
            row["market_result"] = market.get("result")
            row["market_open_time"] = open_time
            row["market_end_time"] = end_time
            row["period_interval"] = 1440
            row["period_label"] = "daily"

            one_market_rows.append(row)
            all_daily_rows.append(row)

        # 3. Save individual market CSV
        df_one_market = pd.DataFrame(one_market_rows)

        df_one_market.to_csv(
            f"{OUT_DIR}/daily_candlesticks_{safe_filename(market_ticker)}.csv",
            index=False
        )

    except Exception as e:
        print(f"  Failed {market_ticker}: {e}")
        failed_tickers.append({
            "ticker": market_ticker,
            "error": str(e)
        })

    time.sleep(0.25)


# 4. Combined daily candlesticks DataFrame
df_all_daily_candlesticks = pd.DataFrame(all_daily_rows)

display(df_all_daily_candlesticks)

print("Total daily candlestick rows:", len(df_all_daily_candlesticks))
print("Columns:", df_all_daily_candlesticks.columns.tolist())

df_all_daily_candlesticks.to_csv(
    f"{OUT_DIR}/all_daily_candlesticks_with_market_info.csv",
    index=False
)


# 5. Market detail summary DataFrame
df_market_details = pd.DataFrame(market_detail_rows)

display(df_market_details)

df_market_details.to_csv(
    f"{OUT_DIR}/market_details_summary.csv",
    index=False
)


# 6. Failed ticker DataFrame
if failed_tickers:
    df_failed_tickers = pd.DataFrame(failed_tickers)
    display(df_failed_tickers)

    df_failed_tickers.to_csv(
        f"{OUT_DIR}/failed_market_tickers.csv",
        index=False
    )


# 7. Zip and download everything
zip_name = "kalshi_new_attempt.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for filename in os.listdir(OUT_DIR):
        filepath = os.path.join(OUT_DIR, filename)
        z.write(filepath, arcname=filename)

files.download(zip_name)

[1/10] Processing KXELECTIONMOVZOHRAN-25-ZMAM-T24...
  Title: Margin of victory for Zohran Mamdani in the Mayoral Election?
  Event: KXELECTIONMOVZOHRAN-25
  Series: None
  Start: 2025-08-13 14:00:00+00:00
  End: 2025-12-03 01:44:48.719034+00:00
  Daily candles: 111
[2/10] Processing KXELECTIONMOVZOHRAN-25-ZMAM-B7.5...
  Title: Margin of victory for Zohran Mamdani in the Mayoral Election?
  Event: KXELECTIONMOVZOHRAN-25
  Series: None
  Start: 2025-08-13 14:00:00+00:00
  End: 2025-12-03 01:44:48.719034+00:00
  Daily candles: 111
[3/10] Processing KXELECTIONMOVZOHRAN-25-ZMAM-B22.5...
  Title: Margin of victory for Zohran Mamdani in the Mayoral Election?
  Event: KXELECTIONMOVZOHRAN-25
  Series: None
  Start: 2025-08-13 14:00:00+00:00
  End: 2025-12-03 01:44:48.719034+00:00
  Daily candles: 111
[4/10] Processing KXELECTIONMOVZOHRAN-25-ZMAM-B19.5...
  Title: Margin of victory for Zohran Mamdani in the Mayoral Election?
  Event: KXELECTIONMOVZOHRAN-25
  Series: None
  Start: 2025-08-13 14:

,end_period_ts,open_interest,price.close,price.high,price.low,price.mean,price.open,price.previous,volume,yes_ask.close,...,market_ticker,market_title,event_ticker,series_ticker,market_status,market_result,market_open_time,market_end_time,period_interval,period_label
0,1755144000,5154.00,0.4100,0.7400,0.1900,0.3164,0.1900,None,5433.00,0.4500,...,KXELECTIONMOVZOHRAN-25-ZMAM-T24,Margin of victory for Zohran Mamdani in the Ma...,KXELECTIONMOVZOHRAN-25,None,finalized,no,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,1440,daily
1,1755230400,6394.00,0.4000,0.7000,0.4000,0.4676,0.4100,0.4100,1858.00,0.4800,...,KXELECTIONMOVZOHRAN-25-ZMAM-T24,Margin of victory for Zohran Mamdani in the Ma...,KXELECTIONMOVZOHRAN-25,None,finalized,no,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,1440,daily
2,1755316800,6429.00,0.3900,0.4000,0.3900,0.3963,0.4000,0.4000,90.00,0.4700,...,KXELECTIONMOVZOHRAN-25-ZMAM-T24,Margin of victory for Zohran Mamdani in the Ma...,KXELECTIONMOVZOHRAN-25,None,finalized,no,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,1440,daily
3,1755403200,6429.00,0.4800,0.4800,0.4700,0.4750,0.4700,0.3900,4.00,0.4700,...,KXELECTIONMOVZOHRAN-25-ZMAM-T24,Margin of victory for Zohran Mamdani in the Ma...,KXELECTIONMOVZOHRAN-25,None,finalized,no,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,1440,daily
4,1755489600,6840.00,0.4100,0.4200,0.4100,0.4100,0.4200,0.4800,468.00,0.4600,...,KXELECTIONMOVZOHRAN-25-ZMAM-T24,Margin of victory for Zohran Mamdani in the Ma...,KXELECTIONMOVZOHRAN-25,None,finalized,no,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,1440,daily
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1105,1764306000,921513.00,0.0100,0.0100,0.0100,0.0100,0.0100,0.0100,348.00,0.0100,...,KXELECTIONMOVZOHRAN-25-ZMAM-B1.5,Margin of victory for Zohran Mamdani in the Ma...,KXELECTIONMOVZOHRAN-25,None,finalized,no,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,1440,daily
1106,1764392400,921514.00,0.0100,0.0100,0.0100,0.0100,0.0100,0.0100,1.00,0.0100,...,KXELECTIONMOVZOHRAN-25-ZMAM-B1.5,Margin of victory for Zohran Mamdani in the Ma...,KXELECTIONMOVZOHRAN-25,None,finalized,no,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,1440,daily
1107,1764478800,921514.00,None,None,None,None,None,0.0100,0.00,0.0100,...,KXELECTIONMOVZOHRAN-25-ZMAM-B1.5,Margin of victory for Zohran Mamdani in the Ma...,KXELECTIONMOVZOHRAN-25,None,finalized,no,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,1440,daily
1108,1764565200,921514.00,None,None,None,None,None,0.0100,0.00,0.0100,...,KXELECTIONMOVZOHRAN-25-ZMAM-B1.5,Margin of victory for Zohran Mamdani in the Ma...,KXELECTIONMOVZOHRAN-25,None,finalized,no,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,1440,daily


Total daily candlestick rows: 1110
Columns: ['end_period_ts', 'open_interest', 'price.close', 'price.high', 'price.low', 'price.mean', 'price.open', 'price.previous', 'volume', 'yes_ask.close', 'yes_ask.high', 'yes_ask.low', 'yes_ask.open', 'yes_bid.close', 'yes_bid.high', 'yes_bid.low', 'yes_bid.open', 'market_ticker', 'market_title', 'event_ticker', 'series_ticker', 'market_status', 'market_result', 'market_open_time', 'market_end_time', 'period_interval', 'period_label']


,market_ticker,market_title,event_ticker,series_ticker,status,result,open_time,close_time,expiration_time,settlement_ts,volume_fp,volume_24h_fp
0,KXELECTIONMOVZOHRAN-25-ZMAM-T24,Margin of victory for Zohran Mamdani in the Ma...,KXELECTIONMOVZOHRAN-25,None,finalized,no,2025-08-13T14:00:00Z,2025-12-03T01:44:48.719034Z,2026-11-04T15:00:00Z,2025-12-03T02:14:52.646486Z,2715311.00,0.00
1,KXELECTIONMOVZOHRAN-25-ZMAM-B7.5,Margin of victory for Zohran Mamdani in the Ma...,KXELECTIONMOVZOHRAN-25,None,finalized,no,2025-08-13T14:00:00Z,2025-12-03T01:44:48.719034Z,2026-11-04T15:00:00Z,2025-12-03T02:14:52.646486Z,3398478.00,0.00
2,KXELECTIONMOVZOHRAN-25-ZMAM-B22.5,Margin of victory for Zohran Mamdani in the Ma...,KXELECTIONMOVZOHRAN-25,None,finalized,no,2025-08-13T14:00:00Z,2025-12-03T01:44:48.719034Z,2026-11-04T15:00:00Z,2025-12-03T02:14:52.646486Z,922621.00,0.00
3,KXELECTIONMOVZOHRAN-25-ZMAM-B19.5,Margin of victory for Zohran Mamdani in the Ma...,KXELECTIONMOVZOHRAN-25,None,finalized,no,2025-08-13T14:00:00Z,2025-12-03T01:44:48.719034Z,2026-11-04T15:00:00Z,2025-12-03T02:14:52.646486Z,819153.00,0.00
4,KXELECTIONMOVZOHRAN-25-ZMAM-B16.5,Margin of victory for Zohran Mamdani in the Ma...,KXELECTIONMOVZOHRAN-25,None,finalized,no,2025-08-13T14:00:00Z,2025-12-03T01:44:48.719034Z,2026-11-04T15:00:00Z,2025-12-03T02:14:52.646486Z,923736.00,0.00
5,KXELECTIONMOVZOHRAN-25-ZMAM-B13.5,Margin of victory for Zohran Mamdani in the Ma...,KXELECTIONMOVZOHRAN-25,None,finalized,no,2025-08-13T14:00:00Z,2025-12-03T01:44:48.719034Z,2026-11-04T15:00:00Z,2025-12-03T02:14:52.646486Z,1454986.00,0.00
6,KXELECTIONMOVZOHRAN-25-ZMAM-B10.5,Margin of victory for Zohran Mamdani in the Ma...,KXELECTIONMOVZOHRAN-25,None,finalized,yes,2025-08-13T14:00:00Z,2025-12-03T01:44:48.719034Z,2026-11-04T15:00:00Z,2025-12-03T02:14:52.646486Z,2641908.00,0.00
7,KXELECTIONMOVZOHRAN-25-ZMAM-T0,Margin of victory for Zohran Mamdani in the Ma...,KXELECTIONMOVZOHRAN-25,None,finalized,no,2025-08-13T14:00:00Z,2025-12-03T01:44:48.719034Z,2026-11-04T15:00:00Z,2025-12-03T02:14:52.646486Z,1488079.00,0.00
8,KXELECTIONMOVZOHRAN-25-ZMAM-B4.5,Margin of victory for Zohran Mamdani in the Ma...,KXELECTIONMOVZOHRAN-25,None,finalized,no,2025-08-13T14:00:00Z,2025-12-03T01:44:48.719034Z,2026-11-04T15:00:00Z,2025-12-03T02:14:52.646486Z,1689089.00,0.00
9,KXELECTIONMOVZOHRAN-25-ZMAM-B1.5,Margin of victory for Zohran Mamdani in the Ma...,KXELECTIONMOVZOHRAN-25,None,finalized,no,2025-08-13T14:00:00Z,2025-12-03T01:44:48.719034Z,2026-11-04T15:00:00Z,2025-12-03T02:14:52.646486Z,1256254.00,0.00


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
import json
import time
import os
import re
import zipfile
import requests
import pandas as pd
from urllib.parse import urljoin
from google.colab import files

BASE_URL = "https://external-api.kalshi.com/trade-api/v2"

market_tickers = [
    "KXELECTIONMOVZOHRAN-25-ZMAM-T24",
    "KXELECTIONMOVZOHRAN-25-ZMAM-B7.5",
    "KXELECTIONMOVZOHRAN-25-ZMAM-B22.5",
    "KXELECTIONMOVZOHRAN-25-ZMAM-B19.5",
    "KXELECTIONMOVZOHRAN-25-ZMAM-B16.5",
    "KXELECTIONMOVZOHRAN-25-ZMAM-B13.5",
    "KXELECTIONMOVZOHRAN-25-ZMAM-B10.5",
    "KXELECTIONMOVZOHRAN-25-ZMAM-T0",
    "KXELECTIONMOVZOHRAN-25-ZMAM-B4.5",
    "KXELECTIONMOVZOHRAN-25-ZMAM-B1.5",
]

OUT_DIR = "kalshi_zmam_daily_candlesticks_with_rules"
os.makedirs(OUT_DIR, exist_ok=True)

session = requests.Session()
session.headers.update({
    "User-Agent": "kalshi-zmam-daily-candlesticks-with-rules-colab/1.0"
})


def safe_filename(x):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(x))


def get_json(path, params=None, max_retries=10, timeout=30):
    url = urljoin(BASE_URL + "/", path.lstrip("/"))
    params = params or {}

    for attempt in range(max_retries):
        resp = session.get(url, params=params, timeout=timeout)

        if resp.status_code == 429:
            sleep_s = min(5 * (attempt + 1), 60)
            print(f"Rate limited. Sleeping {sleep_s}s...")
            time.sleep(sleep_s)
            continue

        try:
            data = resp.json()
        except Exception:
            data = {"raw_text": resp.text}

        if resp.status_code >= 400:
            print("Bad URL:", resp.url)
            print("Kalshi response:", json.dumps(data, indent=2)[:3000])
            resp.raise_for_status()

        return data

    raise RuntimeError(f"Failed after {max_retries} retries: {url}")


def flatten_dict(d, parent_key=""):
    out = {}

    for k, v in d.items():
        key = f"{parent_key}.{k}" if parent_key else k

        if isinstance(v, dict):
            out.update(flatten_dict(v, key))
        elif isinstance(v, list):
            out[key] = json.dumps(v)
        else:
            out[key] = v

    return out


all_daily_rows = []
market_detail_rows = []
failed_tickers = []

for i, market_ticker in enumerate(market_tickers, start=1):
    print(f"[{i}/{len(market_tickers)}] Processing {market_ticker}...")

    try:
        # 1. Get historical market details
        market_data = get_json(f"/historical/markets/{market_ticker}")
        market = market_data.get("market", market_data)

        market_title = market.get("title")
        event_ticker = market.get("event_ticker")
        series_ticker = market.get("series_ticker")
        no_sub_title = market.get("no_sub_title")
        rules_primary = market.get("rules_primary")
        rules_secondary = market.get("rules_secondary")

        market_detail_rows.append({
            "market_ticker": market_ticker,
            "market_title": market_title,
            "event_ticker": event_ticker,
            "series_ticker": series_ticker,
            "no_sub_title": no_sub_title,
            "rules_primary": rules_primary,
            "rules_secondary": rules_secondary,
            "yes_sub_title": market.get("yes_sub_title"),
            "status": market.get("status"),
            "result": market.get("result"),
            "open_time": market.get("open_time"),
            "close_time": market.get("close_time"),
            "expiration_time": market.get("expiration_time"),
            "settlement_ts": market.get("settlement_ts"),
            "volume_fp": market.get("volume_fp"),
            "volume_24h_fp": market.get("volume_24h_fp"),
        })

        open_time = pd.to_datetime(
            market.get("open_time"),
            utc=True,
            errors="coerce"
        )

        end_time = pd.to_datetime(
            market.get("close_time") or market.get("expiration_time") or market.get("settlement_ts"),
            utc=True,
            errors="coerce"
        )

        if pd.isna(open_time) or pd.isna(end_time):
            raise ValueError("Missing open_time or end_time")

        start_ts = int(open_time.timestamp())
        end_ts = int(end_time.timestamp())

        print("  Title:", market_title)
        print("  Event:", event_ticker)
        print("  Series:", series_ticker)
        print("  No subtitle:", no_sub_title)
        print("  Start:", open_time)
        print("  End:", end_time)

        # 2. Get daily candlesticks
        candles_data = get_json(
            f"/historical/markets/{market_ticker}/candlesticks",
            params={
                "start_ts": start_ts,
                "end_ts": end_ts,
                "period_interval": 1440
            }
        )

        candles = candles_data.get("candlesticks", [])
        print("  Daily candles:", len(candles))

        one_market_rows = []

        for candle in candles:
            row = flatten_dict(candle)

            row["market_ticker"] = market_ticker
            row["market_title"] = market_title
            row["event_ticker"] = event_ticker
            row["series_ticker"] = series_ticker
            row["no_sub_title"] = no_sub_title
            row["rules_primary"] = rules_primary
            row["rules_secondary"] = rules_secondary
            row["yes_sub_title"] = market.get("yes_sub_title")
            row["market_status"] = market.get("status")
            row["market_result"] = market.get("result")
            row["market_open_time"] = open_time
            row["market_end_time"] = end_time
            row["period_interval"] = 1440
            row["period_label"] = "daily"

            one_market_rows.append(row)
            all_daily_rows.append(row)

        # 3. Save individual market CSV
        df_one_market = pd.DataFrame(one_market_rows)

        df_one_market.to_csv(
            f"{OUT_DIR}/daily_candlesticks_{safe_filename(market_ticker)}.csv",
            index=False
        )

    except Exception as e:
        print(f"  Failed {market_ticker}: {e}")
        failed_tickers.append({
            "ticker": market_ticker,
            "error": str(e)
        })

    time.sleep(0.25)


# 4. Combined daily candlesticks DataFrame
df_all_daily_candlesticks = pd.DataFrame(all_daily_rows)

display(df_all_daily_candlesticks)

print("Total daily candlestick rows:", len(df_all_daily_candlesticks))
print("Columns:", df_all_daily_candlesticks.columns.tolist())

df_all_daily_candlesticks.to_csv(
    f"{OUT_DIR}/all_daily_candlesticks_with_rules.csv",
    index=False
)


# 5. Market detail summary DataFrame
df_market_details = pd.DataFrame(market_detail_rows)

display(df_market_details)

df_market_details.to_csv(
    f"{OUT_DIR}/market_details_summary_with_rules.csv",
    index=False
)


# 6. Failed ticker DataFrame
if failed_tickers:
    df_failed_tickers = pd.DataFrame(failed_tickers)
    display(df_failed_tickers)

    df_failed_tickers.to_csv(
        f"{OUT_DIR}/failed_market_tickers.csv",
        index=False
    )


# 7. Zip and download everything
zip_name = "kalshi_again.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for filename in os.listdir(OUT_DIR):
        filepath = os.path.join(OUT_DIR, filename)
        z.write(filepath, arcname=filename)

files.download(zip_name)

[1/10] Processing KXELECTIONMOVZOHRAN-25-ZMAM-T24...
  Title: Margin of victory for Zohran Mamdani in the Mayoral Election?
  Event: KXELECTIONMOVZOHRAN-25
  Series: None
  No subtitle: 24% or more
  Start: 2025-08-13 14:00:00+00:00
  End: 2025-12-03 01:44:48.719034+00:00
  Daily candles: 111
[2/10] Processing KXELECTIONMOVZOHRAN-25-ZMAM-B7.5...
  Title: Margin of victory for Zohran Mamdani in the Mayoral Election?
  Event: KXELECTIONMOVZOHRAN-25
  Series: None
  No subtitle: 6-8.99%
  Start: 2025-08-13 14:00:00+00:00
  End: 2025-12-03 01:44:48.719034+00:00
  Daily candles: 111
[3/10] Processing KXELECTIONMOVZOHRAN-25-ZMAM-B22.5...
  Title: Margin of victory for Zohran Mamdani in the Mayoral Election?
  Event: KXELECTIONMOVZOHRAN-25
  Series: None
  No subtitle: 21-23.99%
  Start: 2025-08-13 14:00:00+00:00
  End: 2025-12-03 01:44:48.719034+00:00
  Daily candles: 111
[4/10] Processing KXELECTIONMOVZOHRAN-25-ZMAM-B19.5...
  Title: Margin of victory for Zohran Mamdani in the Mayoral Elect

,end_period_ts,open_interest,price.close,price.high,price.low,price.mean,price.open,price.previous,volume,yes_ask.close,...,no_sub_title,rules_primary,rules_secondary,yes_sub_title,market_status,market_result,market_open_time,market_end_time,period_interval,period_label
0,1755144000,5154.00,0.4100,0.7400,0.1900,0.3164,0.1900,None,5433.00,0.4500,...,24% or more,If the popular vote margin of victory is 24% o...,The margin of victory is calculated by taking ...,24% or more,finalized,no,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,1440,daily
1,1755230400,6394.00,0.4000,0.7000,0.4000,0.4676,0.4100,0.4100,1858.00,0.4800,...,24% or more,If the popular vote margin of victory is 24% o...,The margin of victory is calculated by taking ...,24% or more,finalized,no,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,1440,daily
2,1755316800,6429.00,0.3900,0.4000,0.3900,0.3963,0.4000,0.4000,90.00,0.4700,...,24% or more,If the popular vote margin of victory is 24% o...,The margin of victory is calculated by taking ...,24% or more,finalized,no,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,1440,daily
3,1755403200,6429.00,0.4800,0.4800,0.4700,0.4750,0.4700,0.3900,4.00,0.4700,...,24% or more,If the popular vote margin of victory is 24% o...,The margin of victory is calculated by taking ...,24% or more,finalized,no,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,1440,daily
4,1755489600,6840.00,0.4100,0.4200,0.4100,0.4100,0.4200,0.4800,468.00,0.4600,...,24% or more,If the popular vote margin of victory is 24% o...,The margin of victory is calculated by taking ...,24% or more,finalized,no,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,1440,daily
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1105,1764306000,921513.00,0.0100,0.0100,0.0100,0.0100,0.0100,0.0100,348.00,0.0100,...,0-2.99%,If the popular vote margin of victory is 0-2.9...,The margin of victory is calculated by taking ...,0-2.99%,finalized,no,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,1440,daily
1106,1764392400,921514.00,0.0100,0.0100,0.0100,0.0100,0.0100,0.0100,1.00,0.0100,...,0-2.99%,If the popular vote margin of victory is 0-2.9...,The margin of victory is calculated by taking ...,0-2.99%,finalized,no,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,1440,daily
1107,1764478800,921514.00,None,None,None,None,None,0.0100,0.00,0.0100,...,0-2.99%,If the popular vote margin of victory is 0-2.9...,The margin of victory is calculated by taking ...,0-2.99%,finalized,no,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,1440,daily
1108,1764565200,921514.00,None,None,None,None,None,0.0100,0.00,0.0100,...,0-2.99%,If the popular vote margin of victory is 0-2.9...,The margin of victory is calculated by taking ...,0-2.99%,finalized,no,2025-08-13 14:00:00+00:00,2025-12-03 01:44:48.719034+00:00,1440,daily


Total daily candlestick rows: 1110
Columns: ['end_period_ts', 'open_interest', 'price.close', 'price.high', 'price.low', 'price.mean', 'price.open', 'price.previous', 'volume', 'yes_ask.close', 'yes_ask.high', 'yes_ask.low', 'yes_ask.open', 'yes_bid.close', 'yes_bid.high', 'yes_bid.low', 'yes_bid.open', 'market_ticker', 'market_title', 'event_ticker', 'series_ticker', 'no_sub_title', 'rules_primary', 'rules_secondary', 'yes_sub_title', 'market_status', 'market_result', 'market_open_time', 'market_end_time', 'period_interval', 'period_label']


,market_ticker,market_title,event_ticker,series_ticker,no_sub_title,rules_primary,rules_secondary,yes_sub_title,status,result,open_time,close_time,expiration_time,settlement_ts,volume_fp,volume_24h_fp
0,KXELECTIONMOVZOHRAN-25-ZMAM-T24,Margin of victory for Zohran Mamdani in the Ma...,KXELECTIONMOVZOHRAN-25,None,24% or more,If the popular vote margin of victory is 24% o...,The margin of victory is calculated by taking ...,24% or more,finalized,no,2025-08-13T14:00:00Z,2025-12-03T01:44:48.719034Z,2026-11-04T15:00:00Z,2025-12-03T02:14:52.646486Z,2715311.00,0.00
1,KXELECTIONMOVZOHRAN-25-ZMAM-B7.5,Margin of victory for Zohran Mamdani in the Ma...,KXELECTIONMOVZOHRAN-25,None,6-8.99%,If the popular vote margin of victory is 6-8.9...,The margin of victory is calculated by taking ...,6-8.99%,finalized,no,2025-08-13T14:00:00Z,2025-12-03T01:44:48.719034Z,2026-11-04T15:00:00Z,2025-12-03T02:14:52.646486Z,3398478.00,0.00
2,KXELECTIONMOVZOHRAN-25-ZMAM-B22.5,Margin of victory for Zohran Mamdani in the Ma...,KXELECTIONMOVZOHRAN-25,None,21-23.99%,If the popular vote margin of victory is 21-23...,The margin of victory is calculated by taking ...,21-23.99%,finalized,no,2025-08-13T14:00:00Z,2025-12-03T01:44:48.719034Z,2026-11-04T15:00:00Z,2025-12-03T02:14:52.646486Z,922621.00,0.00
3,KXELECTIONMOVZOHRAN-25-ZMAM-B19.5,Margin of victory for Zohran Mamdani in the Ma...,KXELECTIONMOVZOHRAN-25,None,18-20.99%,If the popular vote margin of victory is 18-20...,The margin of victory is calculated by taking ...,18-20.99%,finalized,no,2025-08-13T14:00:00Z,2025-12-03T01:44:48.719034Z,2026-11-04T15:00:00Z,2025-12-03T02:14:52.646486Z,819153.00,0.00
4,KXELECTIONMOVZOHRAN-25-ZMAM-B16.5,Margin of victory for Zohran Mamdani in the Ma...,KXELECTIONMOVZOHRAN-25,None,15-17.99%,If the popular vote margin of victory is 15-17...,The margin of victory is calculated by taking ...,15-17.99%,finalized,no,2025-08-13T14:00:00Z,2025-12-03T01:44:48.719034Z,2026-11-04T15:00:00Z,2025-12-03T02:14:52.646486Z,923736.00,0.00
5,KXELECTIONMOVZOHRAN-25-ZMAM-B13.5,Margin of victory for Zohran Mamdani in the Ma...,KXELECTIONMOVZOHRAN-25,None,12-14.99%,If the popular vote margin of victory is 12-14...,The margin of victory is calculated by taking ...,12-14.99%,finalized,no,2025-08-13T14:00:00Z,2025-12-03T01:44:48.719034Z,2026-11-04T15:00:00Z,2025-12-03T02:14:52.646486Z,1454986.00,0.00
6,KXELECTIONMOVZOHRAN-25-ZMAM-B10.5,Margin of victory for Zohran Mamdani in the Ma...,KXELECTIONMOVZOHRAN-25,None,9-11.99%,If the popular vote margin of victory is 9-11....,The margin of victory is calculated by taking ...,9-11.99%,finalized,yes,2025-08-13T14:00:00Z,2025-12-03T01:44:48.719034Z,2026-11-04T15:00:00Z,2025-12-03T02:14:52.646486Z,2641908.00,0.00
7,KXELECTIONMOVZOHRAN-25-ZMAM-T0,Margin of victory for Zohran Mamdani in the Ma...,KXELECTIONMOVZOHRAN-25,None,Less than 0%,If the popular vote margin of victory is 0% or...,The margin of victory is calculated by taking ...,Less than 0%,finalized,no,2025-08-13T14:00:00Z,2025-12-03T01:44:48.719034Z,2026-11-04T15:00:00Z,2025-12-03T02:14:52.646486Z,1488079.00,0.00
8,KXELECTIONMOVZOHRAN-25-ZMAM-B4.5,Margin of victory for Zohran Mamdani in the Ma...,KXELECTIONMOVZOHRAN-25,None,3-5.99%,If the popular vote margin of victory is 3-5.9...,The margin of victory is calculated by taking ...,3-5.99%,finalized,no,2025-08-13T14:00:00Z,2025-12-03T01:44:48.719034Z,2026-11-04T15:00:00Z,2025-12-03T02:14:52.646486Z,1689089.00,0.00
9,KXELECTIONMOVZOHRAN-25-ZMAM-B1.5,Margin of victory for Zohran Mamdani in the Ma...,KXELECTIONMOVZOHRAN-25,None,0-2.99%,If the popular vote margin of victory is 0-2.9...,The margin of victory is calculated by taking ...,0-2.99%,finalized,no,2025-08-13T14:00:00Z,2025-12-03T01:44:48.719034Z,2026-11-04T15:00:00Z,2025-12-03T02:14:52.646486Z,1256254.00,0.00


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>